# Importing

In [ ]:
import json
import re
import time
import os, pickle,hashlib
import copy
from typing import Dict, List, Tuple
import numpy as np
import networkx as nx
from datetime import datetime
from tqdm import tqdm

import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader


from sklearn.metrics.pairwise import cosine_similarity
from dataclasses import dataclass
from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, StoppingCriteria, StoppingCriteriaList, pipeline)

from huggingface_hub import login
import warnings
warnings.filterwarnings('ignore', message='Setting `pad_token_id`')

from sentence_transformers import SentenceTransformer
sentence_model = SentenceTransformer('all-MiniLM-L6-v2')

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from peft.utils.peft_types import TaskType

hf_token = "Huggingface Token"
login(hf_token)

# Multi Component Reward Function

In [ ]:
class EmbeddingModelManager:
    _instance = None
    _model = None

    def __new__(cls):
        if cls._instance is None:
            cls._instance = super(EmbeddingModelManager, cls).__new__(cls)
        return cls._instance

    def get_model(self, model_name='all-MiniLM-L6-v2'):
        if self._model is None:
            print(f"Loading SentenceTransformer: {model_name}")
            self._model = SentenceTransformer(model_name)
            self._model = self._model.cpu()
        return self._model

    def cleanup(self):
        if self._model is not None:
            del self._model
            self._model = None
        torch.cuda.empty_cache()
        
embedding_manager = EmbeddingModelManager()

In [ ]:
class LLMJudgeVerifier:    
    def __init__(self, 
                 model_id="Qwen/Qwen2.5-14B-Instruct",
                 device_map="auto", 
                 torch_dtype="auto", 
                 use_4bit=True):
        self.model_id = model_id
        
        # Configure quantization
        if use_4bit:
            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.bfloat16,
                bnb_4bit_use_double_quant=True
            )
            print(f"Loading {model_id} with 4-bit quantization...")
        else:
            bnb_config = None
            print(f"Loading {model_id}...")
        
        # Handle torch_dtype
        if torch_dtype == "auto":
            dtype = "auto"
        elif isinstance(torch_dtype, str):
            dtype = getattr(torch, torch_dtype, "auto")
        else:
            dtype = torch_dtype
        
        # Create pipeline
        from transformers import pipeline
        
        self.pipe = pipeline(
            "text-generation",
            model=model_id,
            torch_dtype=dtype,
            device_map=device_map,
            model_kwargs={"quantization_config": bnb_config} if bnb_config else {},
            trust_remote_code=True
        )
        print(f" LLM Judge Verifier initialized")
        print(f"   Model: {model_id}")
        print(f"   4-bit: {use_4bit}")

    def _judge_to_cpu(self):
        if self.reward_function.fact_verification_system is not None:
            judge = self.reward_function.fact_verification_system.llm_judge
            if judge is not None and hasattr(judge, 'model'):
                judge.model = judge.model.to('cpu')
                torch.cuda.empty_cache()

    def _judge_to_gpu(self):
        if self.reward_function.fact_verification_system is not None:
            judge = self.reward_function.fact_verification_system.llm_judge
            if judge is not None and hasattr(judge, 'model'):
                judge.model = judge.model.to(self.device)

    def _build_prompt(self, fact_text, context):
        return f"""You are a medical fact verification specialist.

    Rate the medical accuracy of this fact (0.0-1.0):
    - 1.0: Definitely correct, established medical knowledge
    - 0.7-0.9: Likely correct, standard medical practice
    - 0.4-0.6: Uncertain or context-dependent
    - 0.0-0.3: Likely incorrect, contradicts medical evidence

    FACT: {fact_text}

    CONTEXT: {context if context else "None"}

    Respond with JSON only:
    {{"confidence_score": 0.95, "reasoning": "brief explanation"}}"""
    
    def verify_fact(self, fact, context=""):
        """Verify fact and update fact.llm_score in place"""
        fact_text = fact.text if hasattr(fact, 'text') else str(fact)
        prompt = self._build_prompt(fact_text, context)
        
        try:
            outputs = self.pipe(
            prompt,
            max_new_tokens=300,
            temperature=0,
            do_sample=True, 
            return_full_text=False
        )
            
            response_text = outputs[0]["generated_text"]
            score = self._parse_score(response_text)
            fact.llm_score = score
            
        except Exception as e:
            print(f"Error in LLM verification: {e}")
            fact.llm_score = 0.5
    
    def batch_verify_facts(self, facts_list, context=""):
        """Verify multiple facts"""
        results = []
        for fact in facts_list:
            self.verify_fact(fact, context)
            results.append({
                'confidence_score': fact.llm_score,
                'reasoning': 'LLM verified',
                'medical_category': fact.category if hasattr(fact, 'category') else 'GENERAL'
            })
        return results
    
    
    def _parse_score(self, response_text):
        """Extract confidence score from response"""
        try:
            json_match = re.search(r'\{[^{}]*"confidence_score"[^{}]*\}', response_text, re.DOTALL)
            if json_match:
                result = json.loads(json_match.group(0))
                return float(result.get('confidence_score', 0.5))
            score_match = re.search(r'confidence_score["\s:]+([0-9.]+)', response_text)
            if score_match:
                return float(score_match.group(1))
            return 0.5
        except Exception as e:
            print(f"Parse error: {e}")
            return 0.5

In [ ]:
from collections import defaultdict

class PrimeKGVerifier:
    def __init__(self, primekg_path, cache_dir = "primekg_cache"):
        self.cache_dir = cache_dir
        self._fact_cache = {}
        self._embedding_cache = {}
        os.makedirs(cache_dir, exist_ok=True)
        
        self.embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
        
        # Load PrimeKG
        print("Loading PrimeKG...")
        
        self.kg_df = self._load_primekg(primekg_path)
        
        print("Building concept and relationship indices...")
        self.concept_index = self._build_concept_index()
        self.relation_index = self._build_relation_index()
        
        print("Loading concept embeddings...")
        self.concept_embeddings = self._load_or_generate_embeddings()
        
        print(f"PrimeKG loaded: {len(self.concept_index)} unique concepts, "
              f"{len(self.kg_df)} relationships")
    
    def _load_primekg(self, path):
        
        """Load PrimeKG from CSV"""
        cache_file = os.path.join(self.cache_dir, "primekg_processed.pkl")
        
        if os.path.exists(cache_file):
            print("Loading from cache...")
            return pd.read_pickle(cache_file)
        
        print(f"Loading PrimeKG from {path}...")
        df = pd.read_csv(path, low_memory=False)
        
        # Keep relevant columns
        required_cols = ['x_id', 'x_type', 'x_name', 'relation', 'y_id', 'y_type', 'y_name']
        df = df[required_cols]
        
        # Clean text
        df['x_name'] = df['x_name'].fillna('').str.lower()
        df['y_name'] = df['y_name'].fillna('').str.lower()
        df['relation'] = df['relation'].fillna('unknown')
        
        # Save cache
        df.to_pickle(cache_file)
        print(f"Cached processed PrimeKG ({len(df)} edges)")
        
        return df
    
    def _build_concept_index(self) -> Dict[str, Dict]:
        """Build index: concept_id -> {name, type, synonyms}"""
        cache_file = os.path.join(self.cache_dir, "concept_index.pkl")
        
        if os.path.exists(cache_file):
            with open(cache_file, 'rb') as f:
                return pickle.load(f)
        
        concept_index = {}
        
        # Index from x nodes
        for _, row in self.kg_df[['x_id', 'x_name', 'x_type']].drop_duplicates().iterrows():
            if row['x_id'] not in concept_index:
                concept_index[row['x_id']] = {
                    'name': row['x_name'],
                    'type': row['x_type'],
                    'synonyms': set([row['x_name']])
                }
            else:
                concept_index[row['x_id']]['synonyms'].add(row['x_name'])
        
        # Index from y nodes
        for _, row in self.kg_df[['y_id', 'y_name', 'y_type']].drop_duplicates().iterrows():
            if row['y_id'] not in concept_index:
                concept_index[row['y_id']] = {
                    'name': row['y_name'],
                    'type': row['y_type'],
                    'synonyms': set([row['y_name']])
                }
            else:
                concept_index[row['y_id']]['synonyms'].add(row['y_name'])
        
        # Convert sets to lists for pickling
        for concept_id in concept_index:
            concept_index[concept_id]['synonyms'] = list(concept_index[concept_id]['synonyms'])
        
        with open(cache_file, 'wb') as f:
            pickle.dump(concept_index, f)
        
        return concept_index
    
    def _build_relation_index(self) -> Dict[Tuple[str, str], List[str]]:
        """Build index: (concept1_id, concept2_id) -> [relations]"""
        cache_file = os.path.join(self.cache_dir, "relation_index.pkl")
        
        if os.path.exists(cache_file):
            with open(cache_file, 'rb') as f:
                return pickle.load(f)
        
        relation_index = defaultdict(list)
        
        for _, row in self.kg_df.iterrows():
            key = (row['x_id'], row['y_id'])
            relation_index[key].append(row['relation'])
        
        relation_index = dict(relation_index)
        
        with open(cache_file, 'wb') as f:
            pickle.dump(relation_index, f)
        
        return relation_index
    
    def _load_or_generate_embeddings(self) -> Dict[str, np.ndarray]:
        """Load or generate embeddings for all concepts"""
        cache_file = os.path.join(self.cache_dir, "concept_embeddings.pkl")
        
        if os.path.exists(cache_file):
            with open(cache_file, 'rb') as f:
                return pickle.load(f)
        
        print("Generating concept embeddings (this may take a few minutes)...")
        embeddings = {}
        
        concept_ids = list(self.concept_index.keys())
        batch_size = 256
        
        for i in range(0, len(concept_ids), batch_size):
            batch_ids = concept_ids[i:i+batch_size]
            batch_texts = [
                f"{self.concept_index[cid]['type']}: {self.concept_index[cid]['name']}"
                for cid in batch_ids
            ]
            
            batch_embeddings = self.embedding_model.encode(batch_texts, show_progress_bar=False)
            
            for concept_id, embedding in zip(batch_ids, batch_embeddings):
                embeddings[concept_id] = embedding
            
            if (i + batch_size) % 10000 == 0:
                print(f"  Processed {min(i+batch_size, len(concept_ids))}/{len(concept_ids)} concepts")
        
        with open(cache_file, 'wb') as f:
            pickle.dump(embeddings, f)
        
        print(f" Generated embeddings for {len(embeddings)} concepts")
        return embeddings
    
    def verify_atomic_facts(self, atomic_facts: List, threshold: float = 0.6) -> Dict:
        if not atomic_facts:
            return {
                "verified_count": 0,
                "total_count": 0,
                "verification_score": 0.0,
                "fact_results": []
            }
        
        fact_results = []
        verified_count = 0
        
        for i, fact in enumerate(atomic_facts):
            fact_text = fact.text if hasattr(fact, 'text') else str(fact)
            
            # Verify single fact
            result = self._verify_single_fact(fact_text, threshold)
            result['fact_index'] = i
            result['fact_text'] = fact_text
            
            fact_results.append(result)
            
            if result['verified']:
                verified_count += 1
        
        # Calculate overall score
        verification_score = verified_count / len(atomic_facts) if atomic_facts else 0.0
        
        return {
            "verified_count": verified_count,
            "total_count": len(atomic_facts),
            "verification_score": verification_score,
            "fact_results": fact_results
        }
    def _verify_single_fact_impl(self, fact_text, threshold):
        """
        Core verification logic. Assumes fact_embedding is already in
        self._embedding_cache[fact_text] — computed by _verify_single_fact.
        """
        fact_embedding = self._embedding_cache[fact_text]

        matched_concepts = self._find_concepts_in_text(fact_text, fact_embedding, threshold)

        if not matched_concepts:
            return {
                "verified": False,
                "reason": "no_concepts_matched",
                "confidence": 0.0,
                "matched_concepts": [],
                "relationships": []
            }

        avg_confidence = np.mean([c['similarity'] for c in matched_concepts])

        # Two or more concepts — check for relationships
        if len(matched_concepts) >= 2:
            relationship_verified, relationships = self._verify_relationships(
                matched_concepts, fact_text
            )

            if relationship_verified:
                return {
                    "verified": True,
                    "reason": "relationship_in_kg",
                    "confidence": float(avg_confidence),
                    "matched_concepts": matched_concepts,
                    "relationships": relationships
                }
            elif avg_confidence >= threshold:
                # Both concepts exist with high confidence — likely valid
                return {
                    "verified": True,
                    "reason": "concepts_exist_high_confidence",
                    "confidence": float(avg_confidence),
                    "matched_concepts": matched_concepts,
                    "relationships": []
                }
            else:
                return {
                    "verified": False,
                    "reason": "low_confidence_concepts",
                    "confidence": float(avg_confidence),
                    "matched_concepts": matched_concepts,
                    "relationships": []
                }

        # Single concept — verify it exists with sufficient confidence
        if len(matched_concepts) == 1:
            if matched_concepts[0]['similarity'] >= threshold:
                return {
                    "verified": True,
                    "reason": "concept_exists",
                    "confidence": float(matched_concepts[0]['similarity']),
                    "matched_concepts": matched_concepts,
                    "relationships": []
                }
            else:
                return {
                    "verified": False,
                    "reason": "low_confidence_concept",
                    "confidence": float(matched_concepts[0]['similarity']),
                    "matched_concepts": matched_concepts,
                    "relationships": []
                }

        # Fallback
        return {
            "verified": False,
            "reason": "unknown_error",
            "confidence": 0.0,
            "matched_concepts": matched_concepts,
            "relationships": []
        }
    
    def _verify_single_fact(self, fact_text, threshold):
        """Cached wrapper around the actual verification logic."""
        if fact_text not in self._embedding_cache:
            self._embedding_cache[fact_text] = self.embedding_model.encode(fact_text)
        
        cache_key = (fact_text.lower().strip(), round(threshold, 2))
        if cache_key in self._fact_cache:
            return self._fact_cache[cache_key]
        
        result = self._verify_single_fact_impl(fact_text, threshold)
        self._fact_cache[cache_key] = result
        return result
    
    def _find_concepts_in_text(self, text: str, text_embedding: np.ndarray, 
                            threshold: float, top_k: int = 5) -> List[Dict]:
        text_lower = text.lower()
        matched_concepts = []
        for concept_id, concept_data in self.concept_index.items():
            for synonym in concept_data['synonyms']:
                if synonym and len(synonym) > 3 and synonym in text_lower:
                    similarity = self._compute_similarity(text_embedding, concept_id)
                    
                    if similarity >= threshold * 0.8:
                        matched_concepts.append({
                            'concept_id': concept_id,
                            'name': concept_data['name'],
                            'type': concept_data['type'],
                            'similarity': float(similarity),
                            'match_method': 'exact'
                        })
                        break
        
        if len(matched_concepts) < 2:
            semantic_matches = self._semantic_concept_search(
                text_embedding, threshold * 0.85, top_k=top_k  #  Lower threshold
            )
            
            # Add semantic matches not already found
            existing_ids = {c['concept_id'] for c in matched_concepts}
            for match in semantic_matches:
                if match['concept_id'] not in existing_ids:
                    match['match_method'] = 'semantic'
                    matched_concepts.append(match)
        
        # Sort by similarity
        matched_concepts.sort(key=lambda x: x['similarity'], reverse=True)
        
        return matched_concepts[:top_k]
    
    def _semantic_concept_search(self, query_embedding: np.ndarray, 
                                  threshold: float, top_k: int = 5) -> List[Dict]:
        """Search for concepts using semantic similarity"""
        similarities = []
        
        for concept_id in self.concept_embeddings:
            similarity = self._compute_similarity(query_embedding, concept_id)
            
            if similarity >= threshold:
                similarities.append((concept_id, similarity))
        
        # Sort and take top k
        similarities.sort(key=lambda x: x[1], reverse=True)
        top_matches = similarities[:top_k]
        
        results = []
        for concept_id, similarity in top_matches:
            concept_data = self.concept_index[concept_id]
            results.append({
                'concept_id': concept_id,
                'name': concept_data['name'],
                'type': concept_data['type'],
                'similarity': float(similarity)
            })
        
        return results
    
    def _compute_similarity(self, query_embedding: np.ndarray, concept_id: str) -> float:
        """Compute cosine similarity between query and concept"""
        concept_embedding = self.concept_embeddings.get(concept_id)
        
        if concept_embedding is None:
            return 0.0
        
        return np.dot(query_embedding, concept_embedding) / (
            np.linalg.norm(query_embedding) * np.linalg.norm(concept_embedding)
        )
    
    def _verify_relationships(self, matched_concepts: List[Dict], 
                              fact_text: str) -> Tuple[bool, List[Dict]]:

        relationships = []
        
        # Check all pairs of concepts
        for i, concept1 in enumerate(matched_concepts):
            for concept2 in matched_concepts[i+1:]:
                c1_id = concept1['concept_id']
                c2_id = concept2['concept_id']
                
                # Check forward direction
                forward_key = (c1_id, c2_id)
                if forward_key in self.relation_index:
                    for relation in self.relation_index[forward_key]:
                        relationships.append({
                            'source': concept1['name'],
                            'target': concept2['name'],
                            'relation': relation,
                            'direction': 'forward'
                        })
                
                # Check reverse direction
                reverse_key = (c2_id, c1_id)
                if reverse_key in self.relation_index:
                    for relation in self.relation_index[reverse_key]:
                        relationships.append({
                            'source': concept2['name'],
                            'target': concept1['name'],
                            'relation': relation,
                            'direction': 'reverse'
                        })
        
        return len(relationships) > 0, relationships
    
    def get_statistics(self) -> Dict:
        """Get statistics about the loaded knowledge graph"""
        return {
            "total_concepts": len(self.concept_index),
            "total_relationships": len(self.kg_df),
            "unique_relations": self.kg_df['relation'].nunique(),
            "concept_types": self.kg_df['x_type'].value_counts().to_dict()
        }

In [ ]:
class AtomicFact:
    """Data class for atomic facts"""
    
    def __init__(self, text, category, source_sentence):
        self.text = text
        self.category = category
        self.source_sentence = source_sentence
        self.llm_score = 0.0
        self.kb_score = 0.0
    
    def __repr__(self):
        return f"AtomicFact(text='{self.text[:50]}...', category='{self.category}')"

In [ ]:
class MedicalKnowledgeGraph(PrimeKGVerifier):
    """
    Drop-in replacement for original MedicalKnowledgeGraph.
    Uses PrimeKG for verification.
    """
    
    def __init__(self, primekg_path, cache_dir = "kg_cache"):
        super().__init__(primekg_path, cache_dir)
    
    def verify_fact(self, fact, threshold: float = 0.75) -> float:
        """
        Verify a single fact (backward compatibility).
        
        Returns:
            Confidence score between 0.0 and 1.0
        """
        result = self._verify_single_fact(
            fact.text if hasattr(fact, 'text') else str(fact),
            threshold
        )
        return result['confidence']


class FactualRewardCalculator:
    """Calculate factual rewards from verified facts"""
    
    def __init__(self, agreement_threshold=0.5):
        self.agreement_threshold = agreement_threshold
    
    def compute_factual_reward(self, facts_list):
        """Compute aggregate factual reward"""
        if not facts_list:
            return {
                "factual_reward": 0.0,
                "individual_rewards": [],
                "agreement_rate": 0.0,
                "avg_llm_score": 0.0,
                "avg_kb_score": 0.0,
                "num_facts": 0
            }
        
        # Convert to AtomicFact if needed
        atomic_facts = []
        for fact_item in facts_list:
            if isinstance(fact_item, AtomicFact):
                atomic_facts.append(fact_item)
            else:
                fact = AtomicFact(
                    text=fact_item.get("text", ""),
                    category=fact_item.get("category", "GENERAL"),
                    source_sentence=fact_item.get("source_sentence", "")
                )
                fact.llm_score = fact_item.get("llm_score", 0.0)
                fact.kb_score = fact_item.get("kb_score", 0.0)
                atomic_facts.append(fact)
        
        # Calculate rewards
        individual_rewards = []
        agreement_count = 0
        llm_scores = []
        kb_scores = []
        
        for fact in atomic_facts:
            # Weighted combination: 70% LLM, 30% KB
            fact_reward = 0.7 * fact.llm_score + 0.3 * fact.kb_score
            
            # Bonus for agreement
            if abs(fact.llm_score - fact.kb_score) <= self.agreement_threshold:
                agreement_count += 1
            else:
                fact_reward *= 0.9
            
            individual_rewards.append(fact_reward)
            llm_scores.append(fact.llm_score)
            kb_scores.append(fact.kb_score)
        
        # Aggregates
        factual_reward = sum(individual_rewards) / len(individual_rewards)
        agreement_rate = agreement_count / len(atomic_facts)
        
        return {
            "factual_reward": factual_reward,
            "individual_rewards": individual_rewards,
            "agreement_rate": agreement_rate,
            "avg_llm_score": sum(llm_scores) / len(llm_scores),
            "avg_kb_score": sum(kb_scores) / len(kb_scores),
            "num_facts": len(atomic_facts)
        }


In [ ]:
class AtomicFactVerificationSystem:
    """
    Dual-source fact verification with lazy loading
    """
    
    def __init__(self,
                 agreement_threshold=0.5,
                 verification_model_id="Qwen/Qwen2.5-14B-Instruct",
                 device_map="auto",
                 torch_dtype="auto",
                 use_4bit=True,
                 primekg_path="../Datasets/kg.csv" ):
        print("Initializing Atomic Fact Verification System...")
        
        # Store config for lazy loading
        self.verification_model_id = verification_model_id
        self.device_map = device_map
        self.torch_dtype = torch_dtype
        self.use_4bit = use_4bit
        self.agreement_threshold = agreement_threshold
        
        self.llm_verifier = None
        
        self.kg_verifier = MedicalKnowledgeGraph(primekg_path="../Datasets/kg.csv")

        self._fact_cache = {}
        self._embedding_cache = {}
        
        # Reward calculator
        self.reward_calculator = FactualRewardCalculator(agreement_threshold)
        
        # Start in training mode (fast)
        self.training_mode = True
        
        print(" Verification system initialized (LLM lazy loaded)")


    def verify_reasoning(self, reasoning_text, context="", step=0, warmup_steps=100):
        try:
            # Ensure training_mode exists
            if not hasattr(self, 'training_mode') or self.training_mode is None:
                self.training_mode = True

            # Ensure fact cache exists (O3)
            if not hasattr(self, '_fact_cache'):
                self._fact_cache = {}

            # Check for <facts> tags
            if '<facts>' not in reasoning_text.lower():
                return {
                    'factual_analysis': {
                        'factual_reward': 0.0,
                        'individual_rewards': [],
                        'num_facts': 0
                    },
                    'facts': [],
                    'error': 'No <facts> tags found'
                }

            # Extract facts content
            facts_match = re.search(r'<facts>(.*?)</facts>', reasoning_text,
                                    re.DOTALL | re.IGNORECASE)
            if not facts_match:
                return {
                    'factual_analysis': {'factual_reward': 0.0, 'num_facts': 0},
                    'facts': [],
                    'error': 'Could not parse <facts> tags'
                }

            facts_content = facts_match.group(1).strip()

            # Parse numbered facts
            pattern = r'(\d+)\s*[.):•-]?\s*([^\n\r]+(?:\n(?!\s*\d+\s*[.):•-])[^\n\r]+)*)'
            matches = re.finditer(pattern, facts_content, re.MULTILINE)

            facts_list = []
            for match in matches:
                fact_text = ' '.join(match.group(2).strip().split())
                if len(fact_text) >= 20:
                    facts_list.append(fact_text)

            facts_list = facts_list[:10]

            if not facts_list:
                return {
                    'factual_analysis': {'factual_reward': 0.0, 'num_facts': 0},
                    'facts': [],
                    'error': 'No valid facts extracted'
                }

            atomic_facts = [
                AtomicFact(
                    text=fact_text,
                    category=self._categorize_fact(fact_text),
                    source_sentence=fact_text
                )
                for fact_text in facts_list
            ]

            use_full_verification = (
                not self.training_mode and step >= warmup_steps
            )

            if not use_full_verification:
                # Fast path: heuristic scores for both LLM and KB
                for fact in atomic_facts:
                    fact.llm_score = self._get_training_heuristic_score(fact)
                    fact.kb_score = self._get_training_heuristic_score(fact)

            else:
                # ── O2: Single combined LLM call for extraction + scoring ──
                # Collect facts not already in cache
                uncached_facts = [
                    f for f in atomic_facts
                    if f.text.lower().strip() not in self._fact_cache
                ]

                if uncached_facts:
                    try:
                        if self.llm_verifier is None:
                            self._ensure_llm_loaded()

                        # Build one combined prompt for all uncached facts
                        facts_block = "\n".join(
                            f"{i+1}. {f.text}"
                            for i, f in enumerate(uncached_facts)
                        )
                        combined_prompt = f"""You are a medical fact verification specialist.

    Score each medical fact below for clinical accuracy (0.0-1.0).
    - 1.0: Definitely correct, established medical knowledge
    - 0.7-0.9: Likely correct, standard medical practice  
    - 0.4-0.6: Uncertain or context-dependent
    - 0.0-0.3: Likely incorrect, contradicts medical evidence

    FACTS:
    {facts_block}

    CONTEXT: {context if context else "None"}

    Return ONLY a JSON array, one entry per fact, in the same order:
    [{{"fact_index": 1, "confidence_score": 0.95}}, ...]"""

                        outputs = self.llm_verifier.pipe(
                            combined_prompt,
                            max_new_tokens=400,
                            temperature=0.3,
                            do_sample=True,
                            return_full_text=False
                        )
                        response_text = outputs[0]["generated_text"]

                        # Parse JSON array response
                        json_match = re.search(r'\[.*\]', response_text, re.DOTALL)
                        if json_match:
                            scored = json.loads(json_match.group(0))
                            score_map = {
                                item['fact_index']: float(item.get('confidence_score', 0.5))
                                for item in scored
                            }
                        else:
                            score_map = {}

                        # Assign LLM scores and populate cache
                        for i, fact in enumerate(uncached_facts):
                            fact.llm_score = score_map.get(i + 1, 0.5)
                            # Cache will be completed after KB scoring below

                    except Exception as e:
                        print(f"Batched LLM scoring error: {e}")
                        for fact in uncached_facts:
                            fact.llm_score = 0.5

                # Apply cached LLM scores to cached facts
                for fact in atomic_facts:
                    cache_key = fact.text.lower().strip()
                    if cache_key in self._fact_cache:
                        fact.llm_score = self._fact_cache[cache_key]['llm_score']
                        fact.kb_score = self._fact_cache[cache_key]['kb_score']

                # ── O3: PrimeKG batch verification with cache ──
                # Only verify facts not already cached
                uncached_for_kg = [
                    f for f in atomic_facts
                    if f.text.lower().strip() not in self._fact_cache
                ]

                if uncached_for_kg:
                    try:
                        kg_results = self.kg_verifier.verify_atomic_facts(
                            uncached_for_kg, threshold=0.75
                        )
                        for fact, result in zip(
                            uncached_for_kg, kg_results['fact_results']
                        ):
                            fact.kb_score = result.get('confidence', 0.5)

                            # Store in cache (O3)
                            self._fact_cache[fact.text.lower().strip()] = {
                                'llm_score': fact.llm_score,
                                'kb_score': fact.kb_score
                            }

                    except Exception as e:
                        print(f"PrimeKG batch verification error: {e}")
                        for fact in uncached_for_kg:
                            fact.kb_score = 0.5
                            self._fact_cache[fact.text.lower().strip()] = {
                                'llm_score': fact.llm_score,
                                'kb_score': 0.5
                            }

            # ── Compute aggregate reward ──
            individual_rewards = []
            llm_scores = []
            kb_scores = []

            for fact in atomic_facts:
                fact_reward = 0.7 * fact.llm_score + 0.3 * fact.kb_score
                individual_rewards.append(fact_reward)
                llm_scores.append(fact.llm_score)
                kb_scores.append(fact.kb_score)

            factual_reward = sum(individual_rewards) / len(individual_rewards)

            factual_analysis = {
                'factual_reward': factual_reward,
                'individual_rewards': individual_rewards,
                'agreement_rate': sum(
                    1 for f in atomic_facts
                    if abs(f.llm_score - f.kb_score) <= self.agreement_threshold
                ) / len(atomic_facts),
                'avg_llm_score': sum(llm_scores) / len(llm_scores),
                'avg_kb_score': sum(kb_scores) / len(kb_scores),
                'num_facts': len(atomic_facts)
            }

            facts_as_dicts = [
                {
                    'text': f.text,
                    'category': f.category,
                    'llm_score': f.llm_score,
                    'kb_score': f.kb_score,
                    'source_sentence': f.source_sentence
                }
                for f in atomic_facts
            ]

            return {
                'factual_analysis': factual_analysis,
                'facts': facts_as_dicts,
                'error': None
            }

        except Exception as e:
            print(f"❌ FATAL ERROR in verify_reasoning: {e}")
            import traceback
            traceback.print_exc()

            return {
                'factual_analysis': {
                    'factual_reward': 0.0,
                    'individual_rewards': [],
                    'agreement_rate': 0.0,
                    'avg_llm_score': 0.0,
                    'avg_kb_score': 0.0,
                    'num_facts': 0
                },
                'facts': [],
                'error': f"Fatal error: {str(e)}"
            }
    
    def _get_training_heuristic_score(self, fact):
        """
        Enhanced heuristic scoring for training mode.
        Estimates medical fact quality without expensive LLM calls.
        """
        text = fact.text
        text_lower = text.lower()
        
        # Base score from category
        category_base = {
            'DIAGNOSIS': 0.75,
            'TREATMENT': 0.75, 
            'MECHANISM': 0.80,
            'ANATOMY': 0.85,
            'SYMPTOM': 0.70,
            'GENERAL': 0.65
        }
        score = category_base.get(fact.category, 0.7)
                
        # Medical terminology and specificity
        if re.search(r'\d+\s*(mg|ml|mmol|units|hours|days|weeks|months|years|%)', text_lower):
            score += 0.08  # Specific measurements/doses
        
        if re.search(r'(stage|grade|class|type)\s+[IVX0-9]+', text_lower):
            score += 0.07  # Classification systems
        
        # Strong medical terms
        strong_terms = [
            'is characterized by', 'causes', 'results in', 'indicated for',
            'first-line treatment', 'contraindicated', 'diagnostic criteria',
            'pathognomonic', 'definitive', 'gold standard'
        ]
        if any(term in text_lower for term in strong_terms):
            score += 0.12
        
        # Clinical language
        clinical_terms = [
            'diagnosis', 'treatment', 'therapy', 'disease', 'condition',
            'symptom', 'presentation', 'management', 'prognosis'
        ]
        if any(term in text_lower for term in clinical_terms):
            score += 0.05
        
        # === NEGATIVE INDICATORS ===
        
        # Case-specific language (should be general knowledge)
        case_specific = [
            'the patient', 'this patient', 'the case', 'this case',
            'the individual', 'shows', 'presents with', 'has been',
            'the finding', 'this finding', 'suggests that'
        ]
        if any(term in text_lower for term in case_specific):
            score -= 0.25  # Major penalty
        
        # Uncertainty markers
        uncertain = ['may', 'might', 'could', 'possibly', 'sometimes',
                    'unclear', 'debated', 'controversial']
        if any(w in text_lower for w in uncertain):
            score -= 0.15
        
        # Vague language
        vague = ['various', 'several', 'many', 'some', 'often', 'usually']
        vague_count = sum(1 for v in vague if v in text_lower)
        if vague_count > 0:
            score -= 0.08 * vague_count
        
        # Absolute statements (medical facts are rarely absolute)
        absolutes = ['always', 'never', 'all patients', 'no patients', 
                    'definitely', 'guaranteed', '100%']
        if any(abs_term in text_lower for abs_term in absolutes):
            score -= 0.20
        
        # === LENGTH CHECKS ===
        
        length = len(text)
        if length < 25:
            score -= 0.25  # Too short
        elif length < 40:
            score -= 0.15  # Questionable brevity
        elif length > 250:
            score -= 0.12  # Too verbose
        
        # === STRUCTURE CHECKS ===
        
        # Multiple sentences in one "atomic" fact
        if text.count('.') > 2:
            score -= 0.15
        
        # No proper capitalization
        if not text[0].isupper():
            score -= 0.08
        
        # Missing punctuation
        if not text.rstrip().endswith(('.', '!', '?')):
            score -= 0.05
        
        # Parenthetical clarifications (good)
        if '(' in text and ')' in text:
            score += 0.05
        
        # Clamp to reasonable range
        return max(0.1, min(1.0, score))
        
    def _ensure_llm_loaded(self):
        """Lazy load LLM verifier when needed"""
        if self.llm_verifier is None:
            print("\nLoading 14B verification model...")
            self.llm_verifier = LLMJudgeVerifier(
                model_id=self.verification_model_id,
                device_map=self.device_map,
                torch_dtype=self.torch_dtype,
                use_4bit=self.use_4bit
            )
            print(" Verification model loaded")

    def process_response(self, reasoning_text, context=""):
        """Delegates to verify_reasoning for evaluation (training_mode=False)."""
        original_mode = self.training_mode
        self.training_mode = False
        try:
            result = self.verify_reasoning(
                reasoning_text, context,
                step=999999,      # force full verification
                warmup_steps=100
            )
        finally:
            self.training_mode = original_mode

        # Remap keys to match old process_response return format
        return {
            "facts": result["facts"],
            "factual_analysis": result["factual_analysis"],
            "error": result["error"]
        }
    
    def _parse_facts_from_tags(self, reasoning_text):
        """Extract numbered facts from <facts>...</facts>"""
        facts_match = re.search(r'<facts>(.*?)</facts>', reasoning_text, 
                               re.DOTALL | re.IGNORECASE)
        
        if not facts_match:
            return []
        
        facts_content = facts_match.group(1).strip()
        
        # Pattern: 1. Fact text
        pattern = r'(?:^|\n)\s*(\d+)\s*[.):]\s*(.+?)(?=(?:\n\s*\d+\s*[.):]\s*)|$)'
        matches = re.finditer(pattern, facts_content, re.DOTALL)
        
        facts_list = []
        for match in matches:
            fact_text = match.group(2).strip()
            fact_text = ' '.join(fact_text.split())
            
            if len(fact_text) > 20:
                facts_list.append(fact_text)
        
        print(f" Parsed {len(facts_list)} facts")
        return facts_list[:15]
    
    def _categorize_fact(self, fact_text):
        """Simple keyword-based categorization"""
        fact_lower = fact_text.lower()
        
        if any(w in fact_lower for w in ['diagnos', 'disease', 'condition']):
            return 'DIAGNOSIS'
        elif any(w in fact_lower for w in ['treat', 'therapy', 'medication']):
            return 'TREATMENT'
        elif any(w in fact_lower for w in ['cause', 'mechanism', 'pathway']):
            return 'MECHANISM'
        elif any(w in fact_lower for w in ['symptom', 'present', 'sign']):
            return 'SYMPTOM'
        elif any(w in fact_lower for w in ['anatomy', 'structure', 'organ']):
            return 'ANATOMY'
        else:
            return 'GENERAL'
    
    def _fact_to_dict(self, fact):
        """Convert AtomicFact to dictionary"""
        return {
            "text": fact.text,
            "category": fact.category,
            "llm_score": fact.llm_score,
            "kb_score": fact.kb_score,
            "source_sentence": fact.source_sentence
        }

In [ ]:
class RewardModel:    def __init__(self,
                 # Weights
                 w_accuracy=1.0,
                 w_leak=0.3,
                 w_preamble=0.2,
                 w_grounded=0.5,
                 # Soft gate parameters
                 tau_veracity=0.4,
                 k_sharpness=8.0,
                 below_tau_slope=0.1,
                 # Leak detection thresholds
                 tau_leak=0.7,
                 tau_preamble_words=15,
                 # Violation tracking thresholds
                 violation_threshold_leak=0.1,
                 violation_threshold_preamble=0.3,
                 violation_threshold_factual=0.2,
                 # Model configuration
                 agreement_threshold=0.5,
                 verification_model=None,
                 embedding_model='all-MiniLM-L6-v2',
                 use_4bit_verification=True,
                 device_map={"": 0},
                 torch_dtype="auto"):
        
        # ── Weights ──
        self.w_accuracy = w_accuracy
        self.w_leak = w_leak
        self.w_preamble = w_preamble
        self.w_grounded = w_grounded
        
        # ── Soft gate parameters ──
        self.tau_veracity = tau_veracity      # minimum factuality bar
        self.k_sharpness = k_sharpness        # sigmoid steepness around τ
        self.below_tau_slope = below_tau_slope # gradient signal below threshold
        
        # ── Detection thresholds ──
        self.tau_leak = tau_leak
        self.tau_preamble_words = tau_preamble_words
        
        # ── Violation tracking ──
        self.violation_threshold_leak = violation_threshold_leak
        self.violation_threshold_preamble = violation_threshold_preamble
        self.violation_threshold_factual = violation_threshold_factual
        
        # ── Embedding model for leak detection ──
        self.sentence_model = embedding_manager.get_model(embedding_model)
        leak_phrases = [
            "the correct answer is", "the answer is definitely", "the choice is clearly",
            "option A is the right one", "option B is the right one", 
            "option C is the right one", "option D is the right one",
            "we can conclude the answer is", "therefore the answer is",
            "so the correct choice is", "the final answer is",
            "answer: A", "answer: B", "answer: C", "answer: D"
        ]
        self.leak_embeddings = self.sentence_model.encode(leak_phrases)
        
        # ── Fact verification system ──
        print(f"\nInitializing Fact Verification System...")
        print(f"  Verification model: {verification_model}")
        print(f"  4-bit quantization: {use_4bit_verification}")
        
        if verification_model is not None:
            self.fact_verification_system = AtomicFactVerificationSystem(
                agreement_threshold=agreement_threshold,
                verification_model_id=verification_model,
                device_map=device_map,
                torch_dtype=torch_dtype,
                use_4bit=use_4bit_verification,
                primekg_path="../Datasets/kg.csv")
        else:
            self.fact_verification_system = None
            print("Verification system disabled - training_mode only")
    
    # ==================== SOFT GATE ====================
    
    def _sigmoid_gate(self, r_grounded):
        """
        Smooth on/off switch centered at τ_veracity.
        
        σ(k * (R_grounded - τ)) → 
            ≈ 0 when R_grounded << τ  (reward nearly zeroed)
            = 0.5 when R_grounded = τ  (half reward)
            ≈ 1 when R_grounded >> τ  (full reward passes through)
        
        k_sharpness controls the transition:
            High k → approaches hard gate (risky for GRPO)
            Low k  → gradual transition (better gradient flow)
        """
        x = self.k_sharpness * (r_grounded - self.tau_veracity)
        return 1.0 / (1.0 + np.exp(-x))
    
    # ==================== CORE REWARD COMPUTATION ====================
    
    def compute_total_reward(self, generation, correct_answer, context="", step=0, warmup_steps=100):
            """
            Main reward computation using multiplicative gating.
            
            Architecture:
                1. Format check → hard -1 if invalid
                2. R_accuracy (binary correctness)
                3. p_answerLeak, P_preamble (continuous penalties → multiplicative gates)
                4. R_grounded (factual verification → soft sigmoid gate)
                5. Compose: R = R_accuracy · G_leak · G_preamble · σ(k(R_grounded - τ))
                            + bonus below threshold for gradient signal
            """
            if not self.validate_format(generation):
                return self._format_error_response()
            
            # ── Component scores ──
            r_accuracy = self._compute_accuracy_reward(generation, correct_answer)
            p_answerLeak = self._compute_leak_penalty(generation)
            p_preamble = self._compute_preamble_penalty(generation)
            factual_result = self._compute_grounded_reward(generation, context, step=step, warmup_steps=warmup_steps)
            r_grounded = factual_result["factual_reward"]
            
            # Multiplicative gates on correctness only
            g_leak = 1.0 - self.w_leak * p_answerLeak
            g_preamble = (1.0 - self.w_preamble) if p_preamble > 0 else 1.0

            # Soft factuality gate (independent)
            g_veracity = self._sigmoid_gate(r_grounded)

            # Compose: two independent additive signals
            r_correctness = r_accuracy * g_leak * g_preamble
            r_factual = r_accuracy * self.w_grounded * g_veracity

            r_total = r_correctness + r_factual
            
            return {
                'r_accuracy': r_accuracy,
                'r_grounded': r_grounded,
                'p_answerLeak': p_answerLeak,
                'p_preamble': p_preamble,
                'g_leak': g_leak,
                'g_preamble': g_preamble,
                'g_veracity': g_veracity,
                'r_total': r_total,
                'r_normalized': r_total,
                'factual_analysis': factual_result["factual_analysis"],
                'extracted_facts': factual_result.get("facts", []),
                'factual_error': factual_result.get("error"),
                'format_valid': True
            }
        
    def validate_format(self, generation):
        """Check <facts>...</facts><answer>X</answer> format"""
        think_match = re.search(r'<facts?>\s*(.+?)\s*</facts?>', generation, 
                                re.DOTALL | re.IGNORECASE)
        if not think_match or len(think_match.group(1).strip()) < 5:
            return False
        
        answer_match = re.search(r'<answer>\s*(.*?)\s*</answer>', generation, 
                                 re.IGNORECASE | re.DOTALL)
        if not answer_match:
            return False
        
        answer_letters = re.findall(r'\b([A-D])\b', answer_match.group(1).strip(), 
                                     re.IGNORECASE)
        return len(answer_letters) == 1
    
    def extract_answer_choice(self, generation):
        """Extract answer letter from <answer> tags"""
        match = re.search(r'<answer>\s*([A-D])\s*</answer>', generation, 
                         re.IGNORECASE)
        return match.group(1).upper() if match else None
    
    def extract_think_content(self, generation):
        """Extract reasoning from <facts> tags"""
        match = re.search(r'<facts>\s*(.+?)\s*</facts>', generation, 
                         re.DOTALL | re.IGNORECASE)
        return match.group(1).strip() if match else ""
        
    def _compute_accuracy_reward(self, generation, correct_answer):
        """R_accuracy: 1.0 if correct answer, 0.0 otherwise"""
        predicted = self.extract_answer_choice(generation)
        return 1.0 if predicted == correct_answer.upper() else 0.0
    
    def _compute_leak_penalty(self, generation):
        """
        p_answerLeak: Detect answer leakage in reasoning.
        
        Combines:
            1. Semantic similarity to known leak phrases
            2. Regex pattern matching for explicit answer statements
            3. Early-answer detection (answer stated in first quarter)
        
        Returns [0, 1] penalty. Applied as gate: G_leak = 1 - w_leak * p_answerLeak
        """
        think_content = self.extract_think_content(generation)
        if not think_content:
            return 0.0
        
        # Semantic similarity to leak phrases
        think_embedding = self.sentence_model.encode([think_content])
        similarities = cosine_similarity(think_embedding, self.leak_embeddings)[0]
        semantic_sim = float(np.max(similarities))
        
        # Pattern matching
        patterns = [
            r'\b(answer|choice|option)\s*:?\s*([A-D])\b',
            r'\b([A-D])\s+is\s+(correct|right|the answer)',
            r'\bcorrect\s+answer\s+is\s+([A-D])\b',
            r'^([A-D])[.)]\s*',
        ]
        pattern_penalty = 1.0 if any(re.search(p, think_content, re.IGNORECASE) 
                                      for p in patterns) else 0.0
        
        # Early answer detection (first quarter of reasoning)
        words = think_content.split()
        early_penalty = 0.0
        if len(words) > 5:
            first_quarter = ' '.join(words[:len(words)//4])
            if any(re.search(p, first_quarter, re.IGNORECASE) for p in patterns[:3]):
                early_penalty = 0.5
        
        total_penalty = max(semantic_sim, pattern_penalty, early_penalty)
        return total_penalty if total_penalty > self.tau_leak else 0.0
    
    def _compute_preamble_penalty(self, generation):
        """
        P_preamble: Penalize content outside <facts>/<answer> structure.
        
        Checks:
            1. Excessive text before <facts> tag (> tau_preamble_words)
            2. Content after </answer> tag (> 5 words)
            3. Reasoning between </facts> and <answer> (> 10 words)
        
        Returns 1.0 if any violation, 0.0 otherwise.
        Applied as gate: G_preamble = (1 - w_preamble) if violation else 1.0
        """
        # Pre-think content
        think_start = re.search(r'<facts>', generation, re.IGNORECASE)
        if think_start:
            pre_think = generation[:think_start.start()].strip()
            if len(pre_think.split()) > self.tau_preamble_words:
                return 1.0
        
        # Post-answer content
        answer_end = re.search(r'</answer>', generation, re.IGNORECASE)
        if answer_end:
            post_answer = generation[answer_end.end():].strip()
            if len(post_answer.split()) > 5:
                return 1.0
        
        # Reasoning between tags
        think_match = re.search(r'<facts>(.*?)</facts>', generation, 
                               re.DOTALL | re.IGNORECASE)
        if think_match:
            answer_start = re.search(r'<answer>', generation, re.IGNORECASE)
            if answer_start:
                between = generation[think_match.end():answer_start.start()].strip()
                if len(between.split()) > 10:
                    return 1.0
        
        return 0.0
    
    def _compute_grounded_reward(self, generation, context, step=0, warmup_steps=100):
        try:
            # Just check that facts exist, don't strip them
            think_content = self.extract_think_content(generation)
            if not think_content:
                return {"factual_reward": 0.0, "factual_analysis": {},
                        "facts": [], "error": "No reasoning found"}
    
            if self.fact_verification_system is None:
                return {"factual_reward": 0.7, "factual_analysis": {},
                        "facts": [], "error": None}
    
            # Pass FULL generation so verify_reasoning can find <facts> tags
            result = self.fact_verification_system.verify_reasoning(
                generation, context,
                step=step, warmup_steps=warmup_steps
            )
            return {
                "factual_reward": result.get("factual_analysis", {}).get("factual_reward", 0.0),
                "factual_analysis": result.get("factual_analysis", {}),
                "facts": result.get("facts", []),
                "error": result.get("error")
            }
        except Exception as e:
            return {"factual_reward": 0.0, "factual_analysis": {},
                    "facts": [], "error": f"Verification failed: {e}"}
    
    # ==================== UTILITIES ====================
    
    def _format_error_response(self):
        """Return response for invalid format — hard -1"""
        return {
            'r_accuracy': -1.0, 'r_grounded': 0.0,
            'p_answerLeak': 0.0, 'p_preamble': 1.0,
            'g_leak': 1.0, 'g_preamble': 0.0, 'g_veracity': 0.0,
            'r_total': -1.0, 'r_normalized': -1.0,
            'factual_analysis': {}, 'extracted_facts': [],
            'factual_error': "Invalid format", 'format_valid': False,
            # Legacy aliases
            'r_binary': -1.0, 'p_answer': 0.0,
            'p_structural': 1.0, 'r_factual': 0.0,
        }
    
    def calculate_hacking_rate(self, responses_with_rewards):
        """Calculate violation rates across a batch of responses"""
        if not responses_with_rewards:
            return self._empty_hacking_stats()
        
        valid_responses = [r for r in responses_with_rewards 
                          if r.get('reward_info', {}).get('format_valid', True)]
        
        if not valid_responses:
            return self._empty_hacking_stats()
        
        n = len(valid_responses)
        
        leak_violations = sum(
            1 for r in valid_responses 
            if r['reward_info'].get('p_answerLeak', r['reward_info'].get('p_answer', 0)) 
               > self.violation_threshold_leak
        )
        preamble_violations = sum(
            1 for r in valid_responses 
            if r['reward_info'].get('p_preamble', r['reward_info'].get('p_structural', 0)) 
               > self.violation_threshold_preamble
        )
        factual_violations = sum(
            1 for r in valid_responses 
            if r['reward_info'].get('r_grounded', r['reward_info'].get('r_factual', 1.0)) 
               < self.violation_threshold_factual
        )
        
        return {
            'total_responses': len(responses_with_rewards),
            'valid_responses': n,
            'leak_violation_count': leak_violations,
            'preamble_violation_count': preamble_violations,
            'factual_violation_count': factual_violations,
            'leak_violation_rate': leak_violations / n,
            'preamble_violation_rate': preamble_violations / n,
            'factual_violation_rate': factual_violations / n,
            'overall_violation_rate': (leak_violations + preamble_violations + factual_violations) / (n * 3),
            # Legacy aliases
            'answer_violation_count': leak_violations,
            'structural_violation_count': preamble_violations,
            'answer_violation_rate': leak_violations / n,
            'structural_violation_rate': preamble_violations / n,
        }
    
    def _empty_hacking_stats(self):
        keys = [
            'total_responses', 'valid_responses',
            'leak_violation_count', 'preamble_violation_count', 'factual_violation_count',
            'leak_violation_rate', 'preamble_violation_rate', 'factual_violation_rate',
            'overall_violation_rate',
            'answer_violation_count', 'structural_violation_count',
            'answer_violation_rate', 'structural_violation_rate',
        ]
        return {k: 0 if 'count' in k or 'responses' in k else 0.0 for k in keys}

# Data Processor

In [ ]:
class MedQADataProcessor:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
    def format_prompt(self, question, options):
        SYSTEM_PROMPT = """You are a medical reasoning expert. Structure every response as:

<facts>
1. [atomic medical fact]
2. [atomic medical fact]
...
</facts>
<answer>X</answer>

Rules for facts:
- List 4-8 atomic facts that directly justify your answer.
- Each fact must be a single, verifiable statement of general medical knowledge.
- State facts about conditions, drugs, or mechanisms — not about "the patient" or "this case".
- Facts must be distinct from each other. Do not rephrase the same idea.

Rules for the answer:
- Output exactly one letter (A, B, C, D, ...) inside <answer> tags.
- The answer must be entailed by the facts you listed."""
        options_text = "\n".join(f"{k}: {v}" for k, v in options.items())
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": (
                f"Question: {question}\n\n{options_text}\n\n"
                "Provide your atomic fact-based reasoning in <facts> tags, "
                "then your answer in <answer> tags."
            )},
        ]
        return self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        
    def load_medqa_data(self, file_path):
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
    
        processed_data = []
        for item in data:
            prompt = self.format_prompt(item['question'], item['options'])
            processed_data.append({
                'question': item['question'],
                'options': item['options'],
                'correct_answer': item['answer_idx'],
                'prompt': prompt,
            })
        return processed_data

# Baseline Network

In [ ]:
class BaselineNetwork(nn.Module):
    def __init__(self, input_dim, hidden_dim=256):
        super().__init__()
        self.input_norm = nn.LayerNorm(input_dim)
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim // 2, 1)
        )

        self._initialize_weights()

    def _initialize_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight, gain=0.1)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)

    def forward(self, x):
        x = x.float()
        x = torch.clamp(x, min=-10.0, max=10.0)
        x = self.input_norm(x)
        output = self.network(x)
        output = torch.clamp(output.squeeze(-1), min=-5.0, max=5.0)
        return output

# Policy Trainer

In [ ]:
class StopOnSequence(StoppingCriteria):
    def __init__(self, stop_ids):
        self.stop_ids = stop_ids

    def __call__(self, input_ids, scores, **kwargs):
        seq_len = len(self.stop_ids)
        if input_ids.shape[1] < seq_len:
            return False
        stop = torch.tensor(self.stop_ids, device=input_ids.device)
        return bool(torch.all(input_ids[0, -seq_len:] == stop))

def create_regularized_reward_head(hidden_dim, device, dropout=0.3, weight_decay=0.01):
        reward_head = nn.Sequential(
            nn.LayerNorm(hidden_dim),          # Normalize hidden states before scoring
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),               # 0.3 instead of 0.1
            nn.Linear(hidden_dim // 2, hidden_dim // 4),
            nn.ReLU(),
            nn.Dropout(dropout),               # Second dropout layer
            nn.Linear(hidden_dim // 4, 1),
        ).to(device, dtype=torch.float32)
     
        # Xavier init with small gain to start near zero
        for module in reward_head.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight, gain=0.1)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
     
        return reward_head

In [ ]:
class PolicyTrainer:
    def __init__(self, 
        model_path, 
        reward_config=None, 
        use_baseline=True,
        use_learnable_reward=True, 
        log_file=None, 
        group_size=8, 
        beta=0.01,
        epsilon=0.2,
        use_qlora=True,
        lora_r=16,
        lora_alpha=32,
        clip_advantage=3.0,
        skip_lora=False):
        
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {self.device}")
        print(f"Loading tokenizer and model from local path: {model_path}")
        self.use_learnable_reward = use_learnable_reward
        self.tokenizer = AutoTokenizer.from_pretrained(model_path,trust_remote_code=True,padding_side='left')
        self.data_processor = MedQADataProcessor(self.tokenizer)

        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
            self.tokenizer.pad_token_id = self.tokenizer.eos_token_id
            self.tokenizer.padding_side = 'left' #added
        if self.tokenizer.eos_token is None:
            self.tokenizer.eos_token = self.tokenizer.pad_token or self.tokenizer.unk_token
        if self.tokenizer.eos_token_id is None:
            self.tokenizer.eos_token_id = self.tokenizer.convert_tokens_to_ids(self.tokenizer.eos_token)

        if use_qlora:
            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.bfloat16
            )
            
            self.model = AutoModelForCausalLM.from_pretrained(
                model_path,
                quantization_config=bnb_config,
                device_map="auto",
                trust_remote_code=True
            )
            
            self.model = prepare_model_for_kbit_training(self.model)
            if not skip_lora:
            # Add LoRA
                lora_config = LoraConfig(
                    r=lora_r,
                    lora_alpha=lora_alpha,
                    inference_mode=False, 
                    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
                    lora_dropout=0.05,
                    bias="none",
                    task_type=TaskType.CAUSAL_LM
                )
            
                self.model = get_peft_model(self.model, lora_config)
                self.model.print_trainable_parameters()
        else:
            # Original implementation
            self.model = AutoModelForCausalLM.from_pretrained(
                model_path,
                dtype=torch.float16,
                device_map="auto",
                low_cpu_mem_usage=True,
                ignore_mismatched_sizes=True,
                trust_remote_code=True
            )

        self.device = next(self.model.parameters()).device
        print(f"Using device: {self.device}")
        
        self.metrics_tracker = MetricsTracker()

        self._baseline_cache_path = "logs/baseline_responses.json"
        
        self.use_learnable_reward = True
        hidden_dim = self.model.config.hidden_size
        print(f"Model hidden dimension: {hidden_dim}")

        if self.use_learnable_reward:
            self.reward_head = create_regularized_reward_head(hidden_dim, self.device)
            self.reward_optimizer = torch.optim.AdamW(
                self.reward_head.parameters(), lr=1e-5, weight_decay=0.01
            )
        print(f"Learnable reward head initialized on {self.device}")


        self.model.train()

        self.reward_function = RewardModel(**reward_config)
        if self.reward_function.fact_verification_system is not None:
            self.reward_function.fact_verification_system.training_mode = True
            pass
        else:
            print("No verification system - using heuristic rewards only")

        self.use_baseline = use_baseline
        if self.use_baseline:
            self.baseline_network = BaselineNetwork(input_dim=hidden_dim)
            self.baseline_network = self.baseline_network.to(self.device)
            self.baseline_optimizer = optim.Adam(self.baseline_network.parameters(), lr=1e-5)
            print("Baseline network initialized")

        self.policy_optimizer = torch.optim.AdamW(
            self.model.parameters(), 
            lr=1e-5,
            betas=(0.9, 0.95),
            weight_decay=0.01
        )
        self.reward_history = []

        # GRPO parameters
        self.group_size = group_size
        self.beta = beta
        self.epsilon = epsilon
        self.clip_advantage = clip_advantage

        adversarial_config = {
            'temperature': 1.2,
            'max_examples': 50,
            'preference_margin': 0.5,
            'validation_threshold': 0.7
        }
        self.adversarial_trainer = AdversarialTrainer(self, adversarial_config)

        #  ADD: Create frozen reference model
        print("Creating frozen reference model for KL penalty...")
        self.reference_model = None
        print(f"Model device: {next(self.model.parameters()).device}")

        # Set up logging
        self.log_file = log_file or f"training_log_{time.strftime('%Y%m%d_%H%M%S')}.txt"

        # Clear the log file at start
        with open(self.log_file, 'w', encoding='utf-8') as f:
            f.write(f"Training Log Started: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write("=" * 80 + "\n\n")

        print("PolicyTrainer initialization complete!")

    def _generate_text(self, prompt, max_new_tokens=768, temperature=0.7, top_p=0.9, num_return_sequences=1):
        inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=768)
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        prompt_length = inputs['input_ids'].shape[1]
    
        stop_ids = self.tokenizer.encode("</answer>", add_special_tokens=False)
    
        self.model.eval()
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=temperature,
                top_p=top_p,
                num_return_sequences=num_return_sequences,
                pad_token_id=self.tokenizer.pad_token_id,
                eos_token_id=self.tokenizer.eos_token_id,          # normal EOS as fallback
                stopping_criteria=StoppingCriteriaList([StopOnSequence(stop_ids)])
            )
    
        responses = [
            self.tokenizer.decode(outputs[i][prompt_length:], skip_special_tokens=True).strip()
            for i in range(num_return_sequences)
        ]
        return responses if num_return_sequences > 1 else responses[0]

    def _judge_to_cpu(self):
        if self.reward_function.fact_verification_system is not None:
            judge = self.reward_function.fact_verification_system.llm_verifier
            if judge is not None and hasattr(judge, 'pipe') and judge.pipe is not None:
                pipe_model = judge.pipe.model
                if getattr(pipe_model, 'is_loaded_in_4bit', False):
                    return
                judge.pipe.model = pipe_model.to('cpu')
                torch.cuda.empty_cache()
    
    def _judge_to_gpu(self):
        if self.reward_function.fact_verification_system is not None:
            judge = self.reward_function.fact_verification_system.llm_verifier
            if judge is not None and hasattr(judge, 'pipe') and judge.pipe is not None:
                pipe_model = judge.pipe.model
                if getattr(pipe_model, 'is_loaded_in_4bit', False):
                    return
                judge.pipe.model = pipe_model.to(self.device)
                
    def _ensure_baseline_responses(self, eval_prompts):
        if os.path.exists(self._baseline_cache_path):
            with open(self._baseline_cache_path, "r", encoding="utf-8") as f:
                return json.load(f)

        print("Generating baseline responses using existing model...")
        self._baseline_model.eval()
        baseline_results = []
        for prompt in eval_prompts:
            inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=768)
            inputs = {k: v.to(self.device) for k, v in inputs.items()}
            with torch.no_grad():
                outputs = self._baseline_model.generate(
                    **inputs,
                    max_new_tokens=100,
                    do_sample=True,
                    temperature=0.7,
                    top_p=0.9,
                    pad_token_id=self.tokenizer.pad_token_id,
                    eos_token_id=self.tokenizer.eos_token_id
                )

            base_text = self.tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:],
                                              skip_special_tokens=True).strip()
            baseline_results.append(base_text)

        os.makedirs("logs", exist_ok=True)
        with open(self._baseline_cache_path, "w", encoding="utf-8") as f:
            json.dump(baseline_results, f, indent=2)
        return baseline_results

    def log_epoch_comparisons(self, eval_prompts, epoch, correct_answers=None, context_list=None):
        results = []
        if epoch % 5 != 0:
            return []
        for i, prompt in enumerate(eval_prompts):
            context = context_list[i] if context_list else ""
            correct_answer = correct_answers[i] if correct_answers else None

            self.model.eval()
            baseline_texts = self._ensure_baseline_responses(eval_prompts)
            base_text = baseline_texts[i]

            adv_text, _, _, _ = self.generate_response_with_logprobs(prompt)

            # Compute rewards for adversarial response
            reward_info = self.reward_function.compute_total_reward(
                adv_text, correct_answer, context
            )

            entry = {
                "epoch": epoch,
                "prompt": prompt,
                "baseline_response": base_text,
                "adversarial_response": adv_text,
                "correct_answer": correct_answer,
                "reward_info": reward_info,
            }
            results.append(entry)

            # Console logging
            print("=" * 80)
            print(f"[Epoch {epoch}] Prompt: {prompt}")
            print(f"Correct Answer: {correct_answer}")
            print("\n--- Baseline Response ---")
            print(base_text)
            print("\n--- Adversarial Response ---")
            print(adv_text)
            print("\nReward breakdown:")
            print(reward_info)

        # Save to JSON file for later analysis
        os.makedirs("logs", exist_ok=True)
        log_path = f"logs/epoch_{epoch}_comparisons.json"
        with open(log_path, "w") as f:
            json.dump(results, f, indent=2)

        return results

    def generate_response_with_logprobs(self, prompt, max_new_tokens=768):
        response_text = self._generate_text(prompt, max_new_tokens=max_new_tokens)
        if len(generated_ids) == 0:
            return response_text, None, None

        # ── Single forward pass for both log probs AND hidden states ──
        full_ids = outputs[0].unsqueeze(0)
        full_mask = torch.ones_like(full_ids)

        full_outputs = self.model(
            input_ids=full_ids,
            attention_mask=full_mask,
            output_hidden_states=True,
        )
        logits = full_outputs.logits[0]  # [seq_len, vocab]

        # Log probs for response tokens
        # logits[t] predicts token[t+1], so logits[input_length-1] predicts first generated token
        response_logits = logits[input_length - 1 : input_length - 1 + len(generated_ids)]
        response_logits = response_logits.clamp(-50, 50)
        log_probs = torch.log_softmax(response_logits, dim=-1)
        token_log_probs = log_probs.gather(
            1, generated_ids.unsqueeze(-1)
        ).squeeze(-1)  # [response_len]

        # Hidden state from the same pass (response portion only)
        hidden = full_outputs.hidden_states[-1][0]  # [seq_len, hidden_dim]
        response_hidden = hidden[input_length:]
        avg_hidden_state = response_hidden.clamp(-10, 10).mean(dim=0)

        return response_text, token_log_probs, avg_hidden_state

    def generate_and_evaluate_with_facts(self, prompt, correct_answer, context=""):
        response_text, log_probs, hidden_state = self.generate_response_with_logprobs(prompt)

        rule_reward_info = self.reward_function.compute_total_reward(
            response_text, correct_answer, context=prompt  #  Pass prompt as context
        )

        # Add learnable reward component
        if hasattr(self, 'use_learnable_reward') and self.use_learnable_reward:
            with torch.no_grad():
                learnable_reward = self.reward_head(hidden_state.detach().float()).item()

            # Combine rule-based and learnable rewards
            combined_reward = 0.7 * rule_reward_info['r_normalized'] + 0.3 * learnable_reward

            # Add to reward info
            reward_info = rule_reward_info.copy()
            reward_info['r_learnable'] = learnable_reward
            reward_info['r_combined'] = combined_reward
            reward_info['r_normalized'] = combined_reward
        else:
            reward_info = rule_reward_info

        return response_text, log_probs, hidden_state, reward_info

    def compute_baseline_value(self, hidden_state, training_mode=False):
        """Compute baseline value with enhanced error handling."""
        if not self.use_baseline or hidden_state is None:
            return torch.tensor(0.0, device=self.device, requires_grad=training_mode, dtype=torch.float16)

        # Enhanced validity checks
        if torch.any(torch.isnan(hidden_state)) or torch.any(torch.isinf(hidden_state)):
            print("Warning: Invalid hidden state input to baseline, using fallback")
            return torch.tensor(0.0, device=self.device, requires_grad=training_mode, dtype=torch.float16)

        # Clamp input to prevent extreme values
        hidden_state_input = torch.clamp(hidden_state, min=-10.0, max=10.0)

        try:
            baseline_value = self.baseline_network(hidden_state_input)

            # Check for invalid baseline output
            if torch.isnan(baseline_value) or torch.isinf(baseline_value):
                print("Warning: Baseline network produced invalid output, using fallback")
                baseline_value = torch.tensor(0.0, device=self.device, requires_grad=training_mode, dtype=torch.float16)
            else:
                baseline_value = torch.clamp(baseline_value, min=-5.0, max=5.0)

        except Exception as e:
            print(f"Warning: Error in baseline network: {e}, using fallback")
            baseline_value = torch.tensor(0.0, device=self.device, requires_grad=training_mode, dtype=torch.float16)

        if training_mode:
            return baseline_value
        else:
            return baseline_value.item()

    def _generate_conservative_response(self, prompt, correct_answer):
        inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=768)
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=100,
                do_sample=True,
                temperature=0.7,
                top_p=0.8,
                pad_token_id=self.tokenizer.pad_token_id,
                eos_token_id=self.tokenizer.eos_token_id
            )

        response = self.tokenizer.decode(
            outputs[0][inputs['input_ids'].shape[1]:],
            skip_special_tokens=True
        ).strip()

        return response

    def _generate_high_quality_response(self, prompt, correct_answer):
        """Generate a high-quality response that should score well"""

        # Use conservative generation parameters for high quality
        inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=768)
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=200,
                do_sample=True,
                temperature=0.6,  # Lower temperature for more focused responses
                top_p=0.9,
                pad_token_id=self.tokenizer.pad_token_id,
                eos_token_id=self.tokenizer.eos_token_id
            )

        response = self.tokenizer.decode(
            outputs[0][inputs['input_ids'].shape[1]:],
            skip_special_tokens=True
        ).strip()

        # Ensure proper format
        if '<facts>' not in response:
            response = f"<facts>Let me analyze this medical scenario systematically. {response}</facts>"
        if '<answer>' not in response:
            response += f"<answer>{correct_answer}</answer>"

        return response

    def _analyze_response_quality(self, response, reward_info, correct_answer):
        violations = {
            'correctness_violations': 0,
            'answer_leaking_violations': 0,
            'format_violations': 0,
            'factual_violations': 0,
            'bad_ood_high_rewards': 0
        }
        if reward_info.get('r_accuracy', 0) <= 0:
            violations['correctness_violations'] = 1

        if reward_info.get('p_answerLeak', 0) > 0.1:
            violations['answer_leaking_violations'] = 1

        if reward_info.get('p_preamble', 0) > 0.1 or not self.reward_function.validate_format(response):
            violations['format_violations'] = 1

        if reward_info.get('r_grounded', 1.0) < 0.3:
            violations['factual_violations'] = 1

        # Goal 5: Less Rewards to Bad OOD Reasoning
        has_violations = sum(violations.values()) > 0
        high_reward = reward_info.get('r_total', 0) > 0.5
        if has_violations and high_reward:
            violations['bad_ood_high_rewards'] = 1

        return violations

    def evaluate_model(self, test_data_path, max_examples):
        test_dataset = self.data_processor.load_medqa_data(test_data_path)
        if self.reward_function.fact_verification_system is not None:
            self.reward_function.fact_verification_system.training_mode = False

        if max_examples is not None:
            test_dataset = test_dataset[:max_examples]

        print(f"Evaluating on {max_examples} test examples...")

        self.model.eval()
        if self.use_baseline:
            self.baseline_network.eval()

        correct_predictions = 0
        total_examples = len(test_dataset)

        # Track individual violations for standard deviation calculation
        violation_records = {
            'correctness_violations': [],
            'answer_leaking_violations': [],
            'format_violations': [],
            'factual_violations': [],
            'bad_ood_high_rewards': []
        }

        # Enhanced reward tracking including factual scores
        reward_components = {'r_accuracy': [], 'p_answerLeak': [], 'p_preamble': [], 'r_grounded': []}
        all_rewards = []
        format_violations = 0
        evaluation_data = []

        with torch.no_grad():
            for i, test_item in enumerate(test_dataset):
                prompt = test_item['prompt']
                correct_answer = test_item['correct_answer']

                response = self.generate_response(prompt)

                # Enhanced reward computation with factual verification
                reward_info = self.reward_function.compute_total_reward(response, correct_answer)

                evaluation_data.append({
                    'response': response,
                    'reward_info': reward_info,
                    'correct_answer': correct_answer
                })

                # Track reward components
                for key in reward_components:
                    reward_components[key].append(reward_info[key])

                all_rewards.append(reward_info['r_normalized'])

                predicted_answer = self.reward_function.extract_answer_choice(response)
                is_correct = predicted_answer and predicted_answer.upper() == correct_answer.upper()
                if is_correct:
                    correct_predictions += 1

                format_ok = self.reward_function.validate_format(response)
                format_violations += int(not format_ok)

                # Track individual violations for std calculation
                violations = self._analyze_response_quality(response, reward_info, correct_answer)
                for key in violation_records.keys():
                    violation_records[key].append(violations[key])

                if i % 100 == 0:
                    print(f"Evaluated {i + 1}/{total_examples} examples...")

        format_violation_rate = format_violations / total_examples
        accuracy = correct_predictions / total_examples
        avg_rewards = {key: np.mean(values) for key, values in reward_components.items()}
        hacking_stats = self.reward_function.calculate_hacking_rate(evaluation_data)

        # Calculate standard deviations
        reward_std = np.std(all_rewards) if len(all_rewards) > 1 else 0.0

        test_metrics = {
            "accuracy": accuracy,
            "correctness_violations_rate": 1 - accuracy,
            "answer_leaking_violations_rate": hacking_stats["answer_violation_rate"],
            "format_violations_rate": format_violation_rate,
            "factual_violations_rate": hacking_stats["factual_violation_rate"],
            "bad_ood_high_rewards_rate": hacking_stats["overall_violation_rate"],
            "avg_reward": np.mean(all_rewards),
            "avg_factual": avg_rewards["r_grounded"],
            "avg_answer_penalty": avg_rewards.get("p_answerLeak", 0.0),
            "avg_structural_penalty": avg_rewards.get("p_preamble", 0.0),
            "avg_factual_reward": avg_rewards.get("r_grounded", 0.0),
            "avg_total_reward": np.mean(all_rewards),
            "correct_predictions": correct_predictions,
            "total_examples": total_examples,
            "reward_history": self.reward_history,
            "std_reward": reward_std,
            "std_factual": np.std(reward_components['r_grounded']) if len(reward_components['r_grounded']) > 1 else 0.0
        }

        # Add violation standard deviations
        for key in violation_records.keys():
            rate_key = f"{key}_rate"
            std_key = f"{key}_std"

            # Calculate violation rate and std
            test_metrics[rate_key] = test_metrics.get(rate_key, np.mean(violation_records[key]))
            test_metrics[std_key] = np.std(violation_records[key]) if len(violation_records[key]) > 1 else 0.0

        # Use MetricsTracker instead of manual printing
        self.metrics_tracker.print_metrics_with_std(test_metrics, "Test Evaluation Results")

        return test_metrics

    def generate_response(self, prompt, max_new_tokens=768) -> str:
        self.model.eval()
        if isinstance(prompt, list):
            prompt = prompt[0]  # Take first element if it's a list
        
        
        inputs = self.tokenizer(
            prompt, 
            return_tensors="pt", 
            truncation=True, 
            max_length=768,
            padding=True
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                pad_token_id=self.tokenizer.pad_token_id,
                eos_token_id=self.tokenizer.eos_token_id
            )
        
        response = self.tokenizer.decode(
            outputs[0][inputs['input_ids'].shape[1]:],
            skip_special_tokens=True
        ).strip()
        
        return response

    def analyze_hacking_sensitivity(self, test_data_path, max_examples, tau_answer_range=None, tau_preamble_range=None):
        if tau_answer_range is None:
            tau_answer_range = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5]
        if tau_preamble_range is None:
            tau_preamble_range = [5, 10, 15, 20, 25, 30, 35, 40, 45, 50]

        test_dataset = self.data_processor.load_medqa_data(test_data_path)
        if max_examples is not None:
            test_dataset = test_dataset[:max_examples]

        print(f"Running sensitivity analysis on {len(test_dataset)} examples...")
        print(f"Testing tau_answer: {tau_answer_range}")
        print(f"Testing tau_preamble: {tau_preamble_range}")

        print("Generating responses...")
        self.model.eval()
        if self.use_baseline:
            self.baseline_network.eval()

        evaluation_data = []
        with torch.no_grad():
            for i, test_item in enumerate(test_dataset):
                prompt = test_item['prompt']
                correct_answer = test_item['correct_answer']
                response = self.generate_response(prompt)

                evaluation_data.append({
                    'response': response,
                    'correct_answer': correct_answer,
                    'prompt': prompt
                })

        # Store original thresholds
        original_tau_answer = self.reward_function.tau_answer
        original_tau_preamble = self.reward_function.tau_preamble

        sensitivity_results = []

        print("\nTesting threshold combinations...")
        for tau_answer in tau_answer_range:
            for tau_structural in tau_preamble_range:
                print(f"Testing tau_answer={tau_answer}, tau_preamble={tau_structural}")

                self.reward_function.tau_answer = tau_answer
                self.reward_function.tau_preamble = tau_structural

                responses_with_rewards = []
                for item in evaluation_data:
                    # Enhanced reward computation with factual verification
                    reward_info = self.reward_function.compute_total_reward(
                        item['response'], item['correct_answer']
                    )
                    responses_with_rewards.append({
                        'response': item['response'],
                        'reward_info': reward_info,
                        'correct_answer': item['correct_answer']
                    })

                hacking_stats = self.reward_function.calculate_hacking_rate(responses_with_rewards)

                positive_rewards = sum(1 for item in responses_with_rewards
                                       if item['reward_info']['r_total'] > 0)
                avg_answer_penalty = np.mean([item['reward_info']['p_answerLeak']
                                              for item in responses_with_rewards])
                avg_structural_penalty = np.mean([item['reward_info']['p_preamble']
                                                  for item in responses_with_rewards])
                avg_factual_reward = np.mean([item['reward_info']['r_grounded']
                                              for item in responses_with_rewards])

                sensitivity_results.append({
                    'tau_answer': tau_answer,
                    'tau_preamble': tau_structural,
                    'answer_violation_rate': hacking_stats['answer_violation_rate'],
                    'structural_violation_rate': hacking_stats['structural_violation_rate'],
                    'factual_violation_rate': hacking_stats['factual_violation_rate'],
                    'overall_violation_rate': hacking_stats['overall_violation_rate'],
                    'answer_violation_count': hacking_stats['answer_violation_count'],
                    'structural_violation_count': hacking_stats['structural_violation_count'],
                    'factual_violation_count': hacking_stats['factual_violation_count'],
                    'positive_reward_count': positive_rewards,
                    'positive_reward_rate': positive_rewards / len(responses_with_rewards),
                    'avg_answer_penalty': avg_answer_penalty,
                    'avg_structural_penalty': avg_structural_penalty,
                    'avg_factual_reward': avg_factual_reward
                })

        # Restore original thresholds
        self.reward_function.tau_answer = original_tau_answer
        self.reward_function.tau_preamble = original_tau_preamble

        return {
            'sensitivity_results': sensitivity_results,
            'tau_answer_range': tau_answer_range,
            'tau_preamble_range': tau_preamble_range,
            'total_examples': len(test_dataset)
        }

    def diagnose_factual_failures(self, test_data_path, n_examples=5):
        test_dataset = self.data_processor.load_medqa_data(test_data_path)[:n_examples]
        
        for i, item in enumerate(test_dataset):
            response = self.generate_response(item['prompt'])
            
            print(f"\n{'='*60}")
            print(f"Example {i+1}")
            print(f"{'='*60}")
            
            # Check extraction
            think_content = self.reward_function.extract_think_content(response)
            print(f"Extracted reasoning ({len(think_content)} chars):")
            print(think_content[:300] if think_content else "EMPTY!")
            
            # Check factual verification
            if self.reward_function.fact_verification_system:
                result = self.reward_function.fact_verification_system.verify_reasoning(
                    think_content, context=""
                )
                print(f"\nFactual result: {result}")
            else:
                print("\nNo fact verification system!")
            
            # Full reward breakdown
            reward_info = self.reward_function.compute_total_reward(
                response, item['correct_answer']
            )
            print(f"\nReward breakdown:")
            for k, v in reward_info.items():
                if isinstance(v, (int, float)):
                    print(f"  {k}: {v:.3f}")

    def train_reward_model_stage1(self, train_data_path, num_epochs, batch_size,
                                      val_fraction=0.2, patience=3, min_delta=0.005):    
        if not hasattr(self, 'reward_head'):
            print("No learnable reward head found - skipping reward model training")
            return
     
        # ── Load and split data ──
        full_dataset = self.data_processor.load_medqa_data(train_data_path)
     
        # Deterministic shuffle for reproducibility
        rng = np.random.RandomState(42)
        indices = rng.permutation(len(full_dataset))
        val_size = max(1, int(len(full_dataset) * val_fraction))
        val_indices = indices[:val_size]
        train_indices = indices[val_size:]
     
        train_dataset = [full_dataset[i] for i in train_indices]
        val_dataset = [full_dataset[i] for i in val_indices]
     
        print(f"Stage 1: Training reward model")
        print(f"  Train: {len(train_dataset)} examples, Val: {len(val_dataset)} examples")
        print(f"  Max epochs: {num_epochs}, Patience: {patience}")
     
        # ── Freeze policy, train reward head ──
        self.model.eval()
        self.reward_head.train()
     
        # ── Early stopping state ──
        best_val_loss = float('inf')
        best_epoch = -1
        epochs_no_improve = 0
        best_reward_head_state = copy.deepcopy(self.reward_head.state_dict())
     
        for epoch in range(num_epochs):
            epoch_start_time = time.time()
     
            # ── Train phase ──
            self.reward_head.train()
            train_loss, train_n = self._reward_head_epoch(
                train_dataset, batch_size, update_weights=True
            )
     
            # ── Validation phase ──
            self.reward_head.eval()
            with torch.no_grad():
                val_loss, val_n = self._reward_head_epoch(
                    val_dataset, batch_size, update_weights=False
                )
     
            epoch_time = time.time() - epoch_start_time
            avg_train = train_loss / max(train_n, 1)
            avg_val = val_loss / max(val_n, 1)
            gap = avg_val - avg_train
     
            print(
                f"Reward model epoch {epoch+1}/{num_epochs}, "
                f"Train Loss: {avg_train:.4f}, Val Loss: {avg_val:.4f}, "
                f"Gap: {gap:.4f}, Time: {epoch_time:.1f}s"
            )
     
            # ── Early stopping check ──
            if avg_val < best_val_loss - min_delta:
                best_val_loss = avg_val
                best_epoch = epoch + 1
                epochs_no_improve = 0
                best_reward_head_state = copy.deepcopy(self.reward_head.state_dict())
                print(f"   New best val loss: {best_val_loss:.4f}")
            else:
                epochs_no_improve += 1
                print(f"  No improvement for {epochs_no_improve}/{patience} epochs")
     
            # ── Overfitting warning ──
            if gap > 0.1:
                print(f"   Large train-val gap ({gap:.4f}) — reward head is overfitting")
     
            if epochs_no_improve >= patience:
                print(f"\n Early stopping triggered at epoch {epoch+1} "
                      f"(best was epoch {best_epoch})")
                break
     
        # ── Restore best checkpoint ──
        self.reward_head.load_state_dict(best_reward_head_state)
        print(f"Stage 1 complete: Restored best reward head from epoch {best_epoch} "
              f"(val loss: {best_val_loss:.4f})")
     
     
    def _reward_head_epoch(self, dataset, batch_size, update_weights=True):
        """
        Run one epoch of reward head training (or validation).
        Returns (total_loss, num_samples).
        """
        total_loss = 0.0
        num_samples = 0
     
        if update_weights:
            self.reward_optimizer.zero_grad()
     
        for batch_start in range(0, len(dataset), batch_size):
            batch_end = min(batch_start + batch_size, len(dataset))
            batch_items = dataset[batch_start:batch_end]
     
            for item in batch_items:
                try:
                    prompt = item['prompt']
                    correct_answer = item['correct_answer']
     
                    # Generate response (no gradients for policy)
                    with torch.no_grad():
                        response_text, _, hidden_state = self.generate_response_with_logprobs(prompt)
     
                    if hidden_state is None:
                        continue
     
                    # Compute target score using rule-based reward
                    rule_reward_info = self.reward_function.compute_total_reward(
                        response_text, correct_answer
                    )
                    target_score = rule_reward_info['r_normalized']
     
                    # Predict score using learnable reward head
                    predicted_score = self.reward_head(hidden_state.detach().float()).squeeze()
                    target_tensor = torch.tensor(
                        target_score, device=self.device, dtype=torch.float32
                    )
     
                    loss = nn.functional.mse_loss(predicted_score, target_tensor)
     
                    if update_weights:
                        loss.backward()
     
                    total_loss += loss.item()
                    num_samples += 1
     
                except Exception as e:
                    continue
     
            if update_weights and num_samples > 0:
                torch.nn.utils.clip_grad_norm_(self.reward_head.parameters(), max_norm=1.0)
                self.reward_optimizer.step()
                self.reward_optimizer.zero_grad()
     
        return total_loss, num_samples

    # ==================== GRPO METHODS ====================
    
    def compute_group_advantages(self, rewards):
        mean = rewards.mean(dim=1, keepdim=True)
        std = rewards.std(dim=1, keepdim=True)        
        advantages = (rewards - mean) / (std + 1e-8)
        return advantages

    def compute_reference_logprobs(self, prompt, response_text):
        full_text = prompt + response_text
        inputs = self.tokenizer(full_text, return_tensors="pt", truncation=True, max_length=768)
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        prompt_length = self.tokenizer(prompt, return_tensors="pt")['input_ids'].shape[1]
    
        with torch.no_grad():
            # Disable LoRA — base model weights are the reference
            with self.model.disable_adapter():
                outputs = self.model(**inputs)
    
        logits = outputs.logits[0]
        response_ids = inputs['input_ids'][0, prompt_length:]
        log_probs = F.log_softmax(logits[prompt_length-1:-1], dim=-1)
        return log_probs[torch.arange(len(response_ids)), response_ids]

    def compute_token_weights(self, response_text, correct_answer):
        """Weight tokens by their proximity to the answer region."""
        tokens = self.tokenizer.encode(response_text, add_special_tokens=False)
        weights = torch.ones(len(tokens), device=self.device)
        
        # Find answer region in token space
        answer_pattern = f"<answer>{correct_answer}</answer>"
        answer_start = response_text.lower().find("<answer>")
        answer_end = response_text.lower().find("</answer>") + len("</answer>")
        
        if answer_start == -1:
            return weights  # Uniform if no answer tags
        
        # Convert char positions to approximate token positions
        prefix = response_text[:answer_start]
        answer_region = response_text[answer_start:answer_end]
        
        prefix_tokens = len(self.tokenizer.encode(prefix, add_special_tokens=False))
        answer_tokens = len(self.tokenizer.encode(answer_region, add_special_tokens=False))
        
        # High weight for answer tokens
        for i in range(prefix_tokens, min(prefix_tokens + answer_tokens, len(tokens))):
            weights[i] = 5.0  # Answer tokens matter 5x more
        
        # Medium weight for tokens near answer (last 20% of reasoning)
        reasoning_end = prefix_tokens
        reasoning_boost_start = int(reasoning_end * 0.8)
        for i in range(reasoning_boost_start, reasoning_end):
            weights[i] = 2.0
        
        # Normalize so sum equals original length (preserves gradient scale)
        weights = weights * (len(tokens) / weights.sum())
        
        return weights

    def generate_group_responses_batched(self, prompt, n=None):
        if n is None:
            n = self.group_size
        return self._generate_text(prompt, num_return_sequences=n)
        
    def compute_response_logprobs(self, prompt, response_text):
        full_text = prompt + response_text
        inputs = self.tokenizer(full_text, return_tensors="pt", truncation=True, max_length=768)
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        prompt_length = self.tokenizer(prompt, return_tensors="pt")['input_ids'].shape[1]
    
        with torch.cuda.amp.autocast():  # also add mixed precision
            outputs = self.model(**inputs)
    
        logits = outputs.logits[0]  # [seq_len, vocab]
        response_ids = inputs['input_ids'][0, prompt_length:]  # [resp_len]
    
        # Vectorized: no Python loop
        log_probs = F.log_softmax(logits[prompt_length-1:-1], dim=-1)  # [resp_len, vocab]
        token_log_probs = log_probs[torch.arange(len(response_ids)), response_ids]
        return token_log_probs

    def grpo_train_step(self, batch_items, global_step=0):
        """GRPO with batched generation."""
        self.model.train()
        self.policy_optimizer.zero_grad()
        
        total_policy_loss = 0
        total_kl_penalty = 0
        all_rewards = []
        all_advantages = []
        num_updates = 0
        
        for item in batch_items:
            try:
                prompt = item['prompt']
                correct_answer = item['correct_answer']
                context = item.get('context', '')
                self._judge_to_cpu()
                response_texts = self.generate_group_responses_batched(prompt, self.group_size)
                for i, r in enumerate(response_texts):
                    if len(r.strip()) < 10:
                        print(f"WARNING: Empty/short response {i}: '{r[:50]}'")
                group_responses = []
                for response_text in response_texts:
                    log_probs = self.compute_response_logprobs(prompt, response_text)
                    
                    if log_probs is None or not log_probs.requires_grad:
                        continue
                    
                    ref_log_probs = self.compute_reference_logprobs(prompt, response_text)
                    token_weights = self.compute_token_weights(response_text, correct_answer)
                    
                    group_responses.append({
                        'text': response_text,
                        'log_probs': log_probs,
                        'ref_log_probs': ref_log_probs,
                        'token_weights': token_weights
                    })
                
                if len(group_responses) < 2:
                    continue
                
                # Compute rewards
                group_rewards = []
                self._judge_to_gpu()
                with torch.no_grad():
                    for resp in group_responses:
                        reward_info = self.reward_function.compute_total_reward(
                            resp['text'], correct_answer, context,
                            step=global_step
                        )
                        group_rewards.append(reward_info['r_normalized'])
                self._judge_to_cpu()
                torch.cuda.empty_cache()
                
                group_rewards = torch.tensor(group_rewards, device=self.device, dtype=torch.float32)
                num_skipped = 0
                if group_rewards.std() < 1e-4:
                    num_skipped += 1
                    continue
                advantages = self.compute_group_advantages(group_rewards.unsqueeze(0)).squeeze(0)
                advantages = torch.clamp(advantages, -self.clip_advantage, self.clip_advantage)
                for i, (resp, advantage) in enumerate(zip(group_responses, advantages)):
                    log_probs = resp['log_probs']
                    ref_log_probs = resp['ref_log_probs']
                    token_weights = resp['token_weights']
                    
                    min_len = min(len(log_probs), len(ref_log_probs), len(token_weights))
                    if min_len == 0:
                        continue
                    
                    log_probs = log_probs[:min_len]
                    ref_log_probs = ref_log_probs[:min_len]
                    token_weights = token_weights[:min_len].to(self.device)
                    
                    kl_div = (log_probs - ref_log_probs).mean()
                    if kl_div.item() > 10.0:
                        continue
                    kl_div = torch.clamp(kl_div, min=0.0, max=10.0)
                    
                    weighted_log_probs = (log_probs * token_weights).sum()
                    policy_loss = -(weighted_log_probs * advantage.detach()) + self.beta * kl_div
                    policy_loss = torch.clamp(policy_loss, min=-50, max=50)
                    
                    policy_loss.backward()
                    
                    total_policy_loss += policy_loss.item()
                    total_kl_penalty += kl_div.item()
                    all_rewards.append(group_rewards[i].item())
                    all_advantages.append(advantage.item())
                    num_updates += 1
            
            except Exception as e:
                print(f"  Error: {e}")
                continue
        
        if num_updates > 0:
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=0.5)
            self.policy_optimizer.step()
        
        return {
            'policy_loss': total_policy_loss / num_updates if num_updates > 0 else 0.0,
            'kl_penalty': total_kl_penalty / num_updates if num_updates > 0 else 0.0,
            'avg_reward': np.mean(all_rewards) if all_rewards else 0.0,
            'avg_advantage': np.mean(all_advantages) if all_advantages else 0.0,
            'std_reward': np.std(all_rewards) if all_rewards else 0.0,
            'std_advantage': np.std(all_advantages) if all_advantages else 0.0
        }
    def validate_on_holdout(self, val_data_path, max_examples=50):
        """Validate model on holdout set"""
        val_dataset = self.data_processor.load_medqa_data(val_data_path)
        val_dataset = val_dataset[:max_examples]
        
        metrics = self.evaluate_model(
            test_data_path=val_data_path,
            max_examples=max_examples
        )
        
        return metrics['accuracy'], metrics['avg_total_reward']
    

    def train_policy_stage2(self,
                        train_data_path,
                        num_epochs,
                        batch_size=2,
                        val_data_path=None,
                        patience=3,
                        kl_threshold_batch=0.5,
                        kl_threshold_epoch=0.3,
                        kl_warmup_batches=20,
                        adaptive_kl=True,
                        kl_target=0.05,
                        ):
        train_dataset = self.data_processor.load_medqa_data(train_data_path)
        print(f"Stage 2 (GRPO): Training policy on {len(train_dataset)} examples")
        print(f"GRPO parameters: group_size={self.group_size}, beta={self.beta}")
        print(f"KL safety: batch_threshold={kl_threshold_batch}, "
              f"epoch_threshold={kl_threshold_epoch}, adaptive={adaptive_kl}")
    
        best_val_accuracy = 0.0
        best_val_reward = -float('inf')
        epochs_without_improvement = 0
        global_step = 0
        original_beta = self.beta
        kl_history = []
    
        for epoch in range(num_epochs):
            epoch_start_time = time.time()
            print(f"\n{'='*60}")
            print(f"Epoch {epoch + 1}/{num_epochs} (beta={self.beta:.4f})")
            print(f"{'='*60}")
    
            self.model.train()
    
            epoch_metrics = {'policy_loss': [], 'kl_penalty': [], 'rewards': [], 'advantages': []}
            num_batches = 0
            failed_batches = 0
            kl_skipped_batches = 0
            running_kl_window = []
            running_kl_window_size = 20
    
            for batch_start in range(0, len(train_dataset), batch_size):
                batch_end = min(batch_start + batch_size, len(train_dataset))
                batch_items = train_dataset[batch_start:batch_end]
    
                try:
                    # ── KL check BEFORE applying gradients ──
                    # Compute metrics without stepping the optimizer
                    metrics = self.grpo_train_step(batch_items, global_step=global_step)
                    batch_kl = metrics['kl_penalty']
    
                    if num_batches >= kl_warmup_batches and batch_kl > kl_threshold_batch:
                        kl_skipped_batches += 1
                        self.policy_optimizer.zero_grad()  # discard accumulated grads
                        if kl_skipped_batches <= 5 or kl_skipped_batches % 10 == 0:
                            print(f"  Batch {num_batches+1}: KL={batch_kl:.4f} > "
                                  f"{kl_threshold_batch} (skipped #{kl_skipped_batches})")
                    else:
                        torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=0.5)
                        self.policy_optimizer.step()
    
                    self.policy_optimizer.zero_grad()
                    global_step += 1
    
                    for key in ['policy_loss', 'kl_penalty']:
                        epoch_metrics[key].append(metrics[key])
                    epoch_metrics['rewards'].append(metrics['avg_reward'])
                    epoch_metrics['advantages'].append(metrics['avg_advantage'])
                    num_batches += 1
    
                    if num_batches % 10 == 0:
                        print(f"  Batch {num_batches}: "
                              f"Loss={metrics['policy_loss']:.4f}, "
                              f"Reward={metrics['avg_reward']:.4f}, "
                              f"KL={batch_kl:.4f}")
    
                    # ── Running KL check for mid-epoch stopping ──
                    running_kl_window.append(batch_kl)
                    if len(running_kl_window) > running_kl_window_size:
                        running_kl_window.pop(0)
    
                    if (num_batches >= kl_warmup_batches + running_kl_window_size and
                            len(running_kl_window) == running_kl_window_size):
                        running_avg_kl = np.mean(running_kl_window)
                        if running_avg_kl > kl_threshold_epoch:
                            print(f"\n  🛑 Mid-epoch KL stop: running avg "
                                  f"KL={running_avg_kl:.4f} > {kl_threshold_epoch}")
                            break
    
                except Exception as e:
                    print(f"  Error in batch {batch_start}-{batch_end}: {e}")
                    self.policy_optimizer.zero_grad()
                    failed_batches += 1
                    continue
    
            if num_batches == 0:
                print(f"\nEpoch {epoch + 1}: NO successful batches!")
                continue
    
            epoch_time = time.time() - epoch_start_time
            avg_kl = np.mean(epoch_metrics['kl_penalty'])
            avg_reward = np.mean(epoch_metrics['rewards'])
    
            print(f"\nEpoch {epoch + 1} Summary:")
            print(f"  Time: {epoch_time:.2f}s ({epoch_time/60:.2f} min)")
            print(f"  Batches: {num_batches} ok / {failed_batches} failed / {kl_skipped_batches} KL-skipped")
            print(f"  Avg Policy Loss: {np.mean(epoch_metrics['policy_loss']):.4f}")
            print(f"  Avg KL: {avg_kl:.4f}")
            print(f"  Avg Reward: {avg_reward:.4f} (±{np.std(epoch_metrics['rewards']):.4f})")
            print(f"  Avg Advantage: {np.mean(epoch_metrics['advantages']):.4f} "
                  f"(±{np.std(epoch_metrics['advantages']):.4f})")
    
            kl_history.append(avg_kl)
    
            if adaptive_kl:
                self._adjust_beta(avg_kl, kl_target, original_beta)
    
            if avg_kl > kl_threshold_epoch:
                print(f"\nEpoch avg KL ({avg_kl:.4f}) > threshold ({kl_threshold_epoch}). Stopping.")
                break
    
            if len(kl_history) >= 2:
                kl_increase = kl_history[-1] - kl_history[-2]
                if kl_increase > 0 and kl_history[-1] > kl_threshold_epoch * 0.7:
                    print(f"  KL trending up: {kl_history[-2]:.4f} → {kl_history[-1]:.4f} "
                          f"(+{kl_increase:.4f})")
    
            if val_data_path:
                print(f"\n{'─'*60}")
                val_accuracy, val_reward = self.validate_on_holdout(val_data_path, max_examples=50)
                print(f"  Val Accuracy: {val_accuracy:.2%}, Val Reward: {val_reward:.4f}")
    
                improved = (val_accuracy > best_val_accuracy or
                            (val_accuracy == best_val_accuracy and val_reward > best_val_reward))
    
                if improved:
                    best_val_accuracy = val_accuracy
                    best_val_reward = val_reward
                    epochs_without_improvement = 0
                    print(f"   New best model!")
                else:
                    epochs_without_improvement += 1
                    print(f"  No improvement ({epochs_without_improvement}/{patience})")
    
                if epochs_without_improvement >= patience:
                    print(f"\nEarly stopping at epoch {epoch+1}")
                    break
    
        if adaptive_kl:
            print(f"\nBeta trajectory: {original_beta:.4f} → {self.beta:.4f}")
        print(f"KL history: {[f'{k:.4f}' for k in kl_history]}")
        print("Stage 2 (GRPO) complete")
     
     
    def _adjust_beta(self, current_kl, kl_target, original_beta,
                     adjustment_factor=1.5, max_beta_multiplier=10.0):
        min_beta = original_beta / 2.0
        max_beta = original_beta * max_beta_multiplier
     
        if current_kl > kl_target * 1.5:
            new_beta = min(self.beta * adjustment_factor, max_beta)
            if new_beta != self.beta:
                print(f"  KL too high ({current_kl:.4f} > {kl_target*1.5:.4f}), "
                      f"beta: {self.beta:.4f} → {new_beta:.4f}")
                self.beta = new_beta
        elif current_kl < kl_target / 1.5:
            new_beta = max(self.beta / adjustment_factor, min_beta)
            if new_beta != self.beta:
                print(f"  KL low ({current_kl:.4f} < {kl_target/1.5:.4f}), "
                      f"beta: {self.beta:.4f} → {new_beta:.4f}")
                self.beta = new_beta
                
    def train_adversarial_stage3(self, train_data_path, num_cycles=3):
        """Stage 3: Adversarial training"""
        print("\n" + "=" * 60)
        print("STAGE 3: ADVERSARIAL TRAINING")
        print("=" * 60)
        
        train_dataset = self.data_processor.load_medqa_data(train_data_path)
        prompts = [item['prompt'] for item in train_dataset[:100]]
        answers = [item['correct_answer'] for item in train_dataset[:100]]
        
        results = self.adversarial_trainer.run_adversarial_training_cycle(
            prompts, answers, num_cycles
        )
        
        print("\nAdversarial training complete:")
        print(f"  Final robustness: {results['final_robustness']:.2f}")
        print(f"  Cycles completed: {results['total_cycles']}")
        
        return results

    def train_reward_policy(self, train_data_path, test_data_path, 
                            stage1_epochs, stage2_epochs, batch_size=2, 
                            max_eval_examples=50, save_checkpoint_path=None):
        """Train reward model and policy (Stages 1 & 2)"""
        print("TRAINING: Reward Model + Policy (GRPO)")
        print("=" * 40)
        
        # Stage 1: Reward head training
        self.train_reward_model_stage1(train_data_path, stage1_epochs, batch_size)
        epoch_start_time = time.time()
        # Stage 2: Policy training
        self.train_policy_stage2(train_data_path, stage2_epochs, batch_size)
        
        # Evaluate
        print("\nEvaluating Policy Training Results:")
        metrics = self.evaluate_model(test_data_path, max_eval_examples)
        self.metrics_tracker.print_metrics_with_std(metrics, "POLICY STAGE RESULTS")
        
        # Save checkpoint
        if save_checkpoint_path:
            self.save_model_checkpoint(save_checkpoint_path, "policy_complete")
            print(f"Checkpoint saved to: {save_checkpoint_path}")
        
        return metrics

    def train_adversarial(self, train_data_path, test_data_path, 
                        num_cycles=3, max_eval_examples=20):
        """Train adversarial stage only (Stage 3)"""
        print("TRAINING: Adversarial Stage Only")
        print("=" * 40)
        
        # Run adversarial training
        adversarial_results = self.train_adversarial_stage3(train_data_path, num_cycles)
        
        # Evaluate
        print("\nEvaluating Final Results:")
        final_metrics = self.evaluate_model(test_data_path, max_eval_examples)
        self.metrics_tracker.print_metrics_with_std(final_metrics, "FINAL ADVERSARIAL RESULTS")
        
        return {
            'metrics': final_metrics,
            'adversarial_results': adversarial_results
        }

    def train_combined(self, train_data_path, test_data_path, 
                    stage1_epochs, stage2_epochs, batch_size=2, 
                    max_eval_examples=20, adversarial_cycles=3,
                    save_dir=None):
        """Full training pipeline: Stages 1, 2, and 3"""
        print("\n" + "=" * 60)
        print("COMBINED TRAINING: ALL STAGES")
        print("=" * 60)
        
        # Stage 1 & 2: Reward + Policy
        print("\nSTAGE 1-2: REWARD MODEL + POLICY")
        print("-" * 50)
        
        reward_policy_metrics = self.train_reward_policy(
            train_data_path=train_data_path,
            test_data_path=test_data_path,
            stage1_epochs=stage1_epochs,
            stage2_epochs=stage2_epochs,
            batch_size=batch_size,
            max_eval_examples=max_eval_examples
        )
        
        print("\nReward + Policy Training Complete!")
        self.metrics_tracker.print_metrics_with_std(
            reward_policy_metrics, "REWARD+POLICY RESULTS"
        )
        
        if save_dir:
            stage12_dir = os.path.join(save_dir, "stage1-2_checkpoint")
            self.save_model_checkpoint(stage12_dir, "reward_policy_complete")
            print(f"Stage 1-2 checkpoint saved to: {stage12_dir}")
        
        # Stage 3: Adversarial
        print("\n" + "=" * 50)
        print("STAGE 3: ADVERSARIAL TRAINING")
        print("-" * 50)
        
        try:
            adversarial_results = self.train_adversarial(
                train_data_path=train_data_path,
                test_data_path=test_data_path,
                num_cycles=adversarial_cycles,
                max_eval_examples=max_eval_examples
            )
            
            print("\nAdversarial Training Complete!")
            self.metrics_tracker.print_metrics_with_std(
                adversarial_results['metrics'], "FINAL ADVERSARIAL RESULTS"
            )
            
            # Optional: Save final model
            if save_dir:
                final_dir = os.path.join(save_dir, "final_checkpoint")
                self.save_model_checkpoint(final_dir, "all_stages_complete")
                print(f"Final checkpoint saved to: {final_dir}")
            
            return {
                'reward_policy_metrics': reward_policy_metrics,
                'adversarial_metrics': adversarial_results['metrics'],
                'adversarial_results': adversarial_results['adversarial_results'],
                'training_successful': True
            }
        
        except Exception as e:
            print(f"\nERROR during adversarial training: {e}")
            print("Returning Stage 1-2 results only")
            
            return {
                'reward_policy_metrics': reward_policy_metrics,
                'adversarial_metrics': None,
                'adversarial_results': None,
                'training_successful': False,
                'error': str(e)
            }

    def save_model_checkpoint(self, checkpoint_path, stage_name="checkpoint"):
        """Save complete training checkpoint"""
        os.makedirs(checkpoint_path, exist_ok=True)
        
        # Save model
        self.model.save_pretrained(checkpoint_path, safe_serialization=True)
        
        # Save tokenizer
        self.tokenizer.save_pretrained(checkpoint_path)
        
        # Metadata
        checkpoint_data = {
            'stage': stage_name,
            'reward_history': self.reward_history,
            'use_learnable_reward': self.use_learnable_reward,
            'use_baseline': self.use_baseline,
            'group_size': self.group_size,
            'beta': self.beta,
            'epsilon': self.epsilon
        }
        
        # Save components
        if hasattr(self, 'reward_head') and self.reward_head is not None:
            torch.save(self.reward_head.state_dict(), 
                    os.path.join(checkpoint_path, "reward_head.pt"))
            checkpoint_data['has_reward_head'] = True
        else:
            checkpoint_data['has_reward_head'] = False
        
        if hasattr(self, 'baseline_network') and self.baseline_network is not None:
            torch.save(self.baseline_network.state_dict(),
                    os.path.join(checkpoint_path, "baseline_network.pt"))
            checkpoint_data['has_baseline_network'] = True
        else:
            checkpoint_data['has_baseline_network'] = False
        
        # Save optimizers
        torch.save(self.policy_optimizer.state_dict(),
                os.path.join(checkpoint_path, "policy_optimizer.pt"))
        
        if hasattr(self, 'reward_optimizer'):
            torch.save(self.reward_optimizer.state_dict(),
                    os.path.join(checkpoint_path, "reward_optimizer.pt"))
        
        if hasattr(self, 'baseline_optimizer'):
            torch.save(self.baseline_optimizer.state_dict(),
                    os.path.join(checkpoint_path, "baseline_optimizer.pt"))
        
        # Save metadata
        with open(os.path.join(checkpoint_path, "checkpoint_info.json"), "w") as f:
            json.dump(checkpoint_data, f, indent=2)
        
        print(f"Checkpoint saved to {checkpoint_path}")

    @staticmethod
    def _translate_reward_config(cfg: dict) -> dict:
        """Remap legacy reward_config keys to current RewardModel parameter names."""
        key_map = {
            'w_b':         'w_accuracy',
            'w_a':         'w_leak',        # answer-leak weight
            'w_s':         'w_preamble',    # structural penalty weight
            'w_f':         'w_leak',        # alternate old name
            'w_fact':      'w_grounded',
            'tau_answer':  'tau_veracity',
            'tau_preamble':'tau_preamble_words',
            'lambda_s':    None,  
        }
        translated = {}
        for k, v in cfg.items():
            new_key = key_map.get(k, k)
            if new_key is not None:
                translated[new_key] = v
        return translated

    @classmethod
    def load_from_checkpoint(cls, checkpoint_path, reward_config=None, base_model_path=None):
        """Load model from checkpoint"""
        # Load metadata
        with open(os.path.join(checkpoint_path, "checkpoint_info.json"), "r") as f:
            checkpoint_data = json.load(f)
        
        print(f"Loading checkpoint from stage: {checkpoint_data['stage']}")
        
        # Create trainer instance
        trainer = cls(
            model_path=base_model_path,
            reward_config=reward_config,
            use_baseline=checkpoint_data['use_baseline'],
            use_learnable_reward=checkpoint_data.get('use_learnable_reward', True),
            group_size=checkpoint_data.get('group_size', 2),
            beta=checkpoint_data.get('beta', 0.01),
            epsilon=checkpoint_data.get('epsilon', 0.2),
            skip_lora=True

        )

        from peft import PeftModel
        trainer.model = PeftModel.from_pretrained(trainer.model, checkpoint_path)
        print("LoRA adapter loaded")
        
        # Restore training state
        trainer.reward_history = checkpoint_data.get('reward_history', [])
        
        # Load components
        if checkpoint_data.get('has_reward_head', False):
            reward_head_path = os.path.join(checkpoint_path, "reward_head.pt")
            if os.path.exists(reward_head_path):
                trainer.reward_head.load_state_dict(
                    torch.load(reward_head_path, map_location=trainer.device)
                )
                print("Reward head loaded")
        
        if checkpoint_data.get('has_baseline_network', False):
            baseline_path = os.path.join(checkpoint_path, "baseline_network.pt")
            if os.path.exists(baseline_path):
                trainer.baseline_network.load_state_dict(
                    torch.load(baseline_path, map_location=trainer.device)
                )
                print("Baseline network loaded")
        
        policy_opt_path = os.path.join(checkpoint_path, "policy_optimizer.pt")
        if os.path.exists(policy_opt_path):
            try:
                trainer.policy_optimizer.load_state_dict(
                    torch.load(policy_opt_path, map_location=trainer.device)
                )
                print("Policy optimizer loaded")
            except (ValueError, RuntimeError):
                print("Policy optimizer mismatch - reinitializing")
        
        if hasattr(trainer, 'reward_optimizer'):
            reward_opt_path = os.path.join(checkpoint_path, "reward_optimizer.pt")
            if os.path.exists(reward_opt_path):
                try:
                    trainer.reward_optimizer.load_state_dict(
                        torch.load(reward_opt_path, map_location=trainer.device)
                    )
                    print("Reward optimizer loaded")
                except (ValueError, RuntimeError):
                    print("Reward optimizer mismatch - reinitializing")
        
        if hasattr(trainer, 'baseline_optimizer'):
            baseline_opt_path = os.path.join(checkpoint_path, "baseline_optimizer.pt")
            if os.path.exists(baseline_opt_path):
                try:
                    trainer.baseline_optimizer.load_state_dict(
                        torch.load(baseline_opt_path, map_location=trainer.device)
                    )
                    print("Baseline optimizer loaded")
                except (ValueError, RuntimeError):
                    print("Baseline optimizer mismatch - reinitializing")
        
        print(f"Checkpoint loaded from {checkpoint_path}")

        return trainer

    def save_checkpoint(self, checkpoint_dir, tag=None):
        """Save all trainable components to a checkpoint directory."""
        if tag:
            checkpoint_dir = os.path.join(checkpoint_dir, tag) if tag not in checkpoint_dir else checkpoint_dir
        os.makedirs(checkpoint_dir, exist_ok=True)
    
        # 1. Save LoRA adapter
        self.model.save_pretrained(checkpoint_dir)
        print(f"LoRA adapter saved to {checkpoint_dir}")
    
        # 2. Save reward head
        if hasattr(self, 'reward_head') and self.reward_head is not None:
            torch.save(self.reward_head.state_dict(),
                       os.path.join(checkpoint_dir, "reward_head.pt"))
            print("Reward head saved")
    
        # 3. Save baseline network
        if hasattr(self, 'baseline_network') and self.baseline_network is not None:
            torch.save(self.baseline_network.state_dict(),
                       os.path.join(checkpoint_dir, "baseline_network.pt"))
            print("Baseline network saved")
    
        # 4. Save optimizers
        torch.save(self.policy_optimizer.state_dict(),
                   os.path.join(checkpoint_dir, "policy_optimizer.pt"))
        print("Policy optimizer saved")
    
        if hasattr(self, 'reward_optimizer') and self.reward_optimizer is not None:
            torch.save(self.reward_optimizer.state_dict(),
                       os.path.join(checkpoint_dir, "reward_optimizer.pt"))
            print("Reward optimizer saved")
    
        if hasattr(self, 'baseline_optimizer') and self.baseline_optimizer is not None:
            torch.save(self.baseline_optimizer.state_dict(),
                       os.path.join(checkpoint_dir, "baseline_optimizer.pt"))
            print("Baseline optimizer saved")
    
        # 5. Save metadata (this is what load_from_checkpoint reads first)
        checkpoint_info = {
            'stage': 'stage1',
            'use_baseline': self.use_baseline,
            'use_learnable_reward': self.use_learnable_reward,
            'group_size': self.group_size,
            'beta': self.beta,
            'epsilon': self.epsilon,
            'has_reward_head': hasattr(self, 'reward_head') and self.reward_head is not None,
            'has_baseline_network': hasattr(self, 'baseline_network') and self.baseline_network is not None,
            'reward_history': self.reward_history,
        }
        with open(os.path.join(checkpoint_dir, "checkpoint_info.json"), 'w') as f:
            json.dump(checkpoint_info, f, indent=2)
        print("Checkpoint info saved")
    
        print(f"Checkpoint saved to {checkpoint_dir}")
        return checkpoint_dir

In [ ]:
class FormatWarmupDataset(Dataset):
    """Dataset with correctly formatted examples for SFT warmup"""
    
    def __init__(self, data_path, tokenizer, max_length=768):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.examples = []
    
        data_processor = MedQADataProcessor(tokenizer)
        raw_data = data_processor.load_medqa_data(data_path)
    
        for item in raw_data:
            prompt = item['prompt']
            target = self._create_target_response_from_item(item)
            if target:
                self.examples.append({
                    'prompt': prompt,
                    'target': target,
                    'full_text': prompt + target,
                })
        print(f"Created {len(self.examples)} warmup examples")
    
    def __len__(self):
        return len(self.examples)
    
    def __getitem__(self, idx):
        example = self.examples[idx]
        
        # Tokenize full sequence
        encoding = self.tokenizer(
            example['full_text'],
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt'
        )
        
        # Tokenize prompt only (to know where to start computing loss)
        prompt_encoding = self.tokenizer(
            example['prompt'],
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )
        prompt_length = prompt_encoding['input_ids'].shape[1]
        
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'prompt_length': prompt_length
        }

    def _create_target_response_from_item(self, item):
        import random
        correct_answer = item['correct_answer']
        correct_text = item['options'].get(correct_answer, '')
    
        template_sets = [
            [
                "The clinical presentation described is consistent with the pathophysiology of the correct diagnosis.",
                f"{correct_text} is supported by the key findings mentioned in the case.",
                "The differential diagnosis should consider the patient's presenting symptoms.",
                "Laboratory and clinical findings help narrow the diagnostic possibilities.",
                "The correct answer aligns with established medical guidelines.",
                "Other options are less likely given the specific clinical context provided.",
            ],
            [
                "The presenting symptoms point toward a specific underlying mechanism.",
                f"{correct_text} is the most appropriate choice based on the clinical scenario.",
                "The patient's history and examination findings support this conclusion.",
                "Key diagnostic features help distinguish this condition from similar presentations.",
                "Treatment selection depends on the underlying pathophysiology.",
                "The remaining options do not adequately explain all the findings described.",
            ],
            [
                "The described findings are characteristic of a well-defined clinical entity.",
                "The mechanism of action is directly relevant to the patient's presentation.",
                f"{correct_text} addresses the primary clinical concern in this scenario.",
                "Risk factors and epidemiological data support this diagnostic consideration.",
                "Standard-of-care guidelines inform the appropriate management strategy.",
                "Alternative diagnoses fail to account for one or more critical findings.",
            ],
            [
                "Anatomical and physiological principles explain the observed clinical features.",
                "The temporal progression of symptoms is consistent with the expected disease course.",
                f"Evidence-based practice supports {correct_text} in this clinical context.",
                "The pathological process involves specific organ systems and cellular mechanisms.",
                "Diagnostic workup findings corroborate the clinical impression.",
                "Competing diagnoses are excluded based on the available clinical information.",
            ],
        ]
    
        facts_list = random.choice(template_sets)
        random.shuffle(facts_list)
        num_facts = random.randint(5, 7)
        facts_list = facts_list[:num_facts]
    
        facts_text = "\n".join(f"{i+1}. {f}" for i, f in enumerate(facts_list))
        return f"<facts>\n{facts_text}\n</facts>\n<answer>{correct_answer}</answer>"


def sft_warmup_train(
    model,
    tokenizer,
    train_data_path,
    num_epochs=2,
    batch_size=2,
    learning_rate=1e-5,
    device='cuda'
):
    """
    Run SFT warmup to teach format before GRPO
    
    Args:
        model: Your policy model (already loaded with LoRA)
        tokenizer: Tokenizer
        train_data_path: Path to MedQA JSON
        num_epochs: 1-3 epochs is usually enough for format learning
        batch_size: Keep small for memory
        learning_rate: Lower than GRPO (1e-5 to 5e-5)
        device: cuda/cpu
    """
    print("="*60)
    print("SFT WARMUP: Teaching model the output format")
    print("="*60)
    
    # Create dataset
    dataset = FormatWarmupDataset(train_data_path, tokenizer)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    # Optimizer (only for trainable params - LoRA weights)
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable_params, lr=learning_rate)
    
    model.train()
    
    for epoch in range(num_epochs):
        total_loss = 0
        num_batches = 0
        
        progress = tqdm(dataloader, desc=f"Warmup Epoch {epoch+1}/{num_epochs}")
        
        for batch in progress:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            prompt_lengths = batch['prompt_length']
            
            # Forward pass
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
            logits = outputs.logits
            
            # Compute loss only on response tokens (not prompt)
            loss = compute_sft_loss(
                logits, 
                input_ids, 
                prompt_lengths,
                tokenizer.pad_token_id
            )
            
            # Backward
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
            optimizer.step()
            
            total_loss += loss.item()
            num_batches += 1
            
            progress.set_postfix({'loss': f'{loss.item():.4f}'})
        
        avg_loss = total_loss / num_batches
        print(f"Epoch {epoch+1} complete. Avg Loss: {avg_loss:.4f}")
        
        # Quick validation: generate a sample to check format
        validate_format_learning(model, tokenizer, dataset, device)
    
    print("="*60)
    print("SFT Warmup complete")
    print("="*60)
    
    return model


def compute_sft_loss(logits, labels, prompt_lengths, pad_token_id):
    """
    Compute cross-entropy loss only on response tokens
    """
    batch_size, seq_len, vocab_size = logits.shape
    
    # Shift for next-token prediction
    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = labels[:, 1:].contiguous()
    
    # Create mask: 1 for response tokens, 0 for prompt/padding
    loss_mask = torch.ones_like(shift_labels, dtype=torch.float)
    
    for i, prompt_len in enumerate(prompt_lengths):
        # Mask out prompt tokens (don't compute loss on them)
        loss_mask[i, :prompt_len-1] = 0
        # Mask out padding
        loss_mask[i, shift_labels[i] == pad_token_id] = 0
    
    # Compute per-token loss
    loss_fct = nn.CrossEntropyLoss(reduction='none')
    loss = loss_fct(
        shift_logits.view(-1, vocab_size),
        shift_labels.view(-1)
    )
    loss = loss.view(batch_size, -1)
    
    # Apply mask and average
    masked_loss = (loss * loss_mask).sum() / (loss_mask.sum() + 1e-8)
    
    return masked_loss


def validate_format_learning(model, tokenizer, dataset, device, num_samples=2):
    model.eval()
    
    print("\n--- Format Validation ---")
    
    for i in range(min(num_samples, len(dataset))):
        example = dataset.examples[i]
        prompt = example['prompt']
        
        inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1024)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=300,
                temperature=0.7,
                do_sample=True,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )
        
        response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        
        # Check format
        has_facts = bool(re.search(r'<facts>', response, re.IGNORECASE))
        has_answer = bool(re.search(r'<answer>\s*[A-D]\s*</answer>', response, re.IGNORECASE))
        
        status = "✓" if (has_facts and has_answer) else "✗"
        print(f"Sample {i+1}: {status} facts={has_facts}, answer={has_answer}")
        print(f"Response preview: {response[:200]}...")
        print()
    
    model.train()

def add_warmup_to_trainer(trainer_class):
    
    def run_sft_warmup(self, train_data_path, num_epochs=2, learning_rate=1e-5):
        """
        Run SFT warmup before GRPO training
        Usage:
            trainer = PolicyTrainer(...)
            trainer.run_sft_warmup("medqa_train.json", num_epochs=2)
            trainer.train_policy_stage2(...)  # Now GRPO will work
        """
        print("\n" + "="*60)
        print("STAGE 0: SFT WARMUP (Format Learning)")
        print("="*60)
        
        self.model = sft_warmup_train(
            model=self.model,
            tokenizer=self.tokenizer,
            train_data_path=train_data_path,
            num_epochs=num_epochs,
            learning_rate=learning_rate,
            device=self.device
        )
        
        print("Warmup complete. Proceeding to GRPO...\n")
    
    trainer_class.run_sft_warmup = run_sft_warmup
    return trainer_class

In [ ]:
class RewardModelTrainingPipeline:
    def __init__(self, 
                 model_path,
                 group_size,
                 beta,
                 reward_config=None,
                 use_baseline=True,
                 use_learnable_reward=True,
                 output_dir="./stage1_reward_training"):
        
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)
        
        print("="*60)
        print("STAGE 1: REWARD MODEL TRAINING PIPELINE")
        print("="*60)
        
        # Initialize trainer (reuse PolicyTrainer components)
        self.trainer = PolicyTrainer(
            model_path=model_path,
            reward_config=reward_config,
            use_baseline=use_baseline,
            use_learnable_reward=use_learnable_reward,
            group_size=group_size,
            beta=beta,
            log_file=os.path.join(output_dir, "training.log")
        )
        
        #Initialize dedicated MetricsTracker for Stage 1
        self.metrics_tracker = MetricsTracker()
        
        print(f"Stage 1 pipeline initialized")
        print(f"Output directory: {output_dir}")
    
    def run(self, 
            train_data_path,
            test_data_path,
            stage1_epochs,
            stage2_epochs,
            batch_size,
            max_eval_examples):
        """
        Run complete Stage 1 training with full metric tracking
        """
        print("\n" + "="*60)
        print("RUNNING STAGE 1: REWARD MODEL TRAINING")
        print("="*60)
        
        start_time = time.time()
        
        # ========================================
        # Step 1: Initial Evaluation (Epoch 0)
        # ========================================
        print("\n[STAGE 1.0] Initial Evaluation (Pre-Training)...")
        print("-" * 60)
        initial_metrics = self.trainer.evaluate_model(
            test_data_path=test_data_path,
            max_examples=max_eval_examples
        )
        
        #  Record initial metrics as epoch 0
        self.metrics_tracker.record_epoch(0, initial_metrics)
        self.metrics_tracker.print_metrics_with_std(
            initial_metrics, 
            "INITIAL METRICS (PRE-TRAINING)"
        )

        #========================================
        #Step 1b: SFT Warmup (format learning)
        #========================================
        
        print("\n[STAGE 1.1] SFT Warmup for format learning...")
        print("-" * 60)
        self.trainer.model = sft_warmup_train(
            model=self.trainer.model,
            tokenizer=self.trainer.tokenizer,
            train_data_path=train_data_path,
            num_candidates=4,
        )
        
        self.trainer.policy_optimizer = torch.optim.AdamW(
            [p for p in self.trainer.model.parameters() if p.requires_grad],
            lr=1e-6
        )
        
        # ========================================
        # Step 2: Train Reward Head (Stage 1.1)
        # ========================================
        print("\n[STAGE 1.2] Training Reward Head...")
        print("-" * 60)
        self.trainer.reward_function.fact_verification_system.training_mode = True
        
        reward_head_start = time.time()
        self.trainer.train_reward_model_stage1(
            train_data_path=train_data_path,
            num_epochs=stage1_epochs,
            batch_size=batch_size
        )
        reward_head_time = time.time() - reward_head_start
        
        print("\n[STAGE 1.2-EVAL] Evaluating after Reward Head Training...")
        self.trainer.reward_function.fact_verification_system.training_mode = False
        
        post_reward_head_metrics = self.trainer.evaluate_model(
            test_data_path=test_data_path,
            max_examples=max_eval_examples
        )
        
        self.trainer.reward_function.fact_verification_system.training_mode = True
        
        epoch_offset = stage1_epochs
        self.metrics_tracker.record_epoch(epoch_offset, post_reward_head_metrics)
        self.metrics_tracker.print_metrics_with_std(
            post_reward_head_metrics,
            f"AFTER REWARD HEAD TRAINING (EPOCH {epoch_offset})"
        )
        
        # ========================================
        # Step 3: Train Policy with GRPO (Stage 1.2)
        # ========================================
        print("\n[STAGE 1.2] Training Policy with GRPO...")
        print("-" * 60)
        
        policy_start = time.time()
        self.trainer.train_policy_stage2(
            train_data_path, num_epochs=2,
            batch_size=4,
            kl_threshold_batch=0.5,
            kl_threshold_epoch=0.3,
            adaptive_kl=True,
            kl_target=0.05,
        )
        policy_time = time.time() - policy_start
        
        # ========================================
        # Step 4: Final Evaluation (Stage 1.3)
        # ========================================
        print("\n[STAGE 1.3] Final Evaluation...")
        print("-" * 60)
        self.trainer.reward_function.fact_verification_system.training_mode = False
        final_metrics = self.trainer.evaluate_model(
            test_data_path=test_data_path,
            max_examples=max_eval_examples
        )
        
        # Record final metrics
        final_epoch = epoch_offset + stage2_epochs
        self.metrics_tracker.record_epoch(final_epoch, final_metrics)
        self.metrics_tracker.print_metrics_with_std(
            final_metrics,
            f"FINAL METRICS (EPOCH {final_epoch})"
        )
        print("Stage 1 Complete")
        
        # ========================================
        # Print Comparison: Initial vs Final
        # ========================================
        print("\n" + "="*60)
        print("STAGE 1 TRAINING COMPARISON")
        print("="*60)
        
        improvement = {
            'accuracy': final_metrics['accuracy'] - initial_metrics['accuracy'],
            'answer_penalty': initial_metrics['avg_answer_penalty'] - final_metrics['avg_answer_penalty'],
            'structural_penalty': initial_metrics['avg_structural_penalty'] - final_metrics['avg_structural_penalty'],
            'factual_reward': final_metrics['avg_factual_reward'] - initial_metrics['avg_factual_reward'],
            'total_reward': final_metrics['avg_total_reward'] - initial_metrics['avg_total_reward']
        }
        
        print(f"\nPerformance Improvements:")
        print(f"   Accuracy:            {initial_metrics['accuracy']:.2%} → {final_metrics['accuracy']:.2%} ({improvement['accuracy']:+.2%})")
        print(f"   Answer Penalty:      {initial_metrics['avg_answer_penalty']:.2f} → {final_metrics['avg_answer_penalty']:.2f} ({improvement['answer_penalty']:+.2f})")
        print(f"   Structural Penalty:  {initial_metrics['avg_structural_penalty']:.2f} → {final_metrics['avg_structural_penalty']:.2f} ({improvement['structural_penalty']:+.2f})")
        print(f"   Factual Reward:      {initial_metrics['avg_factual_reward']:.2f} → {final_metrics['avg_factual_reward']:.2f} ({improvement['factual_reward']:+.2f})")
        print(f"   Total Reward:        {initial_metrics['avg_total_reward']:.2f} → {final_metrics['avg_total_reward']:.2f} ({improvement['total_reward']:+.2f})")
        
        # ========================================
        # Step 5: Save Checkpoint
        # ========================================
        checkpoint_path = os.path.join(self.output_dir, "stage1_checkpoint")
        print(f"\n[STAGE 1.4] Saving Checkpoint...")
        print("-" * 60)
        self.trainer.save_checkpoint(checkpoint_path)
        
        # Save metadata for Stage 2 to validate
        metadata = {
            'initial_metrics': initial_metrics,
            'final_metrics': final_metrics,
            'improvement': improvement,
            'ready_for_stage2': True,
        }
        with open(os.path.join(checkpoint_path, "stage1_metadata.json"), 'w') as f:
            json.dump(metadata, f, indent=2, default=str)
        
        elapsed_time = time.time() - start_time
        
        # ========================================
        # Save All Metrics
        # ========================================
        metrics_path = os.path.join(self.output_dir, "stage1_metrics.json")
        self.metrics_tracker.save_metrics(metrics_path)
        print(f" All metrics saved to: {metrics_path}")
        
        # Save detailed summary
        summary = {
            'stage': 'stage1_complete',
            'model_path': self.trainer.model.config._name_or_path,
            'stage1_epochs': stage1_epochs,
            'stage2_epochs': stage2_epochs,
            'batch_size': batch_size,
            'initial_metrics': initial_metrics,
            'post_reward_head_metrics': post_reward_head_metrics,
            'final_metrics': final_metrics,
            'improvement': improvement,
            'timing': {
                'reward_head_training': reward_head_time,
                'policy_training': policy_time,
                'total_time': elapsed_time
            },
            'checkpoint_path': checkpoint_path
        }
        
        summary_path = os.path.join(self.output_dir, "stage1_summary.json")
        with open(summary_path, 'w') as f:
            json.dump(summary, f, indent=2)
        
        # ========================================
        # Final Summary
        # ========================================
        print("\n" + "="*60)
        print("STAGE 1 COMPLETE")
        print("="*60)
        print(f"   Training time:")
        print(f"   Reward head: {reward_head_time/60:.1f} minutes")
        print(f"   Policy (GRPO): {policy_time/60:.1f} minutes")
        print(f"   Total: {elapsed_time/60:.1f} minutes")
        print(f"\n Outputs:")
        print(f"   Checkpoint: {checkpoint_path}")
        print(f"   Metrics: {metrics_path}")
        print(f"   Summary: {summary_path}")
        print(f"\n Ready for Stage 2 (Adversarial Training)")
        print(f"   Load from: {checkpoint_path}")
        
        return summary

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import json
import re
import random
from tqdm import tqdm


# ---------------------------------------------------------------------------
# Content quality filter — rejects vacuous / generic facts
# ---------------------------------------------------------------------------
GENERIC_PHRASES = [
    "the clinical presentation",
    "is consistent with",
    "the pathophysiology of the correct",
    "is supported by the key findings",
    "the differential diagnosis should consider",
    "help narrow the diagnostic",
    "aligns with established medical guidelines",
    "less likely given the specific clinical context",
    "presenting symptoms point toward",
    "most appropriate choice based on the clinical scenario",
    "history and examination findings support this conclusion",
    "key diagnostic features help distinguish",
    "treatment selection depends on the underlying",
    "do not adequately explain all the findings",
    "characteristic of a well-defined clinical entity",
    "mechanism of action is directly relevant",
    "addresses the primary clinical concern",
    "risk factors and epidemiological data support",
    "standard-of-care guidelines inform",
    "fail to account for one or more critical findings",
    "anatomical and physiological principles explain",
    "temporal progression of symptoms is consistent",
    "evidence-based practice supports",
    "pathological process involves specific organ",
    "diagnostic workup findings corroborate",
    "competing diagnoses are excluded",
    "the patient's history is relevant",
    "the patient's history supports this conclusion",
    "the patient's history is consistent with",
]


def is_generic_fact(fact_text):
    """Check if a fact is a vacuous template rather than real clinical content."""
    fact_lower = fact_text.lower().strip()
    # Check against known generic phrases
    for phrase in GENERIC_PHRASES:
        if phrase in fact_lower:
            return True
    # Also flag very short facts (less than 8 words) as likely vacuous
    word_count = len(fact_lower.split())
    if word_count < 6:
        return True
    return False


def extract_facts_from_response(response):
    """Extract individual facts from a <facts>...</facts> block."""
    facts_match = re.search(r'<facts>(.*?)</facts>', response, re.DOTALL | re.IGNORECASE)
    if not facts_match:
        return []
    facts_block = facts_match.group(1).strip()
    # Split by numbered lines
    facts = re.findall(r'\d+\.\s*(.+?)(?=\n\d+\.|\Z)', facts_block, re.DOTALL)
    return [f.strip() for f in facts if f.strip()]
    
def dedupe_facts(facts, threshold=0.85):
    if len(facts) <= 1:
        return list(facts)
    embedding_model = EmbeddingModelManager().get_model()
    embeds = embedding_model.encode(facts, normalize_embeddings=True)
    keep = []
    keep_embeds = []
    for f, e in zip(facts, embeds):
        if not keep_embeds or max(np.dot(e, ke) for ke in keep_embeds) < threshold:
            keep.append(f); keep_embeds.append(e)
    return keep

def response_has_substance(response, min_non_generic=3):
    """
    Check that a response contains enough non-generic clinical facts.
    Returns True only if at least `min_non_generic` facts are substantive.
    """
    facts = extract_facts_from_response(response)
    facts = dedupe_facts(facts)
    if len(facts) < 3:
        return False
    non_generic_count = sum(1 for f in facts if not is_generic_fact(f))
    return non_generic_count >= min_non_generic


def validate_response_format(response):
    """Check that response has proper <facts>...</facts> and <answer>X</answer> format."""
    has_facts = bool(re.search(r'<facts>', response, re.IGNORECASE))
    has_answer = bool(re.search(r'<answer>\s*[A-J]\s*</answer>', response, re.IGNORECASE))
    return has_facts and has_answer


def extract_answer_from_response(response):
    """Extract the answer letter from <answer>X</answer>."""
    match = re.search(r'<answer>\s*([A-J])\s*</answer>', response, re.IGNORECASE)
    return match.group(1).upper() if match else None


# ---------------------------------------------------------------------------
# Phase 1: Rejection sampling from the base model
# ---------------------------------------------------------------------------
def generate_candidates_from_base_model(
    model, tokenizer, items, num_candidates=4, max_new_tokens=768,
    temperature=0.7, top_p=0.9, device='cuda', cache_dir='./rs_cache'
):
    os.makedirs(cache_dir, exist_ok=True)

    # --- Build a cache key that actually reflects what affects the output ---
    # Hash the prompts so a different dataset slice gets a different cache.
    prompts_hash = hashlib.md5(
        "||".join(item['prompt'] for item in items).encode()
    ).hexdigest()[:10]

    # Try to identify the model. name_or_path works for HF models.
    model_id = getattr(model.config, '_name_or_path', 'unknown').replace('/', '_')

    cache_path = os.path.join(
        cache_dir,
        f"rs_{model_id}_{len(items)}_{num_candidates}_{max_new_tokens}_{prompts_hash}.pkl"
    )

    # --- Cache hit ---
    if os.path.exists(cache_path):
        print(f"[rejection sampling] cache hit: {cache_path}")
        with open(cache_path, "rb") as f:
            return pickle.load(f)

    model.eval()
    candidates = {}
    print(f"Generating {num_candidates} candidates per question for {len(items)} questions...")


    for idx, item in enumerate(tqdm(items, desc="Rejection sampling")):
        prompt = item['prompt']
        correct_answer = item['correct_answer'].upper()
        item_candidates = []

        inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1536)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        for _ in range(num_candidates):
            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    temperature=temperature,
                    top_p=top_p,
                    do_sample=True,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )

            response = tokenizer.decode(
                outputs[0][inputs['input_ids'].shape[1]:],
                skip_special_tokens=True
            ).strip()

            # Truncate after </answer> to avoid trailing garbage
            answer_end = re.search(r'</answer>', response, re.IGNORECASE)
            if answer_end:
                response = response[:answer_end.end()]

            has_format = validate_response_format(response)
            extracted = extract_answer_from_response(response)
            is_correct = (extracted == correct_answer) if extracted else False
            has_content = response_has_substance(response) if has_format else False

            item_candidates.append({
                'response': response,
                'has_format': has_format,
                'is_correct': is_correct,
                'has_substance': has_content,
            })

        candidates[idx] = item_candidates

    model.train()
    with open(cache_path, "wb") as f:
        pickle.dump(candidates, f)
    print(f"[rejection sampling] saved cache: {cache_path}")
    return candidates

def select_best_candidate(candidates_for_item, correct_answer):
    """
    Priority order:
    1. Correct format + correct answer + substantive facts
    2. Correct format + substantive facts (wrong answer — we fix the answer letter)
    3. Correct format + correct answer (even if generic — still better than pure template)
    4. None — will need teacher or question-aware fallback
    """
    # Priority 1: Perfect candidate
    for c in candidates_for_item:
        if c['has_format'] and c['is_correct'] and c['has_substance']:
            return c['response'], 'base_model_perfect'

    # Priority 2: Good reasoning, wrong answer — fix the answer
    for c in candidates_for_item:
        if c['has_format'] and c['has_substance']:
            fixed = re.sub(
                r'<answer>\s*[A-J]\s*</answer>',
                f'<answer>{correct_answer}</answer>',
                c['response'],
                flags=re.IGNORECASE
            )
            return fixed, 'base_model_answer_fixed'

    # Priority 3: Correct but generic
    for c in candidates_for_item:
        if c['has_format'] and c['is_correct']:
            return c['response'], 'base_model_generic'

    return None, 'needs_fallback'


# ---------------------------------------------------------------------------
# Phase 2: Teacher model fallback (uses the judge LLM)
# ---------------------------------------------------------------------------
TEACHER_PROMPT_TEMPLATE = """You are a medical expert. Given the following clinical question and the correct answer, provide a structured reasoning response.

Question: {question}

Options:
{options_text}

The correct answer is: {correct_answer_letter} ({correct_answer_text})

Respond EXACTLY in this format:
<facts>
1. [First verifiable medical fact supporting the answer]
2. [Second verifiable medical fact]
3. [Third verifiable medical fact]
4. [Fourth verifiable medical fact]
5. [Fifth verifiable medical fact]
6. [Sixth verifiable medical fact]
</facts>
<answer>{correct_answer_letter}</answer>

Requirements for each fact:
- Must be a specific, verifiable medical statement
- Must directly support why the correct answer is right
- Use general medical knowledge, not case-specific observations
- Example: "Nitrofurantoin is safe in second trimester pregnancy" NOT "The patient's history supports this"
"""


def generate_teacher_response(
    teacher_model, teacher_tokenizer, item, device='cuda', max_new_tokens=768):
    """Generate a reasoning trace from the teacher/judge model."""
    correct_answer = item['correct_answer']
    correct_text = item['options'].get(correct_answer, '')

    options_text = "\n".join(f"{k}: {v}" for k, v in item['options'].items())

    teacher_prompt = TEACHER_PROMPT_TEMPLATE.format(
        question=item['question'],
        options_text=options_text,
        correct_answer_letter=correct_answer,
        correct_answer_text=correct_text,
    )

    messages = [
        {"role": "system", "content": "You are a medical reasoning expert."},
        {"role": "user", "content": teacher_prompt},
    ]

    formatted = teacher_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    inputs = teacher_tokenizer(formatted, return_tensors='pt', truncation=True, max_length=768)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = teacher_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.3,  # Lower temperature for more reliable output
            top_p=0.9,
            do_sample=True,
            pad_token_id=teacher_tokenizer.pad_token_id,
            eos_token_id=teacher_tokenizer.eos_token_id,
        )

    response = teacher_tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    ).strip()

    # Truncate after </answer>
    answer_end = re.search(r'</answer>', response, re.IGNORECASE)
    if answer_end:
        response = response[:answer_end.end()]

    return response


# ---------------------------------------------------------------------------
# Phase 3: Question-aware template fallback (last resort)
# ---------------------------------------------------------------------------
def create_question_aware_template(item):
    """
    Last-resort fallback that's at least question-specific.
    Extracts entities from the question and options to build semi-specific facts.
    Still not ideal, but vastly better than fully generic templates.
    """
    correct_answer = item['correct_answer']
    correct_text = item['options'].get(correct_answer, '')
    question = item['question']

    # Extract key medical terms from question (simple heuristic)
    # Look for age, gender, symptoms, lab values, etc.
    age_match = re.search(r'(\d+)[- ]year[- ]old', question)
    gender_match = re.search(r'\b(male|female|man|woman|boy|girl)\b', question, re.IGNORECASE)

    age_str = f"A {age_match.group(1)}-year-old" if age_match else "The"
    gender_str = gender_match.group(1).lower() if gender_match else "patient"

    # Get the wrong options for contrast
    wrong_options = [v for k, v in item['options'].items() if k != correct_answer]
    random.shuffle(wrong_options)
    wrong_example = wrong_options[0] if wrong_options else "alternative diagnoses"

    facts = [
        f"{correct_text} is the most likely answer given the clinical scenario described.",
        f"{age_str} {gender_str} presentation is consistent with findings typically associated with {correct_text}.",
        f"{correct_text} can be differentiated from {wrong_example} based on the specific clinical features mentioned.",
        f"The key findings in this case directly support {correct_text} as the correct diagnosis or management.",
        f"{wrong_example} would be expected to present with different clinical features than those described.",
        f"Medical guidelines and established clinical evidence support {correct_text} in this context.",
    ]

    random.shuffle(facts)
    num_facts = random.randint(5, 6)
    facts = facts[:num_facts]

    facts_text = "\n".join(f"{i+1}. {f}" for i, f in enumerate(facts))
    return f"<facts>\n{facts_text}\n</facts>\n<answer>{correct_answer}</answer>"


# ---------------------------------------------------------------------------
# Main Dataset class with multi-phase target generation
# ---------------------------------------------------------------------------
class FormatWarmupDataset(Dataset):
    def __init__(self, data_path, tokenizer, max_length=768,
                 model=None, teacher_model=None, teacher_tokenizer=None,
                 num_candidates=4, device='cuda'):
        """
        Args:
            data_path: Path to MedQA JSON
            tokenizer: Policy model tokenizer
            max_length: Max sequence length
            model: Base policy model (for rejection sampling). If None, uses templates.
            teacher_model: Teacher/judge model (for fallback). If None, skips teacher phase.
            teacher_tokenizer: Teacher model tokenizer.
            num_candidates: Number of candidates to sample per question.
            device: cuda/cpu
        """
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.examples = []

        # MedQADataProcessor must be defined/imported in the calling code
        data_processor = MedQADataProcessor(tokenizer)
        raw_data = data_processor.load_medqa_data(data_path)
        #raw_data = raw_data[:2]

        stats = {'base_model_perfect': 0, 'base_model_answer_fixed': 0,
                 'base_model_generic': 0, 'teacher': 0, 'template_fallback': 0}

        # --- Phase 1: Rejection sampling from base model ---
        if model is not None:
            candidates = generate_candidates_from_base_model(
                model, tokenizer, raw_data,
                num_candidates=num_candidates,
                device=device
            )
        else:
            candidates = {}

        # --- Phase 2 & 3: Select best or fall back ---
        teacher_needed_indices = []

        for idx, item in enumerate(raw_data):
            prompt = item['prompt']
            correct_answer = item['correct_answer']

            # Try base model candidates first
            if idx in candidates:
                target, source = select_best_candidate(
                    candidates[idx], correct_answer
                )
            else:
                target, source = None, 'needs_fallback'

            if target is not None and source != 'needs_fallback':
                stats[source] += 1
                self.examples.append({
                    'prompt': prompt,
                    'target': target,
                    'full_text': prompt + target,
                    'source': source,
                })
            else:
                teacher_needed_indices.append(idx)

        # --- Phase 2: Teacher model for remaining ---
        if teacher_model is not None and teacher_tokenizer is not None and teacher_needed_indices:
            print(f"\nGenerating teacher responses for {len(teacher_needed_indices)} questions...")
            teacher_model.eval()

            for idx in tqdm(teacher_needed_indices, desc="Teacher generation"):
                item = raw_data[idx]
                prompt = item['prompt']

                teacher_response = generate_teacher_response(
                    teacher_model, teacher_tokenizer, item, device=device
                )

                if validate_response_format(teacher_response):
                    # Ensure correct answer letter
                    extracted = extract_answer_from_response(teacher_response)
                    if extracted != item['correct_answer'].upper():
                        teacher_response = re.sub(
                            r'<answer>\s*[A-J]\s*</answer>',
                            f'<answer>{item["correct_answer"]}</answer>',
                            teacher_response,
                            flags=re.IGNORECASE
                        )

                    stats['teacher'] += 1
                    self.examples.append({
                        'prompt': prompt,
                        'target': teacher_response,
                        'full_text': prompt + teacher_response,
                        'source': 'teacher',
                    })
                else:
                    # Teacher also failed format — use question-aware template
                    template = create_question_aware_template(item)
                    stats['template_fallback'] += 1
                    self.examples.append({
                        'prompt': prompt,
                        'target': template,
                        'full_text': prompt + template,
                        'source': 'template_fallback',
                    })
        else:
            # No teacher available — use question-aware templates for all remaining
            for idx in teacher_needed_indices:
                item = raw_data[idx]
                prompt = item['prompt']
                template = create_question_aware_template(item)
                stats['template_fallback'] += 1
                self.examples.append({
                    'prompt': prompt,
                    'target': template,
                    'full_text': prompt + template,
                    'source': 'template_fallback',
                })

        # Print statistics
        total = len(self.examples)
        print(f"\n{'='*50}")
        print(f"SFT Warmup Dataset Statistics ({total} examples)")
        print(f"{'='*50}")
        print(f"  Base model (perfect):      {stats['base_model_perfect']:>4d}  ({100*stats['base_model_perfect']/max(total,1):.1f}%)")
        print(f"  Base model (answer fixed): {stats['base_model_answer_fixed']:>4d}  ({100*stats['base_model_answer_fixed']/max(total,1):.1f}%)")
        print(f"  Base model (generic):      {stats['base_model_generic']:>4d}  ({100*stats['base_model_generic']/max(total,1):.1f}%)")
        print(f"  Teacher generated:         {stats['teacher']:>4d}  ({100*stats['teacher']/max(total,1):.1f}%)")
        print(f"  Template fallback:         {stats['template_fallback']:>4d}  ({100*stats['template_fallback']/max(total,1):.1f}%)")
        print(f"{'='*50}")

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        example = self.examples[idx]

        encoding = self.tokenizer(
            example['full_text'],
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt'
        )

        prompt_encoding = self.tokenizer(
            example['prompt'],
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )
        prompt_length = prompt_encoding['input_ids'].shape[1]

        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'prompt_length': prompt_length
        }


# ---------------------------------------------------------------------------
# Training loop (mostly unchanged, signature updated)
# ---------------------------------------------------------------------------
def sft_warmup_train(
    model,
    tokenizer,
    train_data_path,
    num_epochs=2,
    batch_size=2,
    learning_rate=1e-5,
    device='cuda',
    num_candidates=4,
    teacher_model=None,
    teacher_tokenizer=None,
):
    """
    Run SFT warmup with rejection sampling.

    Args:
        model: Policy model (used both for candidate generation and training)
        tokenizer: Policy tokenizer
        train_data_path: Path to MedQA JSON
        num_epochs: 1-3 epochs
        batch_size: Keep small for memory
        learning_rate: 1e-5 to 5e-5
        device: cuda/cpu
        num_candidates: How many responses to sample per question (4 is usually enough)
        teacher_model: Optional judge/teacher model for fallback generation
        teacher_tokenizer: Optional teacher tokenizer
    """
    print("=" * 60)
    print("SFT WARMUP: Rejection Sampling + Format Learning")
    print("=" * 60)

    # Create dataset with rejection sampling
    dataset = FormatWarmupDataset(
        train_data_path, tokenizer,
        model=model,
        teacher_model=teacher_model,
        teacher_tokenizer=teacher_tokenizer,
        num_candidates=num_candidates,
        device=device,
    )
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    # Optimizer
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable_params, lr=learning_rate)

    model.train()

    for epoch in range(num_epochs):
        total_loss = 0
        num_batches = 0

        progress = tqdm(dataloader, desc=f"Warmup Epoch {epoch+1}/{num_epochs}")

        for batch in progress:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            prompt_lengths = batch['prompt_length']

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits

            loss = compute_sft_loss(logits, input_ids, prompt_lengths, tokenizer.pad_token_id)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
            optimizer.step()

            total_loss += loss.item()
            num_batches += 1
            progress.set_postfix({'loss': f'{loss.item():.4f}'})

        avg_loss = total_loss / num_batches
        print(f"Epoch {epoch+1} complete. Avg Loss: {avg_loss:.4f}")

        validate_format_learning(model, tokenizer, dataset, device)

    print("=" * 60)
    print("SFT Warmup complete! Model should now produce correct format.")
    print("=" * 60)

    return model


def compute_sft_loss(logits, labels, prompt_lengths, pad_token_id):
    """Compute cross-entropy loss only on response tokens."""
    batch_size, seq_len, vocab_size = logits.shape

    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = labels[:, 1:].contiguous()

    loss_mask = torch.ones_like(shift_labels, dtype=torch.float)

    for i, prompt_len in enumerate(prompt_lengths):
        loss_mask[i, :prompt_len-1] = 0
        loss_mask[i, shift_labels[i] == pad_token_id] = 0

    loss_fct = nn.CrossEntropyLoss(reduction='none')
    loss = loss_fct(
        shift_logits.view(-1, vocab_size),
        shift_labels.view(-1)
    )
    loss = loss.view(batch_size, -1)

    masked_loss = (loss * loss_mask).sum() / (loss_mask.sum() + 1e-8)
    return masked_loss


def validate_format_learning(model, tokenizer, dataset, device, num_samples=2):
    """Quick check that model is learning the format."""
    model.eval()
    print("\n--- Format Validation ---")

    for i in range(min(num_samples, len(dataset))):
        example = dataset.examples[i]
        prompt = example['prompt']
        source = example.get('source', 'unknown')

        inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1024)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=300,
                temperature=0.7,
                do_sample=True,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

        response = tokenizer.decode(
            outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
        )

        has_facts = bool(re.search(r'<facts>', response, re.IGNORECASE))
        has_answer = bool(re.search(r'<answer>\s*[A-J]\s*</answer>', response, re.IGNORECASE))
        has_substance = response_has_substance(response) if has_facts else False

        status = "✓" if (has_facts and has_answer) else "✗"
        substance_status = "substantive" if has_substance else "generic"
        print(f"Sample {i+1} (trained on: {source}): {status} facts={has_facts}, answer={has_answer}, content={substance_status}")
        print(f"Response preview: {response[:250]}...")
        print()

    model.train()

# Adversarial Trainer

In [ ]:
def fine_tune_reward_head(
    self,
    preference_pairs,
    num_epochs=3,
    lr=1e-5,
    margin=0.5,
    lambda_align=0.3,
    max_pairs_per_epoch=64,
    log_every=10,
):
    """
    Fine-tune the learnable reward head using adversarial preference pairs.

    For each pair (chosen, rejected) the loss is:

        L_pref   = -log σ( r_chosen - r_rejected - margin )
        L_align  = MSE(r_chosen, R_rule_chosen) + MSE(r_rejected, R_rule_rejected)
        L_total  = L_pref + λ · L_align

    Args:
        preference_pairs: list of dicts, each with keys
            'prompt', 'chosen', 'rejected', 'correct_answer',
            'chosen_reward' (dict with 'r_normalized'),
            'rejected_reward' (dict with 'r_normalized'),
            'vulnerability_type'
        num_epochs:        passes over the preference pairs
        lr:                learning rate (overrides stored optimizer if set)
        margin:            minimum desired score gap (Bradley-Terry margin)
        lambda_align:      weight on the rule-alignment MSE term
        max_pairs_per_epoch: cap to avoid excessive GPU time
        log_every:         print interval

    Returns:
        dict with training statistics
    """
    trainer = self.base_trainer
    reward_head = trainer.reward_head
    device = trainer.device

    if not hasattr(trainer, 'reward_head') or reward_head is None:
        print("No learnable reward head — skipping fine-tuning")
        return {'skipped': True}

    if not preference_pairs:
        print("No preference pairs — skipping reward head fine-tuning")
        return {'num_pairs': 0}

    # Use a dedicated optimizer so the LR is independent of Stage 1
    rh_optimizer = torch.optim.Adam(reward_head.parameters(), lr=lr)

    reward_head.train()
    # Freeze policy during reward head updates
    was_training = trainer.model.training
    trainer.model.eval()

    stats = {
        'pref_losses': [],
        'align_losses': [],
        'total_losses': [],
        'accuracy': [],          # fraction where r_chosen > r_rejected
        'mean_margin': [],       # mean (r_chosen - r_rejected)
        'per_vulnerability': {},
    }

    print(f"\n{'='*60}")
    print(f"  REWARD HEAD FINE-TUNING  ({len(preference_pairs)} pairs, {num_epochs} epochs)")
    print(f"{'='*60}")

    for epoch in range(num_epochs):
        epoch_pref_loss = 0.0
        epoch_align_loss = 0.0
        epoch_total_loss = 0.0
        epoch_correct = 0
        epoch_margins = []
        n_updates = 0

        # Shuffle pairs each epoch
        indices = np.random.permutation(len(preference_pairs))[:max_pairs_per_epoch]

        for step, idx in enumerate(indices):
            pair = preference_pairs[idx]

            # ── Extract hidden states (no grad through the LM backbone) ──
            h_chosen = self._get_hidden_state_for_response(
                pair['prompt'], pair['chosen']
            ).detach().to(device, dtype=torch.float32)

            h_rejected = self._get_hidden_state_for_response(
                pair['prompt'], pair['rejected']
            ).detach().to(device, dtype=torch.float32)

            # ── Reward head forward ──
            r_chosen = reward_head(h_chosen).squeeze()       # scalar
            r_rejected = reward_head(h_rejected).squeeze()   # scalar

            # ── Bradley-Terry preference loss with margin ──
            # We want r_chosen - r_rejected > margin
            pref_loss = -F.logsigmoid(r_chosen - r_rejected - margin)

            # ── Rule-alignment loss ──
            # Anchor predictions to the composite rule-based reward
            rule_chosen = torch.tensor(
                pair['chosen_reward']['r_normalized'],
                device=device, dtype=torch.float32
            )
            rule_rejected = torch.tensor(
                pair['rejected_reward']['r_normalized'],
                device=device, dtype=torch.float32
            )
            align_loss = (
                F.mse_loss(r_chosen, rule_chosen) +
                F.mse_loss(r_rejected, rule_rejected)
            )

            # ── Combined loss ──
            loss = pref_loss + lambda_align * align_loss

            rh_optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(reward_head.parameters(), max_norm=1.0)
            rh_optimizer.step()

            # ── Bookkeeping ──
            with torch.no_grad():
                current_margin = (r_chosen - r_rejected).item()
                is_correct = current_margin > 0

            epoch_pref_loss += pref_loss.item()
            epoch_align_loss += align_loss.item()
            epoch_total_loss += loss.item()
            epoch_correct += int(is_correct)
            epoch_margins.append(current_margin)
            n_updates += 1

            # Track per-vulnerability-type stats
            vtype = pair.get('vulnerability_type', 'unknown')
            if vtype not in stats['per_vulnerability']:
                stats['per_vulnerability'][vtype] = {'correct': 0, 'total': 0, 'margins': []}
            stats['per_vulnerability'][vtype]['total'] += 1
            stats['per_vulnerability'][vtype]['correct'] += int(is_correct)
            stats['per_vulnerability'][vtype]['margins'].append(current_margin)

            if (step + 1) % log_every == 0:
                print(
                    f"  [Epoch {epoch+1}] Step {step+1}/{len(indices)}: "
                    f"pref={pref_loss.item():.4f}  align={align_loss.item():.4f}  "
                    f"margin={current_margin:.3f}  correct={is_correct}"
                )

        # ── Epoch summary ──
        if n_updates > 0:
            acc = epoch_correct / n_updates
            avg_margin = np.mean(epoch_margins)
            stats['pref_losses'].append(epoch_pref_loss / n_updates)
            stats['align_losses'].append(epoch_align_loss / n_updates)
            stats['total_losses'].append(epoch_total_loss / n_updates)
            stats['accuracy'].append(acc)
            stats['mean_margin'].append(avg_margin)

            print(
                f"\n  Epoch {epoch+1}/{num_epochs}: "
                f"loss={epoch_total_loss/n_updates:.4f}  "
                f"pref_acc={acc:.2%}  "
                f"avg_margin={avg_margin:.3f}  "
                f"pairs={n_updates}"
            )

    # ── Per-vulnerability summary ──
    print(f"\n  Per-vulnerability breakdown:")
    for vtype, vstats in stats['per_vulnerability'].items():
        vacc = vstats['correct'] / max(vstats['total'], 1)
        vmarg = np.mean(vstats['margins']) if vstats['margins'] else 0
        print(f"    {vtype:25s}  acc={vacc:.2%}  margin={vmarg:.3f}  n={vstats['total']}")

    # Restore model state
    if was_training:
        trainer.model.train()

    # Copy updated optimizer state back so checkpointing picks it up
    trainer.reward_optimizer = rh_optimizer

    print(f"\n Reward head fine-tuning complete")
    return stats


# ──────────────────────────────────────────────────────────
#  2.  Updated adversarial cycle — replaces
#      run_adversarial_training_cycle_grpo on AdversarialTrainer
# ──────────────────────────────────────────────────────────

def run_adversarial_training_cycle_with_reward_refinement(
    self,
    prompts,
    correct_answers,
    num_cycles=3,
    rh_epochs=3,
    rh_margin=0.5,
    rh_lambda_align=0.3,
):
    """
    Full Stage 2 cycle:
      1. Mine adversarial examples
      2. Build preference pairs
      3. Fine-tune the REWARD HEAD on those pairs   <-- the missing piece
      4. Update the POLICY via adversarial GRPO
      5. Validate
    """
    for cycle in range(num_cycles):
        print(f"\n{'='*60}")
        print(f"  ADVERSARIAL CYCLE {cycle + 1}/{num_cycles}")
        print(f"{'='*60}")

        # ── Step 1: Mine adversarial examples ──
        adversarial_examples = self.generate_adversarial_examples(
            prompts=prompts,
            correct_answers=correct_answers,
            target_reward=0.5,
        )

        if not adversarial_examples:
            print("No adversarial examples found — model is robust")
            break

        # ── Step 2: Build preference pairs ──
        preference_pairs = self.create_preference_pairs(adversarial_examples)

        if not preference_pairs:
            print("No valid preference pairs — skipping cycle")
            continue

        # ── Step 3: Fine-tune REWARD HEAD ──
        rh_stats = self.fine_tune_reward_head(
            preference_pairs,
            num_epochs=rh_epochs,
            margin=rh_margin,
            lambda_align=rh_lambda_align,
        )

        # ── Step 4: Generate clean responses (with updated reward head) ──
        clean_responses = []
        for adv in adversarial_examples:
            clean_resp = self._generate_clean_response(
                adv['prompt'], adv['correct_answer']
            )
            # Re-score with the updated reward head
            clean_reward = self.reward_function.compute_total_reward(
                clean_resp, adv['correct_answer']
            )
            # Add learnable reward component
            h = self._get_hidden_state_for_response(adv['prompt'], clean_resp)
            with torch.no_grad():
                learnable_r = self.base_trainer.reward_head(
                    h.to(self.base_trainer.device, dtype=torch.float32)
                ).item()
            clean_reward['r_learnable'] = learnable_r
            clean_reward['r_normalized'] = (
                0.7 * clean_reward['r_normalized'] + 0.3 * learnable_r
            )

            clean_responses.append({
                'prompt': adv['prompt'],
                'response': clean_resp,
                'reward_info': clean_reward,
                'correct_answer': adv['correct_answer'],
            })

        # Also re-score adversarial examples with updated reward head
        for adv in adversarial_examples:
            h = self._get_hidden_state_for_response(adv['prompt'], adv['response'])
            with torch.no_grad():
                learnable_r = self.base_trainer.reward_head(
                    h.to(self.base_trainer.device, dtype=torch.float32)
                ).item()
            adv['reward_info']['r_learnable'] = learnable_r
            adv['reward_info']['r_normalized'] = (
                0.7 * adv['reward_info']['r_normalized'] + 0.3 * learnable_r
            )

        # ── Step 5: Update POLICY via adversarial GRPO ──
        update_metrics = self.update_policy_with_adversarial_grpo(
            adversarial_examples, clean_responses
        )

        # ── Step 6: Validate ──
        validation = self.validate_policy_improvement(
            prompts[:10], correct_answers[:10]
        )

        print(f"\n  Cycle {cycle+1} summary:")
        print(f"    Adversarial examples:  {len(adversarial_examples)}")
        print(f"    Preference pairs:      {len(preference_pairs)}")
        print(f"    RH pref accuracy:      {rh_stats.get('accuracy', [0])[-1]:.2%}")
        print(f"    Policy loss:           {update_metrics['total_loss']:.4f}")
        print(f"    Adversarial rate:      {validation['adversarial_rate']:.2%}")


# ──────────────────────────────────────────────────────────
#  3.  Patching instructions
# ──────────────────────────────────────────────────────────

def patch_adversarial_trainer(AdversarialTrainerClass):
    """
    Call this once after defining AdversarialTrainer to attach
    the new methods:

        patch_adversarial_trainer(AdversarialTrainer)
    """
    AdversarialTrainerClass.fine_tune_reward_head = fine_tune_reward_head
    AdversarialTrainerClass.run_adversarial_training_cycle_with_reward_refinement = (
        run_adversarial_training_cycle_with_reward_refinement
    )
    print(" AdversarialTrainer patched with reward head fine-tuning")


# ──────────────────────────────────────────────────────────
#  4.  Updated Stage 2 pipeline run loop
# ──────────────────────────────────────────────────────────

def stage2_run_with_reward_refinement(
    pipeline,
    train_data_path,
    test_data_path,
    num_cycles=3,
    max_eval_examples=20,
    rh_epochs=3,
    rh_margin=0.5,
    rh_lambda_align=0.3,
):
    """
    Drop-in replacement for AdversarialTrainingPipeline.run()
    that includes reward head fine-tuning in each cycle.

    Usage:
        stage2_results = stage2_run_with_reward_refinement(
            pipeline=stage2_pipeline,
            train_data_path="...",
            test_data_path="...",
            num_cycles=3,
        )
    """
    import time

    print("\n" + "=" * 60)
    print("RUNNING STAGE 2: ADVERSARIAL TRAINING (WITH REWARD REFINEMENT)")
    print("=" * 60)

    start_time = time.time()

    # ── Pre-adversarial eval ──
    print("\n[STAGE 2.0] Pre-Adversarial Evaluation...")
    pre_metrics = pipeline.trainer.evaluate_model(
        test_data_path=test_data_path,
        max_examples=max_eval_examples,
    )
    pipeline.metrics_tracker.record_epoch(999, pre_metrics)

    # ── Load training data ──
    data_processor = pipeline.trainer.data_processor
    train_dataset = data_processor.load_medqa_data(train_data_path)
    train_prompts = [item['prompt'] for item in train_dataset]
    correct_answers = [item['correct_answer'] for item in train_dataset]

    # ── Adversarial cycles with reward refinement ──
    pipeline.adversarial_trainer.run_adversarial_training_cycle_with_reward_refinement(
        prompts=train_prompts,
        correct_answers=correct_answers,
        num_cycles=num_cycles,
        rh_epochs=rh_epochs,
        rh_margin=rh_margin,
        rh_lambda_align=rh_lambda_align,
    )

    # ── Post-adversarial eval ──
    print("\n[STAGE 2.2] Post-Adversarial Evaluation...")
    post_metrics = pipeline.trainer.evaluate_model(
        test_data_path=test_data_path,
        max_examples=max_eval_examples,
    )
    pipeline.metrics_tracker.record_epoch(1000, post_metrics)

    # ── Comparison ──
    improvement = {
        'accuracy': post_metrics['accuracy'] - pre_metrics['accuracy'],
        'factual_reward': post_metrics['avg_factual_reward'] - pre_metrics['avg_factual_reward'],
        'total_reward': post_metrics['avg_total_reward'] - pre_metrics['avg_total_reward'],
    }

    print(f"\n{'='*60}")
    print("STAGE 2 RESULTS")
    print(f"{'='*60}")
    print(f"  Accuracy:       {pre_metrics['accuracy']:.2%} → {post_metrics['accuracy']:.2%}  ({improvement['accuracy']:+.2%})")
    print(f"  Factual reward: {pre_metrics['avg_factual_reward']:.3f} → {post_metrics['avg_factual_reward']:.3f}  ({improvement['factual_reward']:+.3f})")
    print(f"  Total reward:   {pre_metrics['avg_total_reward']:.3f} → {post_metrics['avg_total_reward']:.3f}  ({improvement['total_reward']:+.3f})")
    print(f"  Time: {time.time() - start_time:.1f}s")

    # ── Save checkpoint ──
    checkpoint_path = pipeline.trainer.save_checkpoint(
        pipeline.output_dir, "stage2_checkpoint"
    )

    return {
        'pre_metrics': pre_metrics,
        'post_metrics': post_metrics,
        'improvement': improvement,
        'checkpoint_path': checkpoint_path,
    }

In [ ]:
class AdversarialTrainer:
    def __init__(self, base_trainer, adversarial_config=None):
        self.base_trainer = base_trainer
        self.model = base_trainer.model
        self.tokenizer = base_trainer.tokenizer
        self.reward_function = base_trainer.reward_function
        self.device = base_trainer.device
        self.kl_coef = getattr(adversarial_config, 'kl_coef', 0.1)
        
        config = adversarial_config or {}
        self.adversarial_temperature = config.get('temperature', 1.2)
        self.max_adversarial_examples = config.get('max_examples', 50)
        self.preference_margin = config.get('preference_margin', 0.5)
        self.validation_threshold = config.get('validation_threshold', 0.7)
        
        self.adversarial_examples_buffer = []
        self.preference_pairs_buffer = []
        print("Adversarial Trainer initialized")
    
    def generate_adversarial_examples(self, prompts, correct_answers=None, target_reward=0.5):
        adversarial_examples = []
        print(f"Generating adversarial examples from {len(prompts)} prompts...")
    
        generation_strategies = [
            {'temperature': 0.7, 'top_p': 0.9, 'do_sample': True},   # Balanced
            {'temperature': 0.9, 'top_p': 0.95, 'do_sample': True},  # Slightly diverse
            {'temperature': 0.5, 'top_p': 0.85, 'do_sample': True},  # Conservative
        ]
    
        shown = 0
    
        for strategy_idx, strategy in enumerate(generation_strategies):
            print(f"  Strategy {strategy_idx + 1}: temp={strategy['temperature']}")
    
            for i, prompt in enumerate(prompts[:self.max_adversarial_examples]):
                try:
                    response = self._generate_with_strategy(prompt, strategy)
                    correct_answer = correct_answers[i] if correct_answers else "A"
                    reward_info = self.reward_function.compute_total_reward(response, correct_answer)
    
                    if self._is_adversarial_example(response, reward_info, target_reward):
                        adv = {
                            'prompt': prompt,
                            'response': response,
                            'reward_info': reward_info,
                            'correct_answer': correct_answer,
                            'strategy':strategy_idx,
                            'vulnerability_type': self._classify_vulnerability(reward_info)
                        }
                        adversarial_examples.append(adv)
    
                        # Show first 1–2 adversarial examples for inspection
                        if shown < 2:
                            print("\n=== Adversarial Example ===")
                            print(f"Prompt:\n{prompt}\n")
                            print(f"Response:\n{response}\n")
                            print(f"Reward info: {reward_info}")
                            print(f"Vulnerability: {adv['vulnerability_type']}")
                            print("===========================\n")
                            shown += 1
    
                except Exception as e:
                    print(f"Error generating adversarial example: {e}")
                    continue
    
        print(f"Found {len(adversarial_examples)} adversarial examples")
        self.adversarial_examples_buffer.extend(adversarial_examples)
        return adversarial_examples


    def _generate_with_strategy(self, prompt, strategy):
        device = self.model.device
    
        # Tokenize with same settings as eval
        inputs = self.tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=512
        ).to(device)
    
        # Save and switch model state
        was_training = self.model.training
        self.model.eval()
    
        try:
            with torch.no_grad():
                with torch.amp.autocast('cuda', enabled=False):
                    outputs = self.model.generate(
                        **inputs,
                        max_new_tokens=600,
                        temperature=strategy.get("temperature", 0.7),
                        top_p=strategy.get("top_p", 0.9),
                        do_sample=strategy.get("do_sample", True),
                        pad_token_id=self.tokenizer.pad_token_id or self.tokenizer.eos_token_id,
                        eos_token_id=self.tokenizer.eos_token_id,
                        no_repeat_ngram_size=3
                    )
    
            # Decode only the generated continuation
            response = self.tokenizer.decode(
                outputs[0][inputs["input_ids"].shape[1]:],
                skip_special_tokens=True
            ).strip()
    
        finally:
            # Restore model state
            if was_training:
                self.model.train()
    
        return response

    
    def _generate_clean_response(self, prompt, correct_answer):
        """Generate a high-quality, non-adversarial response."""
                
        inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=768)
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=500,
                temperature=0.7,
                top_p=0.9,
                do_sample=True,
                pad_token_id=self.tokenizer.pad_token_id,
                eos_token_id=self.tokenizer.eos_token_id
            )
        
        response = self.tokenizer.decode(
            outputs[0][inputs['input_ids'].shape[1]:],
            skip_special_tokens=True
        ).strip()
        
        return response
    
    def _is_adversarial_example(self, response, reward_info, target_reward):
        total_reward = reward_info.get('r_total', 0)
        
        if total_reward < target_reward:
            return False
        
        has_issues = (
            reward_info.get('p_answerLeak', 0) > 0.05 or
            reward_info.get('p_preamble', 0) > 0.05 or 
            reward_info.get('r_grounded', 1.0) < 0.4 or
            not self.reward_function.validate_format(response)
        )
        return has_issues
    
    def _classify_vulnerability(self, reward_info):
        """Classify the type of vulnerability being exploited."""
        if reward_info.get('p_answerLeak', 0) > 0.05:
            return "answer_leaking"
        elif reward_info.get('p_preamble', 0) > 0.05:
            return "structural_gaming"
        elif reward_info.get('r_grounded', 1.0) < 0.4:
            return "factual_exploitation"
        else:
            return "format_gaming"
    
    def create_preference_pairs(self, adversarial_examples):
        preference_pairs = []
        
        for adv_example in adversarial_examples:
            clean_response = self._generate_clean_response(
                adv_example['prompt'],
                adv_example['correct_answer']
            )
            
            clean_reward_info = self.reward_function.compute_total_reward(
                clean_response, adv_example['correct_answer']
            )
            
            clean_quality = self._assess_response_quality(clean_response, clean_reward_info, adv_example['correct_answer'])
            adv_quality = self._assess_response_quality(adv_example['response'], adv_example['reward_info'], adv_example['correct_answer'])
            
            if clean_quality > adv_quality:
                    preference_pairs.append({
                        'prompt': adv_example['prompt'],
                        'chosen': clean_response,
                        'rejected': adv_example['response'],
                        'correct_answer': adv_example['correct_answer'],
                        'chosen_reward': clean_reward_info,
                        'rejected_reward': adv_example['reward_info'],
                        'vulnerability_type': adv_example['vulnerability_type']
                    })
        
        print(f"Created {len(preference_pairs)} preference pairs")
        self.preference_pairs_buffer.extend(preference_pairs)
        
        return preference_pairs
    
    
    def _ensure_proper_format(self, response, correct_answer):
        """Ensure response has proper <think> and <answer> tags."""
        # Add think tags if missing
        if '<think>' not in response.lower():
            reasoning_part = response.split('<answer>')[0].strip()
            answer_part = response.split('<answer>')[1] if '<answer>' in response else ''
            response = f"<think>{reasoning_part}</think>"
            if answer_part:
                response += f"<answer>{answer_part}"
        
        # Add answer tags if missing
        if '<answer>' not in response.lower():
            response += f"\n<answer>{correct_answer}</answer>"
        
        return response
    
    def update_policy_with_preferences(self, adversarial_examples, clean_responses):
        """
        Use GRPO-style update: treat adversarial + clean as a group,
        compute relative advantages, upweight clean responses.
        """
        self.base_trainer.model.train()
        self.base_trainer.policy_optimizer.zero_grad()

        total_loss = 0
        num_updates = 0

        # Group by prompt
        prompt_groups = {}
        for adv in adversarial_examples:
            prompt = adv['prompt']
            if prompt not in prompt_groups:
                prompt_groups[prompt] = {'adversarial': [], 'clean': []}
            prompt_groups[prompt]['adversarial'].append(adv)

        for clean in clean_responses:
            prompt = clean['prompt']
            if prompt in prompt_groups:
                prompt_groups[prompt]['clean'].append(clean)

        for prompt, group in prompt_groups.items():
            if not group['clean']:
                continue

            # Collect all responses for this prompt
            all_responses = []
            all_rewards = []

            for adv in group['adversarial']:
                all_responses.append(adv['response'])
                # Penalize adversarial: use negative or reduced reward
                all_rewards.append(-abs(adv['reward_info']['r_total']))

            for clean in group['clean']:
                all_responses.append(clean['response'])
                all_rewards.append(clean['reward_info']['r_total'])

            if len(all_responses) < 2:
                continue

            # Compute GRPO-style group advantages
            rewards_tensor = torch.tensor(all_rewards, device=self.device)
            advantages = self._compute_group_advantages(rewards_tensor)

            # Policy gradient with advantages
            for response, advantage in zip(all_responses, advantages):
                log_probs = self._get_response_logprobs(prompt, response)

                # GRPO loss: -advantage * log_prob
                loss = -advantage * log_probs.sum()
                loss = loss / len(prompt_groups)  # Normalize
                loss.backward()
                total_loss += loss.item()

            num_updates += 1

        torch.nn.utils.clip_grad_norm_(self.base_trainer.model.parameters(), 1.0)
        self.base_trainer.policy_optimizer.step()

        return {'loss': total_loss / max(num_updates, 1)}


    def update_policy_with_adversarial_grpo(self,adversarial_examples, clean_responses, num_steps = 1):
        """
        Update policy using GRPO with adversarial examples as negative samples
        and clean responses as positive samples.
        """
        self.model.train()
        
        total_loss = 0.0
        total_policy_loss = 0.0
        total_kl_loss = 0.0
        num_pairs = 0
        
        for adv, clean in zip(adversarial_examples, clean_responses):
            if adv['prompt'] != clean['prompt']:
                continue
                
            prompt = adv['prompt']
            
            adv_reward = adv['reward_info'].get('r_total', 0.0)
            clean_reward = clean['reward_info'].get('r_total', 0.0)
            
            # Skip if clean isn't meaningfully better
            if clean_reward <= adv_reward + 0.2:
                continue
            
            # FIX: Use reward difference as margin, not baseline subtraction
            reward_margin = clean_reward - adv_reward  # Always positive here
            
            for _ in range(num_steps):
                self.base_trainer.policy_optimizer.zero_grad()
                
                adv_log_prob = self._compute_log_prob(prompt, adv['response'])
                clean_log_prob = self._compute_log_prob(prompt, clean['response'])
                
                # FIX: Contrastive loss - increase clean, decrease adversarial
                # We want: clean_log_prob > adv_log_prob by at least some margin
                policy_loss = -reward_margin * (clean_log_prob - adv_log_prob)
                
                # KL regularization
                kl_loss = torch.tensor(0.0, device=self.device)
                if hasattr(self.base_trainer, 'reference_model') and self.base_trainer.reference_model is not None:
                    with torch.no_grad():
                        ref_clean_log_prob = self._compute_log_prob(prompt, clean['response'], use_reference=True)
                        ref_adv_log_prob = self._compute_log_prob(prompt, adv['response'], use_reference=True)
                    
                    kl_clean = clean_log_prob - ref_clean_log_prob
                    kl_adv = adv_log_prob - ref_adv_log_prob
                    kl_loss = self.kl_coef * (kl_clean.abs() + kl_adv.abs())  # Penalize deviation
                
                loss = policy_loss + kl_loss
                loss.backward()
                
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                self.base_trainer.policy_optimizer.step()
                
                total_loss += loss.item()
                total_policy_loss += policy_loss.item()
                total_kl_loss += kl_loss.item() if isinstance(kl_loss, torch.Tensor) else kl_loss
            
            num_pairs += 1
        
        num_updates = max(num_pairs * num_steps, 1)
        
        return {
            'total_loss': total_loss / num_updates,
            'policy_loss': total_policy_loss / num_updates,
            'kl_loss': total_kl_loss / num_updates,
            'num_pairs': num_pairs,
            'num_updates': num_updates
        }
    
    def _compute_log_prob(self, prompt: str, response: str, use_reference: bool = False) -> torch.Tensor:
        """Compute log probability of response given prompt."""
        model = self.base_trainer.reference_model if use_reference else self.model
        
        full_text = prompt + response
        prompt_ids = self.tokenizer.encode(prompt, return_tensors='pt').to(self.device)
        full_ids = self.tokenizer.encode(full_text, return_tensors='pt').to(self.device)
        
        prompt_len = prompt_ids.shape[1]
        
        with torch.set_grad_enabled(not use_reference):
            outputs = model(full_ids)
            logits = outputs.logits
        
        log_probs = torch.log_softmax(logits[:, :-1, :], dim=-1)
        target_ids = full_ids[:, 1:]
        
        token_log_probs = log_probs.gather(-1, target_ids.unsqueeze(-1)).squeeze(-1)
        response_log_prob = token_log_probs[:, prompt_len-1:].sum()
        
        return response_log_prob

    def _compute_group_advantages(self, rewards):
        """
        GRPO: normalize rewards within group
        """
        mean = rewards.mean()
        std = rewards.std() + 1e-8
        advantages = (rewards - mean) / std
        return advantages

    def _get_response_logprobs(self, prompt, response):
        """Compute log probabilities for a given response"""
        full_text = prompt + " " + response
        inputs = self.tokenizer(full_text, return_tensors="pt", truncation=True)
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        
        # Forward pass through model
        outputs = self.base_trainer.model(**inputs)
        logits = outputs.logits
        
        # Get log probs for the response tokens
        prompt_len = len(self.tokenizer(prompt)['input_ids'])
        response_logits = logits[0, prompt_len-1:-1, :]
        response_tokens = inputs['input_ids'][0, prompt_len:]
        
        log_probs = F.log_softmax(response_logits, dim=-1)
        token_log_probs = log_probs.gather(1, response_tokens.unsqueeze(-1)).squeeze(-1)
        
        return token_log_probs

    def analyze_response_quality(self, response, reward_info, correct_answer):
        violations = {
            'correctness_violations': 0,
            'answer_leaking_violations': 0,
            'format_violations': 0,
            'factual_violations': 0,
            'bad_ood_high_rewards': 0
        }

        # Goal 1: Improved Correctness
        predicted_answer = self.reward_function.extract_answer_choice(response)
        if not predicted_answer or predicted_answer.upper() != correct_answer.upper():
            violations['correctness_violations'] = 1

        # Goal 2: Less Answer Leaking
        if reward_info.get('p_answerLeak', 0) > 0.1:
            violations['answer_leaking_violations'] = 1

        # Goal 3: Less Format Violations
        if reward_info.get('p_preamble', 0) > 0.1 or not self.reward_function.validate_format(response):
            violations['format_violations'] = 1

        # Goal 4: Less Factual Errors
        if reward_info.get('r_grounded', 1.0) < 0.3:
            violations['factual_violations'] = 1

        # Goal 5: Less Rewards to Bad OOD Reasoning
        # High total reward despite violations = bad OOD getting undeserved high reward
        has_violations = sum(violations.values()) > 0
        high_reward = reward_info.get('r_total', 0) > 0.5
        if has_violations and high_reward:
            violations['bad_ood_high_rewards'] = 1

        return violations

    def _assess_response_quality(self, response, reward_info, correct_answer=None):
        """
        Assess overall response quality using existing analysis method.
        
        Returns:
            Quality score between 0 and 1 (higher = better quality)
        """
        # Use existing violation analysis
        violations = self.analyze_response_quality(response, reward_info, correct_answer or "A")
        
        # Convert violations to quality score
        total_violations = sum(violations.values())
        max_violations = len(violations)  # 5 possible violation types
        
        # Quality = 1 - (violation_ratio)
        quality_score = 1.0 - (total_violations / max_violations)
        
        return quality_score
    
    def _get_hidden_state_for_response(self, prompt, response):
        """Get hidden state for a specific response."""
        full_text = prompt + " " + response
        inputs = self.tokenizer(full_text, return_tensors="pt", truncation=True, max_length=768)
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = self.model(**inputs, output_hidden_states=True)
            hidden_state = outputs.hidden_states[-1][0].mean(dim=0)  # Average over sequence
        
        return hidden_state
    
    def validate_policy_improvement(self, test_prompts, test_answers):
        """Validate that POLICY generates fewer adversarial examples"""
    
        print("Measuring policy improvement...")
        
        # Generate responses with current policy
        responses_and_rewards = []
        for prompt, answer in zip(test_prompts[:20], test_answers[:20]):
            response = self.base_trainer.generate_response(prompt)
            reward_info = self.reward_function.compute_total_reward(response, answer)
            responses_and_rewards.append((response, reward_info))
        
        # Count how many are adversarial
        adversarial_count = sum(
            1 for resp, r_info in responses_and_rewards
            if self._is_adversarial_example(resp, r_info, target_reward=0.5)
        )
        
        # Lower is better - policy should generate fewer adversarial examples
        adversarial_rate = adversarial_count / len(responses_and_rewards)
        improvement_score = 1.0 - adversarial_rate
        
        print(f"  Adversarial generation rate: {adversarial_rate:.2%}")
        print(f"  Improvement score: {improvement_score:.2%}")
        
        return {
            'adversarial_rate': adversarial_rate,
            'improvement_score': improvement_score,
            'total_tested': len(responses_and_rewards)
        }
    
    def run_adversarial_training_cycle_grpo(self, prompts, correct_answers, num_cycles=3):
        for cycle in range(num_cycles):
            print(f"\n=== Adversarial Cycle {cycle + 1}/{num_cycles} ===")

            # Step 1: Generate adversarial examples
            adversarial_examples = self.generate_adversarial_examples(
                prompts=prompts,
                correct_answers=correct_answers,
                target_reward=0.5
            )

            if not adversarial_examples:
                print("No adversarial examples found - model is robust")
                break

            # Step 2: Generate clean responses for same prompts
            clean_responses = []
            for adv in adversarial_examples:
                clean_resp = self._generate_clean_response(
                    adv['prompt'],
                    adv['correct_answer']
                )
                clean_reward = self.reward_function.compute_total_reward(
                    clean_resp,
                    adv['correct_answer']
                )
                clean_responses.append({
                    'prompt': adv['prompt'],
                    'response': clean_resp,
                    'reward_info': clean_reward,
                    'correct_answer': adv['correct_answer']
                })

            # Step 3: GRPO update (not DPO)
            update_metrics = self.update_policy_with_adversarial_grpo(
                adversarial_examples,
                clean_responses
            )

            # Step 4: Validate
            validation = self.validate_policy_improvement(
                prompts[:10],
                correct_answers[:10]
            )

            print(f"  Loss: {update_metrics['total_loss']:.4f}")
            print(f"  Adversarial rate: {validation['adversarial_rate']:.2%}")
    
    def clear_buffers(self):
        """Clear adversarial example and preference pair buffers."""
        self.adversarial_examples_buffer.clear()
        self.preference_pairs_buffer.clear()
        print("Adversarial training buffers cleared")

In [ ]:
patch_adversarial_trainer(AdversarialTrainer)

In [ ]:
class AdversarialTrainingPipeline:
    """
    Stage 2: Adversarial training on top of Stage 1 model
    """
    
    def __init__(self,
                 stage1_checkpoint_path,
                 base_model_path,
                 reward_config=None,
                 adversarial_config=None,
                 output_dir="./stage2_adversarial_training",
                 trainer=None):
        """
        Initialize Stage 2 pipeline
        """
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)
        self.adversarial_config = adversarial_config or {}
        
        print("="*60)
        print("STAGE 2: ADVERSARIAL TRAINING PIPELINE")
        print("="*60)
        
        # Validate Stage 1 checkpoint
        if not os.path.exists(stage1_checkpoint_path):
            raise FileNotFoundError(f"Stage 1 checkpoint not found: {stage1_checkpoint_path}")
        
        stage1_metadata_path = os.path.join(stage1_checkpoint_path, "stage1_metadata.json")
        if not os.path.exists(stage1_metadata_path):
            raise FileNotFoundError(f"Stage 1 metadata not found. Invalid checkpoint: {stage1_checkpoint_path}")
        
        # Load Stage 1 metadata
        with open(stage1_metadata_path, 'r') as f:
            self.stage1_metadata = json.load(f)
        
        if not self.stage1_metadata.get('ready_for_stage2', False):
            raise ValueError("Stage 1 checkpoint is not ready for Stage 2 training")
        
        #  Print Stage 1 summary
        print(f"Stage 1 checkpoint validated")
        print(f"\n Stage 1 Results:")
        print(f"   Initial accuracy: {self.stage1_metadata['initial_metrics']['accuracy']:.2%}")
        print(f"   Final accuracy:   {self.stage1_metadata['final_metrics']['accuracy']:.2%}")
        print(f"   Improvement:      {self.stage1_metadata['improvement']['accuracy']:+.2%}")
        
        # Load reward config from Stage 1 if not provided
        if reward_config is None:
            reward_config = self.stage1_metadata.get('reward_config', {})
            print(f"   Using reward config from Stage 1")
        
        # Load trained model from Stage 1
        print(f"\n Loading model from Stage 1...")
        if trainer is not None:
            self.trainer = trainer
            print("Using in-memory trainer from Stage 1")
        else:
            self.trainer = PolicyTrainer.load_from_checkpoint(
                checkpoint_path=stage1_checkpoint_path,
                reward_config=reward_config,
                base_model_path=base_model_path
            )
        
        # Initialize adversarial trainer
        print(f"\n Initializing adversarial trainer...")
        self.adversarial_trainer = self.trainer.adversarial_trainer
        
        # Update adversarial config if provided
        if adversarial_config:
            config = adversarial_config
            self.adversarial_trainer.adversarial_temperature = config.get('temperature', 1.2)
            self.adversarial_trainer.max_adversarial_examples = config.get('max_examples', 50)
            self.adversarial_trainer.preference_margin = config.get('preference_margin', 0.5)
            print(f"   Temperature: {self.adversarial_trainer.adversarial_temperature}")
            print(f"   Max examples per strategy: {self.adversarial_trainer.max_adversarial_examples}")
            print(f"   Preference margin: {self.adversarial_trainer.preference_margin}")
        
        #  Initialize dedicated MetricsTracker for Stage 2
        self.metrics_tracker = MetricsTracker()

        #  Reinitialize fact verification system for Stage 2
        if not hasattr(self.trainer.reward_function, 'fact_verification_system') or \
           self.trainer.reward_function.fact_verification_system is None:
            print("Reinitializing fact verification system for adversarial training...")
            self.trainer.reward_function.fact_verification_system  = AtomicFactVerificationSystem(
                agreement_threshold=0.5,
                verification_model_id="Qwen/Qwen2.5-14B-Instruct",
                device_map={"": 0},
                torch_dtype="auto",
                use_4bit=True,
                primekg_path="../Datasets/kg.csv")
        
        self.adversarial_config = adversarial_config
        
        print(f"\n Stage 2 pipeline initialized")
        print(f"   Output directory: {output_dir}")
    
    def run(self,
            train_data_path,
            test_data_path,
            num_cycles=3,
            max_eval_examples=20):
        """
        Run complete Stage 2 adversarial training with full metric tracking
        """
        print("\n" + "="*60)
        print("RUNNING STAGE 2: ADVERSARIAL TRAINING")
        print("="*60)
        
        start_time = time.time()
        
        # ========================================
        # Step 1: Pre-Adversarial Evaluation
        # ========================================
        print("\n[STAGE 2.0] Pre-Adversarial Evaluation...")
        print("-" * 60)
        pre_metrics = self.trainer.evaluate_model(
            test_data_path=test_data_path,
            max_examples=max_eval_examples
        )
        
        #  Record pre-adversarial metrics as epoch 999
        self.metrics_tracker.record_epoch(999, pre_metrics)
        self.metrics_tracker.print_metrics_with_std(
            pre_metrics, 
            "PRE-ADVERSARIAL METRICS (FROM STAGE 1)"
        )
        
        # ========================================
        # Step 2: Run Adversarial Training Cycles
        # ========================================
        print("\n[STAGE 2.1] Running Adversarial Training Cycles...")
        print("="*60)
        
        adversarial_start = time.time()
        cycle_results = []
        
        # Load training data
        data_processor = MedQADataProcessor(self.trainer.tokenizer)
        train_dataset = data_processor.load_medqa_data(train_data_path)
        train_prompts  = [item['prompt'] for item in train_dataset]
        correct_answers = [item['correct_answer'] for item in train_dataset]
        
        for cycle in range(num_cycles):
            print(f"\n--- Cycle {cycle + 1}/{num_cycles} ---")
            
            # Step 1: Generate adversarial examples
            adversarial_examples = self.adversarial_trainer.generate_adversarial_examples(
                prompts=train_prompts,
                correct_answers=correct_answers,
                target_reward=self.adversarial_config['target_reward']
            )
            
            if not adversarial_examples:
                print("No adversarial examples found, skipping cycle")
                continue
            
            # Step 2: Generate clean responses
            clean_responses = []
            for adv in adversarial_examples:
                clean_resp_text = self.adversarial_trainer._generate_clean_response(
                    adv['prompt'], 
                    adv['correct_answer']
                )
                clean_reward = self.trainer.reward_function.compute_total_reward(
                    clean_resp_text, 
                    adv['correct_answer']
                )
                clean_responses.append({
                    'prompt': adv['prompt'],
                    'response': clean_resp_text,
                    'reward_info': clean_reward,
                    'correct_answer': adv['correct_answer']
                })
        
            # Step 3: Update policy
            update_metrics = self.adversarial_trainer.update_policy_with_adversarial_grpo(
                adversarial_examples,
                clean_responses
            )
            
            # Step 4: Validate
            validation_metrics = self.adversarial_trainer.validate_policy_improvement(
                train_prompts[:10], correct_answers[:10]
            )
            
            # Step 5: Record cycle results
            cycle_results.append({
                'cycle': cycle + 1,
                'num_adversarial': len(adversarial_examples),
                'policy_loss': update_metrics['total_loss'],
                'adversarial_rate': validation_metrics['adversarial_rate'],
                'robustness_score': validation_metrics.get('robustness_score', 0.0),
                'improvement_score': validation_metrics['improvement_score']
            })
            
            print(f"  Adversarial examples: {len(adversarial_examples)}")
            print(f"  Policy loss: {update_metrics['total_loss']:.4f}")
            print(f"  Adversarial rate: {validation_metrics['adversarial_rate']:.2%}")
            print(f"  Improvement: {validation_metrics['improvement_score']:.2%}")
            
            # Optional: Early stop if getting worse
            if validation_metrics['improvement_score'] < -0.1:
                print("Model degrading, stopping early")
                break
        
        # ========================================
        # Step 3: Post-Adversarial Evaluation
        # ========================================
        print("\n" + "="*60)
        print("[STAGE 2.2] Post-Adversarial Final Evaluation...")
        print("="*60)
        post_metrics = self.trainer.evaluate_model(
            test_data_path=test_data_path,
            max_examples=max_eval_examples
        )
        
        #  Record post-adversarial metrics as epoch 1000
        self.metrics_tracker.record_epoch(1000, post_metrics)
        self.metrics_tracker.print_metrics_with_std(
            post_metrics,
            "POST-ADVERSARIAL METRICS (FINAL)"
        )
        
        # ========================================
        # Print Adversarial Training Impact
        # ========================================
        print("\n" + "="*60)
        print("ADVERSARIAL TRAINING IMPACT ANALYSIS")
        print("="*60)
        
        #  Use MetricsTracker's built-in comparison method
        if 999 in self.metrics_tracker.epoch_metrics and 1000 in self.metrics_tracker.epoch_metrics:
            self.metrics_tracker.print_adversarial_impact_analysis()
        
        # Compute detailed improvement
        improvement = {
            'accuracy_change': post_metrics['accuracy'] - pre_metrics['accuracy'],
            'factual_violation_change': pre_metrics['factual_violations_rate'] - post_metrics['factual_violations_rate'],
            'answer_leaks_change': pre_metrics['answer_leaking_violations_rate'] - post_metrics['answer_leaking_violations_rate'],
            'format_violation_change': pre_metrics['format_violations_rate'] - post_metrics['format_violations_rate'],
            'factual_reward_change': post_metrics['avg_factual'] - pre_metrics['avg_factual'],
            'total_reward_change': post_metrics['avg_reward'] - pre_metrics['avg_reward'],
            'bad_ood_rewards_change': pre_metrics['bad_ood_high_rewards_rate'] - post_metrics['bad_ood_high_rewards_rate']
        }
        
        print(f"\n Detailed Changes (Pre → Post Adversarial):")
        print(f"   Accuracy:               {pre_metrics['accuracy']:.2%} → {post_metrics['accuracy']:.2%} ({improvement['accuracy_change']:+.2%})")
        print(f"   Factual Violations:     {pre_metrics['factual_violations_rate']:.2%} → {post_metrics['factual_violations_rate']:.2%} ({improvement['factual_violation_change']:+.2%})")
        print(f"   Answer Leaks:           {pre_metrics['answer_leaking_violations_rate']:.2%} → {post_metrics['answer_leaking_violations_rate']:.2%} ({improvement['answer_leaks_change']:+.2%})")
        print(f"   Factual Reward:         {pre_metrics['avg_factual']:.3f} → {post_metrics['avg_factual']:.3f} ({improvement['factual_reward_change']:+.3f})")
        print(f"   Total Reward:           {pre_metrics['avg_reward']:.3f} → {post_metrics['avg_reward']:.3f} ({improvement['total_reward_change']:+.3f})")
        
        # ========================================
        # Step 4: Save Final Checkpoint
        # ========================================
        checkpoint_path = os.path.join(self.output_dir, "stage2_checkpoint")
        print(f"\n[STAGE 2.3] Saving Final Checkpoint...")
        print("-" * 60)
        self._save_stage2_checkpoint(
            checkpoint_path, 
            pre_metrics, 
            post_metrics, 
            improvement,
            cycle_results
        )
        
        elapsed_time = time.time() - start_time
        
        # ========================================
        # Save All Metrics
        # ========================================
        metrics_path = os.path.join(self.output_dir, "stage2_metrics.json")
        self.metrics_tracker.save_metrics(metrics_path)
        print(f" All metrics saved to: {metrics_path}")
        
        # Save detailed summary
        adversarial_results = {
            'total_cycles': len(cycle_results),
            'cycle_results': cycle_results,
            'final_robustness': cycle_results[-1]['robustness_score'] if cycle_results else 0.0
        }
        
        summary = {
            'stage': 'stage2_complete',
            'num_cycles': num_cycles,
            'pre_adversarial_metrics': pre_metrics,
            'post_adversarial_metrics': post_metrics,
            'improvement': improvement,
            'adversarial_results': adversarial_results,
            'timing': {
                'adversarial_training': adversarial_time,
                'total_time': elapsed_time
            },
            'checkpoint_path': checkpoint_path,
            'stage1_metadata': self.stage1_metadata
        }
        
        summary_path = os.path.join(self.output_dir, "stage2_summary.json")
        with open(summary_path, 'w') as f:
            json.dump(summary, f, indent=2)
        
        # ========================================
        # Final Summary
        # ========================================
        print("\n" + "="*60)
        print("STAGE 2 COMPLETE")
        print("="*60)
        
        print(f"\nFinal Performance:")
        print(f"   Robustness score: {adversarial_results['final_robustness']:.2%}")
        print(f"   Accuracy change: {improvement['accuracy_change']:+.2%}")
        print(f"   Factual violations reduced: {improvement['factual_violation_change']:+.2%}")
        print(f"   Answer leaks reduced: {improvement['answer_leaks_change']:+.2%}")
        
        print(f"\n Outputs:")
        print(f"   Checkpoint: {checkpoint_path}")
        print(f"   Metrics: {metrics_path}")
        print(f"   Summary: {summary_path}")
        
        print(f"\nTRAINING COMPLETE - Model ready for deployment")
        print(f"   Load from: {checkpoint_path}")
        
        return summary
    
    def _save_stage2_checkpoint(self, checkpoint_path, pre_metrics, post_metrics, 
                                improvement, cycle_results):
        """Save Stage 2 checkpoint with all comparison metrics"""
        
        # Use PolicyTrainer's save method
        self.trainer.save_model_checkpoint(
            checkpoint_path=checkpoint_path,
            stage_name="stage2_adversarial_complete"
        )
        
        # Add Stage 2 specific metadata with full metrics
        stage2_metadata = {
            'stage': 'stage2_complete',
            'training_complete': True,
            'pre_adversarial_metrics': pre_metrics,
            'post_adversarial_metrics': post_metrics,
            'improvement': improvement,
            'cycle_results': cycle_results,
            'stage1_metadata': self.stage1_metadata
        }
        
        metadata_path = os.path.join(checkpoint_path, "stage2_metadata.json")
        with open(metadata_path, 'w') as f:
            json.dump(stage2_metadata, f, indent=2)
        
        print(f" Stage 2 checkpoint saved")
        print(f"   Location: {checkpoint_path}")

# Dual Source Fact Verification

In [ ]:
class MedicalKnowledgeGraphVerifier:
    """Knowledge Graph-based verifier"""
    
    def __init__(self):
        print("Initializing Knowledge Graph...")
        self.kg = MedicalKnowledgeGraph("primekg_dbpedia_2023.json")
        print(f" KG: {self.kg.graph.number_of_nodes()} nodes, "
              f"{self.kg.graph.number_of_edges()} edges")
    
    def verify_fact(self, fact, threshold=0.7):
        """Verify against knowledge graph"""
        return self.kg.verify_fact(fact, threshold)

# Training and Evaluation

In [ ]:
# ============================================
# STAGE 1: REWARD MODEL TRAINING
# ============================================

def get_model_name(model_url):
    """Extract clean model name from URL for directory naming"""
    name = model_url.split("/")[-1].lower()
    return name


def run_stage1(stage1_epochs, stage2_epochs, batch_size, model_url, train_data_path, test_data_path):
    """Run Stage 1: Reward model training"""
    
    print("\n" + "="*80)
    print(" "*20 + "STAGE 1: REWARD MODEL TRAINING")
    print("="*80)
    
    # Get model name for directory
    model_name = get_model_name(model_url)
    stage1_output_dir = f"./checkpoints/{model_name}/stage1_reward_training"
    
    print(f"Model: {model_name}")
    print(f"Output directory: {stage1_output_dir}")
    
    # Configuration
    reward_config = {
        'w_accuracy': 1.0,
        'w_leak': 0.3,
        'w_preamble': 0.2,
        'w_grounded': 0.5,
        'tau_veracity': 0.4,
        'k_sharpness': 8.0,
        'tau_leak': 0.7,
        'tau_preamble_words': 15,
        'agreement_threshold': 0.5,
        'verification_model': 'Qwen/Qwen2.5-14B-Instruct',
        'use_4bit_verification': True,
    }
        
    # Initialize Stage 1 pipeline
    stage1_pipeline = RewardModelTrainingPipeline(
        model_path=model_url,
        reward_config=reward_config,
        use_baseline=True,
        use_learnable_reward=True,
        group_size=8,
        beta=0.3,
        output_dir=stage1_output_dir
    )
    
    # Prepare data
    train_df = pd.read_json(train_data_path).iloc[:1000]
    trimmed_path = train_data_path.replace('.json', '_1000.json')
    train_df.to_json(trimmed_path, orient='records')
    stage1_pipeline.trainer.reward_function.fact_verification_system.training_mode = True
    
    # Run Stage 1
    stage1_results = stage1_pipeline.run(
        train_data_path=train_data_path,
        test_data_path=test_data_path,
        stage1_epochs=stage1_epochs,
        stage2_epochs=stage2_epochs,
        batch_size=batch_size,
        max_eval_examples=50
    )
    
    # Add model info to results
    stage1_results['model_name'] = model_name
    stage1_results['model_url'] = model_url
    stage1_results['pipeline'] = stage1_pipeline

    
    print("\n Stage 1 complete!")
    print(f"   Checkpoint: {stage1_results['checkpoint_path']}")
    print(f"   Accuracy: {stage1_results['final_metrics']['accuracy']:.2%}")
    
    return stage1_results


def run_stage2(stage1_checkpoint_path, model_url, trainer):    
    print("\n" + "=" * 80)
    print(" " * 15 + "STAGE 2: ADVERSARIAL TRAINING")
    print("=" * 80)
 
    model_name = get_model_name(model_url)
    stage2_output_dir = f"./checkpoints/{model_name}/stage2_adversarial_training"
 
    adversarial_config = {
        'temperature': 1.2,
        'max_examples': 50,
        'preference_margin': 0.5,
        'target_reward': 0.5,
    }
 
    stage2_pipeline = AdversarialTrainingPipeline(
        stage1_checkpoint_path=stage1_checkpoint_path,
        adversarial_config=adversarial_config,
        base_model_path=model_url,
        output_dir=stage2_output_dir,
        trainer=trainer
        #trainer=stage1_pipeline.trainer
    )
 
    stage2_results = stage2_run_with_reward_refinement(
        pipeline=stage2_pipeline,
        train_data_path="../Datasets/medqa_train_sample.json",
        test_data_path="../Datasets/medqa_test.json",
        num_cycles=3,
        max_eval_examples=20,
        # Reward head fine-tuning hyperparameters
        rh_epochs=3,          # passes over preference pairs per cycle
        rh_margin=0.5,        # Bradley-Terry margin
        rh_lambda_align=0.3,  # weight on rule-alignment MSE
    )
 
    stage2_results['model_name'] = model_name
    stage2_results['model_url'] = model_url
 
    print("\n Stage 2 complete!")
    print(f"   Checkpoint: {stage2_results['checkpoint_path']}")
 
    return stage2_results

## MetricsTracker

In [ ]:
def analyze_logs(log_file):
    """Quick analysis of logged responses"""
    
    with open(log_file, 'r') as f:
        data = [json.loads(line) for line in f]
    
    print("="*80)
    print(f"ANALYSIS OF {len(data)} RESPONSES")
    print("="*80)
    
    # Count issues
    no_facts = sum(1 for d in data if not d['has_facts_tags'])
    no_answer = sum(1 for d in data if not d['has_answer_tags'])
    format_invalid = sum(1 for d in data if not d['format_valid'])
    wrong_answer = sum(1 for d in data if d['extracted_answer'] != d['correct_answer'])
    
    print(f"\nFormat Issues:")
    print(f"  Missing <facts> tags: {no_facts}/{len(data)} ({no_facts/len(data)*100:.1f}%)")
    print(f"  Missing <answer> tags: {no_answer}/{len(data)} ({no_answer/len(data)*100:.1f}%)")
    print(f"  Format validation failed: {format_invalid}/{len(data)} ({format_invalid/len(data)*100:.1f}%)")
    
    print(f"\nAccuracy:")
    print(f"  Wrong answers: {wrong_answer}/{len(data)} ({wrong_answer/len(data)*100:.1f}%)")

In [ ]:
class MetricsTracker:
    def __init__(self):
        self.epoch_metrics = {}
        self.history = []

    def add_epoch_metrics(self, epoch, metrics):
        self.epoch_metrics.append({
            'epoch': epoch,
            'correctness_violations_rate': metrics.get('correctness_violations_rate', 0),
            'answer_leaking_violations_rate': metrics.get('answer_leaking_violations_rate', 0), 
            'format_violations_rate': metrics.get('format_violations_rate', 0),
            'factual_violations_rate': metrics.get('factual_violations_rate', 0),
            'bad_ood_high_rewards_rate': metrics.get('bad_ood_high_rewards_rate', 0),
            'avg_reward': metrics.get('avg_reward', 0),
            'avg_factual': metrics.get('avg_factual', 0)
        })

    def record_epoch(self, epoch, metrics):
        """Record metrics for an epoch"""
        self.epoch_metrics[epoch] = metrics
        self.history.append({'epoch': epoch, **metrics})

    def print_stage_summary(self, stage_name, start_epoch=0, end_epoch=None):
        """Print mean and std for a training stage"""
        if end_epoch is None:
            end_epoch = len(self.epoch_metrics)

        stage_data = self.epoch_metrics[start_epoch:end_epoch]
        if not stage_data:
            print(f"No data for {stage_name}")
            return

        print(f"\n{'='*60}")
        print(f"{stage_name.upper()} SUMMARY (Epochs {start_epoch+1}-{end_epoch})")
        print(f"{'='*60}")
        
        # Extract values for each metric
        metrics = {
            'Correctness Violations': [d['correctness_violations_rate'] for d in stage_data],
            'Answer Leaking': [d['answer_leaking_violations_rate'] for d in stage_data],
            'Format Violations': [d['format_violations_rate'] for d in stage_data], 
            'Factual Violations': [d['factual_violations_rate'] for d in stage_data],
            'Bad OOD Rewards': [d['bad_ood_high_rewards_rate'] for d in stage_data],
            'Average Reward': [d['avg_reward'] for d in stage_data],
            'Factual Reward': [d['avg_factual'] for d in stage_data]
        }
        
        # Print statistics
        for metric_name, values in metrics.items():
            mean_val = np.mean(values)
            std_val = np.std(values)
            print(f"{metric_name:20s}: {mean_val:.2f} ± {std_val:.2f}")
            
        # Calculate improvement (first vs last epoch)
        if len(stage_data) > 1:
            print(f"\n{stage_name} Improvements:")
            first_epoch = stage_data[0]
            last_epoch = stage_data[-1]
            
            # For violation rates, improvement is decrease (negative change)
            for metric in ['correctness_violations_rate', 'answer_leaking_violations_rate', 
                          'format_violations_rate', 'factual_violations_rate', 'bad_ood_high_rewards_rate']:
                change = last_epoch[metric] - first_epoch[metric]
                improvement = -change  # Negative change is improvement for violations
                print(f"  {metric.replace('_', ' ').title():25s}: {improvement:+.2f}")
            
            # For rewards, improvement is increase (positive change) 
            for metric in ['avg_reward', 'avg_factual']:
                change = last_epoch[metric] - first_epoch[metric]
                print(f"  {metric.replace('_', ' ').title():25s}: {change:+.2f}")


    def print_metrics_with_std(self, metrics, title="METRICS", include_std=True):
        """Print metrics in consistent format with optional standard deviations"""
        print(f"\n{title}")
        print("-" * len(title))
        
        # Define metric display configurations
        metric_configs = [
            ('correctness_violations_rate', 'Correctness Violations', 'correctness_violations_std'),
            ('answer_leaking_violations_rate', 'Answer Leakage Rate', 'answer_leaking_violations_std'),
            ('format_violations_rate', 'Bad Format Rate', 'format_violations_std'),
            ('factual_violations_rate', 'Factual Violations', 'factual_violations_std'),
            ('bad_ood_high_rewards_rate', 'High Rewards Rate', 'bad_ood_high_rewards_std'),
            ('avg_reward', 'Average Reward', 'std_reward'),
            ('avg_factual', 'Factual Reward', None),
            ('accuracy', 'Accuracy', None)
        ]
        
        for metric_key, display_name, std_key in metric_configs:
            if metric_key in metrics:
                value = metrics[metric_key]
                if include_std and std_key and std_key in metrics:
                    std_value = metrics[std_key]
                    print(f"{display_name}: {value:.2f} (±{std_value:.2f})")
                else:
                    print(f"{display_name}: {value:.2f}")
    
    def add_adversarial_stage_metrics(self, pre_metrics, post_metrics):
        """Add special adversarial training stage metrics"""
        self.add_epoch_metrics(-1, pre_metrics)
        self.add_epoch_metrics(999, post_metrics)
        
        # Calculate improvement metrics
        improvement_metrics = {}
        for key in pre_metrics:
            if key in post_metrics and isinstance(pre_metrics[key], (int, float)):
                improvement_metrics[f"{key}_improvement"] = post_metrics[key] - pre_metrics[key]
        
        # Store improvement metrics
        self.add_epoch_metrics(1000, improvement_metrics)
    
    def print_adversarial_impact_analysis(self):
        """Print comprehensive adversarial training impact analysis"""
        if -1 in self.epoch_metrics and 999 in self.epoch_metrics:
            pre_metrics = self.epoch_metrics[-1]
            post_metrics = self.epoch_metrics[999]
            
            print("\n" + "=" * 60)
            print("ADVERSARIAL TRAINING IMPACT ANALYSIS")
            print("=" * 60)
            
            print("\nPRE-ADVERSARIAL PERFORMANCE:")
            self.print_metrics_with_std(pre_metrics, "", include_std=True)
            
            print("\nPOST-ADVERSARIAL PERFORMANCE:")
            self.print_metrics_with_std(post_metrics, "", include_std=True)

        else:
            print("Adversarial training metrics not found")
    
    
    def save_metrics(self, filepath="training_metrics.json"):
        """Save metrics to JSON file"""
        with open(filepath, 'w') as f:
            json.dump(self.epoch_metrics, f, indent=2)
        print(f"Metrics saved to {filepath}")

## Llama + RM + ADV on MeQA-USMLE-4Option

### Main run

In [ ]:
# Run Stage 1
stage1_results = run_stage1(stage1_epochs=2, 
                            stage2_epochs=2, 
                            batch_size=4,
                            model_url='meta-llama/Llama-3.1-8B-Instruct',
                            train_data_path = "../Datasets/medqa_train.json",
                            test_data_path = "../Datasets/medqa_test.json")
print(f"\nStage 1 (Reward Model):")
print(f"   Checkpoint: {stage1_results['checkpoint_path']}")
print(f"   Accuracy: {stage1_results['final_metrics']['accuracy']:.2%}")
print("\n" + "="*80)
print(" "*25 + "TRAINING COMPLETE of Stage 1")
print("="*80)

In [ ]:
reward_config = {
        'w_accuracy': 1.0,
        'w_leak': 0.3,
        'w_preamble': 0.2,
        'w_grounded': 0.5,
        'tau_veracity': 0.4,
        'k_sharpness': 8.0,
        'tau_leak': 0.7,
        'tau_preamble_words': 15,
        'agreement_threshold': 0.5,
        'verification_model': 'Qwen/Qwen2.5-14B-Instruct',
        'use_4bit_verification': True,
    }
CKPT = './checkpoints/llama-3.1-8b-instruct/stage1_reward_training/stage1_checkpoint'

stage1_trainer = PolicyTrainer.load_from_checkpoint(
    checkpoint_path=CKPT,
    reward_config=reward_config,
    base_model_path='meta-llama/Llama-3.1-8B-Instruct',
)

# Match the eval-mode state at end of stage 1
if stage1_trainer.reward_function.fact_verification_system is not None:
    stage1_trainer.reward_function.fact_verification_system.training_mode = False

stage2_results = run_stage2(
    stage1_checkpoint_path=CKPT,
    model_url='meta-llama/Llama-3.1-8B-Instruct',
    trainer=stage1_trainer
)
print(f"\nStage 2 (Adversarial):")
print(f"  Location: {stage2_results['checkpoint_path']}")

## Qwen + RM + ADV on MedQA-USMLE-4option

In [ ]:
# Run Stage 1
stage1_results = run_stage1(stage1_epochs=2, 
                            stage2_epochs=2, 
                            batch_size=4,
                            model_url='Qwen/Qwen2.5-7B-Instruct',
                            train_data_path = "../Datasets/medqa_train.json",
                            test_data_path = "../Datasets/medqa_test.json")
print(f"\nStage 1 (Reward Model):")
print(f"   Checkpoint: {stage1_results['checkpoint_path']}")
print(f"   Accuracy: {stage1_results['final_metrics']['accuracy']:.2%}")
print("\n" + "="*80)
print(" "*25 + "TRAINING COMPLETE of Stage 1")
print("="*80)

In [ ]:
reward_config = {
        'w_accuracy': 1.0,
        'w_leak': 0.3,
        'w_preamble': 0.2,
        'w_grounded': 0.5,
        'tau_veracity': 0.4,
        'k_sharpness': 8.0,
        'tau_leak': 0.7,
        'tau_preamble_words': 15,
        'agreement_threshold': 0.5,
        'verification_model': 'Qwen/Qwen2.5-14B-Instruct',
        'use_4bit_verification': True,
    }
CKPT = './checkpoints/qwen2.5-7b-instruct/stage1_reward_training/stage1_checkpoint'

stage1_trainer = PolicyTrainer.load_from_checkpoint(
    checkpoint_path=CKPT,
    reward_config=reward_config,
    base_model_path='Qwen/Qwen2.5-7B-Instruct',
)
# Match the eval-mode state at end of stage 1
if stage1_trainer.reward_function.fact_verification_system is not None:
    stage1_trainer.reward_function.fact_verification_system.training_mode = False

stage2_results = run_stage2(
    stage1_checkpoint_path=CKPT,
    model_url='Qwen/Qwen2.5-7B-Instruct',
    trainer=stage1_trainer
)
print(f"\nStage 2 (Adversarial):")
print(f"  Location: {stage2_results['checkpoint_path']}")

## Llama + RM + ADV on MMLU

In [ ]:
reward_config = {
        'w_accuracy': 1.0,
        'w_leak': 0.3,
        'w_preamble': 0.2,
        'w_grounded': 0.5,
        'tau_veracity': 0.4,
        'k_sharpness': 8.0,
        'tau_leak': 0.7,
        'tau_preamble_words': 15,
        'agreement_threshold': 0.5,
        'verification_model': 'Qwen/Qwen2.5-14B-Instruct',
        'use_4bit_verification': True,
    }

BASE = 'meta-llama/Llama-3.1-8B-Instruct'
EVAL_PATH = "../Datasets/mmlu_pro_health_test.json"
N = 200

# Reward model (Stage 1)
rm_trainer = PolicyTrainer.load_from_checkpoint(
    checkpoint_path=r'./checkpoints/llama-3.1-8b-instruct/stage1_reward_training/stage1_checkpoint',
    reward_config=reward_config,
    base_model_path=BASE,
)
rm_results = rm_trainer.evaluate_model(EVAL_PATH, N)
print("Reward model:")
print(json.dumps({k: v for k, v in rm_results.items() if k != 'reward_history'}, indent=4))

del rm_trainer; torch.cuda.empty_cache()

In [ ]:
# Adversarial model (final)
adv_trainer = PolicyTrainer.load_from_checkpoint(
    checkpoint_path=r'./checkpoints/llama-3.1-8b-instruct/stage2_adversarial_training/stage2_checkpoint',
    reward_config=reward_config,
    base_model_path=BASE,
)
adv_results = adv_trainer.evaluate_model(EVAL_PATH, N)
print("Adversarial model:")
print(json.dumps({k: v for k, v in adv_results.items() if k != 'reward_history'}, indent=4))

In [ ]:
reward_config = {
    'w_b': 1.0,
    'w_a': 0.3,
    'w_s': 0.3,
    'w_f': 0.3,
    'w_fact': 0.5, 
    'tau_answer': 0.7,
    'tau_preamble': 15,
    'lambda_s': 1.0,
    'verification_model': 'Qwen/Qwen2.5-14B-Instruct',
    'use_4bit_verification': True
}

stage2_trainer = PolicyTrainer.load_from_checkpoint(
    checkpoint_path='D:\\factvr\\Code\\checkpoints\\llama-3.1-8b\\stage2_adversarial_training\\stage2_checkpoint',
    reward_config=reward_config,
    base_model_path='meta-llama/Llama-3.1-8B'
)

# 3. Evaluate
final_results = stage2_trainer.evaluate_model("../Datasets/mmlu_pro_health_test.json", 200)
print(json.dumps({k: v for k, v in final_results.items() if k != 'reward_history'}, indent=4))

## Qwen on MMLU

In [ ]:
reward_config = {
        'w_accuracy': 1.0,
        'w_leak': 0.3,
        'w_preamble': 0.2,
        'w_grounded': 0.5,
        'tau_veracity': 0.4,
        'k_sharpness': 8.0,
        'tau_leak': 0.7,
        'tau_preamble_words': 15,
        'agreement_threshold': 0.5,
        'verification_model': 'Qwen/Qwen2.5-14B-Instruct',
        'use_4bit_verification': True,
    }

BASE = 'Qwen/Qwen2.5-7B-Instruct'
EVAL_PATH = "../Datasets/mmlu_pro_health_test.json"
N = 200

# Reward model (Stage 1)
rm_trainer = PolicyTrainer.load_from_checkpoint(
    checkpoint_path=r'./checkpoints/qwen2.5-7b-instruct/stage1_reward_training/stage1_checkpoint',
    reward_config=reward_config,
    base_model_path=BASE,
)
rm_results = rm_trainer.evaluate_model(EVAL_PATH, N)
print("Reward model:")
print(json.dumps({k: v for k, v in rm_results.items() if k != 'reward_history'}, indent=4))

del rm_trainer; torch.cuda.empty_cache()

In [ ]:
# Adversarial model (final)
adv_trainer = PolicyTrainer.load_from_checkpoint(
    checkpoint_path=r'./checkpoints/qwen2.5-7b-instruct/stage2_adversarial_training/stage2_checkpoint',
    reward_config=reward_config,
    base_model_path=BASE,
)
adv_results = adv_trainer.evaluate_model(EVAL_PATH, N)
print("Adversarial model:")
print(json.dumps({k: v for k, v in adv_results.items() if k != 'reward_history'}, indent=4))